# Beyond Sequential Compression
### Joint Knowledge Distillation, Quantization-Aware Training, and Structured Pruning for Efficient On-Device Language Models

This notebook is a complete, resumable research pipeline. Everything it writes (code package, data
manifests, teacher targets, checkpoints, GGUF exports, logs, figures, results) goes to **Google Drive**,
so a disconnected Colab session continues where it stopped when you re-run the cells.

| | Smoke mode (`MODE="smoke"`) | Full mode (`MODE="full"`) |
|---|---|---|
| Teacher / student | SmolLM2-360M-Instruct / SmolLM2-135M-Instruct | Phi-4-mini-instruct / SmolLM2-1.7B-Instruct |
| GPU | T4 (16 GB) or better; CPU works but is slow | A100 40/80 GB recommended |
| Purpose | verifies every stage end-to-end in < 1 h | the paper protocol |
| Numbers | **meaningless** (pipeline test) | research results |

**Run guide (short).** Colab: *Runtime → Change runtime type → GPU*, add an `HF_TOKEN` secret (key icon) if a
model is gated, then *Run all*. Full mode is run twice: first with `PHASE="pilot"` (tune on pilot data only),
then freeze the protocol and run with `PHASE="final"`. The complete guide (Colab, local CUDA workstation,
Android) is in Section 22.

**Honesty rules enforced in code.** No metric is written unless it was computed from model outputs in this run;
held-out evaluation refuses to run before the protocol is frozen, and pilot evaluation refuses to run after;
Android numbers are `PENDING` until real device files are uploaded; desktop timings are labelled as desktop.

---
## 1. Research question, hypotheses, and experimental protocol

**Question.** At a matched deployment budget (same pruned architecture, same INT4 GGUF grid, same training
steps and data), does optimizing distillation, structured pruning and quantization **jointly** (or in
**alternation**) preserve more quality than applying them **sequentially**?

**Pre-registered hypotheses** (tested in Section 21 with Holm correction over the family):

| | Hypothesis | Comparison |
|---|---|---|
| H1 | Joint KD + gated structured pruning + QAT beats sequential KD → prune → INT4 PTQ | E8 vs E7 |
| H2 | Alternating optimization beats sequential | E9 vs E7 |
| H3 | Joint and alternating differ | E8 vs E9 |
| H4 | *Quantization-aware structural selection*: learning the pruning gates **under INT4 fake-quantization** selects better structures than learning them in full precision | E8 vs A_H4_noQAI |
| H5 | KD + QAT beats INT4 PTQ of the uncompressed student | E5 vs E3 |

H4 is the main novelty claim: prior pipelines (Minitron-style prune→distill, ordering studies, BitDistiller-style
QAT+KD, joint prune+PTQ) choose structures in full precision. Here, gate gradients see the INT4 rounding noise.

**Objective.** For a student with structured gates $g=\sigma(\alpha/T)$ and fake-quantized weights $Q(W)$:

$$\mathcal L_{joint}=\lambda_{KD}\,\mathrm{KL}\!\big(p_T^{\tau}\,\|\,p_{S,Q}^{\tau}\big)\tau^2+\lambda_{CE}\,\mathrm{CE}(y,p_{S,FP})+\lambda_{SP}\,R_{SP}(g)+\lambda_Q\,\mathrm{CE}(y,p_{S,Q})$$

* $y$: **teacher-generated responses** (response-level KD; Phi-4-mini and SmolLM2 have different vocabularies, so
  this is the only valid cross-family signal).
* $p_T$: a **vocabulary-identical** logit teacher. Token-level KD from Phi-4-mini is disabled automatically
  (Section 4 check); the logit teacher for compression stages is the distilled uncompressed student E1
  (self-distillation, as in Minitron/BitDistiller).
* $R_{SP}=\frac1L\sum_l(\bar g^{heads}_l-\rho_h)^2+(\bar g^{mlp}_l-\rho_m)^2+\beta\,\overline{g(1-g)}$:
  target-ratio penalty plus polarization so the final hard selection barely perturbs the network.
* $Q(\cdot)$: straight-through fake quantizer **bit-exact to GGUF Q4_0** (verified against llama.cpp's gguf-py),
  so what is trained is exactly what is deployed. $\lambda_{CE}=0$ by default under QAT (one forward per step).

**Experimental matrix** (S = steps of one compression stage; identical batch, LR schedule, data order per seed):

| ID | Pipeline | Trained steps |
|---|---|---|
| E0 | uncompressed student | 0 |
| R1 | control: CE fine-tune on original responses (budget match for non-KD arms) | S |
| E1 | distillation only | S |
| E2 | R1 → structured prune → CE recovery | 2S |
| E3 / E3_8 | INT4 / INT8 PTQ of E0 | 0 |
| E4 | E1 → prune → KD recovery | 2S |
| E5 | E1 → QAT + KD | 2S |
| E6 | R1 → prune → CE recovery under QAT | 2S |
| E7 | E4 → INT4 PTQ (**sequential**) | 2S |
| E8 | E1 → **joint** gates + QAT + KD (hard prune at 60 % of S) | 2S |
| E9 | E1 → **alternating** prune-phase (FP) ↔ quant-phase (QAT) | 2S |
| A_* | ablations (Section 16) | 2S |

**Threats to validity handled by design.** Same final structure size for every pruned arm; same deployment grid;
final checkpoint = last step (no selection on evaluation data); 13-gram decontamination; pilot vs held-out split;
≥ 3 seeds; paired tests on identical items; desktop vs mobile timings never mixed.

---
## 2. Configuration, reproducibility, seeds and hardware

**Purpose.** Mount Drive, write the core package modules, build the single source-of-truth config, seed all RNGs, record hardware.

**Expected outputs.** `experiment.yaml` (with config hash) in `results/`, `hardware.json`, the Drive directory tree.

**Validation.** Asserts: Drive writable, config round-trips through YAML with identical hash, GPU/bf16 detected.

**If something fails.** Drive mount fails → re-run the cell and accept the permission prompt. No GPU → smoke mode still runs on CPU (slowly). To change any setting, edit `CONFIG_OVERRIDES` only; never edit modules by hand (they are rewritten each run).

In [ ]:
# ---- user settings --------------------------------------------------------------------------------
MODE = "smoke"            # "smoke" (pipeline test) | "full" (paper protocol)
PHASE = None              # None -> "final" in smoke mode, "pilot" in full mode. Set "final" after freezing.
CONFIG_OVERRIDES = {}     # dotted keys, e.g. {"train.stage_steps": 300, "prune.mlp_keep_ratio": 0.6}
# ----------------------------------------------------------------------------------------------------
import os, sys, json, importlib
PHASE = PHASE or ("final" if MODE == "smoke" else "pilot")
DRIVE_ROOT = os.environ.get("JLC_DRIVE_ROOT",
                            "/content/drive/MyDrive/jlc_research" + ("_smoke" if MODE == "smoke" else ""))
if DRIVE_ROOT.startswith("/content/drive"):
    try:
        from google.colab import drive
        if not os.path.ismount("/content/drive"):
            drive.mount("/content/drive")
    except ImportError:
        print("Not running on Colab: using a local directory instead of Google Drive.")
PKG = f"{DRIVE_ROOT}/code"
os.makedirs(f"{PKG}/jlc", exist_ok=True)
if PKG not in sys.path:
    sys.path.insert(0, PKG)
print("MODE =", MODE, "| PHASE =", PHASE, "| DRIVE_ROOT =", DRIVE_ROOT)

### 2.1 Package source (`jlc/`)

All implementation code lives in the `jlc` package, written to `<DRIVE_ROOT>/code/jlc/` by the cells below on every run (the notebook is the single source of truth). Sections 4-22 only *call* it.

| Module | Used in | Responsibility |
|---|---|---|
| config | 2 | all hyper-parameters, smoke/full modes, protocol freezing |
| env | 2-3 | seeds, hardware, memory guards, exact child-process RSS measurement |
| testing | 2 | offline CI fixtures hook (inactive on Colab) |
| ckpt | all | Drive layout, atomic writes, checksums, resumable checkpoints, safe loading |
| models | 4 | loading, revision pinning, inspection, vocabulary compatibility |
| prune | 4, 9-11, 14 | GQA-aware importance, gates, masks, physical pruning, verification |
| data | 5 | datasets, dedup, decontamination, manifests, SFT tensors |
| teacher | 6 | cached teacher generation |
| evaluate | 7, 17 | GSM8K, IFEval, perplexity, size/load/RAM |
| quant | 12-15 | GGUF-exact grids, STE fake-quant, torchao alternative, PTQ |
| train | 8-16 | training engine and the joint objective |
| experiments | 7-18 | condition matrix (stage graph), evaluation and export drivers |
| pipeline | 8-16 | phase-aware orchestration |
| analysis | 9 | sensitivity analysis |
| export | 18-19 | GGUF conversion, validation, Android kit |
| bench | 20 | runtime benchmarking |
| stats | 21 | aggregation, tests, Pareto, figures |
| report | 22 | report and artifact manifest |

In [ ]:
%%writefile {PKG}/jlc/__init__.py
"""jlc: Joint LLM Compression research package (generated by the notebook into Google Drive)."""
__version__ = "1.0.0"

**Module `config.py`** — every hyper-parameter, both modes, protocol freezing.

In [ ]:
%%writefile {PKG}/jlc/config.py
"""Centralised, versioned experiment configuration.

Every value used anywhere in the pipeline lives here. The resolved config is
written to `experiment.yaml` and hashed; each run records that hash so a result
can always be traced back to the exact protocol that produced it.
"""
from __future__ import annotations

import copy
import dataclasses as dc
import hashlib
import json
from pathlib import Path
from typing import Any

import yaml


@dc.dataclass
class ModelCfg:
    teacher_id: str = "microsoft/Phi-4-mini-instruct"
    teacher_revision: str | None = None          # pinned to a commit SHA in Section 4
    student_id: str = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
    student_revision: str | None = None
    teacher_load_4bit: bool | None = None         # None = decide from available VRAM
    dtype: str = "bfloat16"                       # falls back to float16 on pre-Ampere GPUs
    attn_implementation: str = "sdpa"
    trust_remote_code: bool = False               # never needed for Llama/Phi3 in transformers>=4.48


@dc.dataclass
class DataCfg:
    ultrachat_id: str = "HuggingFaceH4/ultrachat_200k"
    ultrachat_train_split: str = "train_sft"
    ultrachat_dev_split: str = "test_sft"
    gsm8k_id: str = "openai/gsm8k"
    gsm8k_config: str = "main"
    wikitext_id: str = "Salesforce/wikitext"
    wikitext_config: str = "wikitext-2-raw-v1"
    ifeval_id: str = "google/IFEval"
    revisions: dict = dc.field(default_factory=dict)  # filled with resolved commit SHAs
    n_ultrachat_train: int = 20000
    n_gsm8k_train_prompts: int = 5000             # GSM8K *train* questions sent to the teacher
    n_gsm8k_dev: int = 300                        # held out from GSM8K train for pilot tuning only
    n_ultrachat_dev: int = 500
    n_calibration: int = 256                      # pruning importance / imatrix calibration samples
    max_prompt_tokens: int = 512
    contamination_ngram: int = 13
    min_prompt_chars: int = 16


@dc.dataclass
class TeacherGenCfg:
    backend: str = "auto"                         # auto | vllm | hf
    max_new_tokens: int = 768
    temperature: float = 0.0                      # greedy => deterministic targets
    top_p: float = 1.0
    batch_size: int = 32
    shard_size: int = 256
    exclude_truncated_from_training: bool = True  # recorded in the manifest, never silent
    system_prompt: str | None = None


@dc.dataclass
class TrainCfg:
    stage_steps: int = 600                        # S: budget of ONE compression stage
    seq_len: int = 1024
    micro_batch: int = 4
    grad_accum: int = 8
    lr: float = 2e-5
    gate_lr: float = 5e-2
    weight_decay: float = 0.0
    warmup_ratio: float = 0.05
    max_grad_norm: float = 1.0
    optimizer: str = "auto"                       # auto | adamw | adamw8bit
    gradient_checkpointing: bool = True
    ckpt_every: int = 100
    keep_last_ckpts: int = 2
    save_optimizer_state: bool = True
    log_every: int = 10
    dev_eval_every: int = 200
    seeds: tuple = (0, 1, 2)
    pilot_seed: int = 1234


@dc.dataclass
class LossCfg:
    # L_joint = l_kd*L_KD + l_ce*L_CE + l_sp*R_SP + l_q*L_Q  (see Section 1 / losses.py)
    l_kd: float = 1.0
    l_ce: float = 0.0       # CE through the full-precision forward; >0 doubles forward cost under QAT
    l_q: float = 1.0        # CE through the fake-quantised forward (only when QAT is active)
    l_sp: float = 1.0
    l_ce_noqat: float = 1.0 # CE weight when QAT is inactive (L_Q undefined there)
    kd_temperature: float = 2.0
    kd_mode: str = "forward_kl"                  # forward_kl | reverse_kl | jsd


@dc.dataclass
class PruneCfg:
    granularity: str = "kv_group"                 # kv_group: prune a KV head with all its query heads
    head_keep_ratio: float = 0.5                  # SmolLM2-1.7B (32 MHA heads, hidden 2048): reloadable
                                                  # head counts are powers of two -> {1.0, 0.5, 0.25, ...}
    head_count_policy: str = "hf_valid"
    mlp_keep_ratio: float = 0.70
    mlp_multiple: int = 64                        # kept channels rounded to this (Q4_0 rows need %32==0)
    importance: str = "activation"                # activation | taylor | magnitude
    uniform_across_layers: bool = True            # keeps a standard LlamaConfig -> HF reload + GGUF export
    sensitivity_ratios: tuple = (0.9, 0.8, 0.7, 0.6, 0.5)
    gate_init: float = 3.0                        # logit; sigmoid(3)=0.95
    gate_temperature_start: float = 1.0
    gate_temperature_end: float = 0.1
    hard_prune_at: float = 0.6                    # fraction of the joint stage after which structure is frozen


@dc.dataclass
class QuantCfg:
    qat_backend: str = "q4_0_exact"               # q4_0_exact (verified vs gguf-py) | torchao_intx
    w_bits: int = 4
    group_size: int = 32                          # == GGUF Q4_0 / Q8_0 block size
    a_bits: int | None = None                     # weight-only
    exclude: tuple = ("embed_tokens", "lm_head")  # kept at Q8_0 in GGUF (token-embd/output tensors)
    gguf_types: tuple = ("F16", "Q8_0", "Q4_0")
    gguf_embedding_type: str = "q8_0"
    qat_ramp_fraction: float = 0.0                # fake-quant is on from step 0 by default


@dc.dataclass
class AltCfg:
    cycle_steps: int = 50                         # E9: steps per (prune-phase, QAT-phase) half-cycle


@dc.dataclass
class EvalCfg:
    gsm8k_n: int | None = None                    # None = full test set (1319)
    ifeval_n: int | None = None                   # None = all 541
    gsm8k_max_new_tokens: int = 512
    ifeval_max_new_tokens: int = 1024
    eval_batch_size: int = 16
    ppl_ctx: int = 512                            # llama.cpp-compatible chunking
    ppl_max_chunks: int | None = None
    bootstrap_iters: int = 10000
    alpha: float = 0.05


@dc.dataclass
class BenchCfg:
    llama_cpp_repo: str = "https://github.com/ggml-org/llama.cpp"
    llama_cpp_ref: str = "master"                 # pin to a release tag for the final run (recorded)
    prompt_lengths: tuple = (128, 512)
    gen_length: int = 128
    repetitions: int = 5
    warmup_runs: int = 1
    threads: int = 4
    parity_ppl_rel_tol: float = 0.03


@dc.dataclass
class PathsCfg:
    drive_root: str = "/content/drive/MyDrive/jlc_research"
    local_root: str = "/content/jlc_local"        # fast scratch; never the source of truth

    def sub(self, name: str) -> Path:
        p = Path(self.drive_root) / name
        p.mkdir(parents=True, exist_ok=True)
        return p


@dc.dataclass
class Config:
    mode: str = "full"
    model: ModelCfg = dc.field(default_factory=ModelCfg)
    data: DataCfg = dc.field(default_factory=DataCfg)
    teacher_gen: TeacherGenCfg = dc.field(default_factory=TeacherGenCfg)
    train: TrainCfg = dc.field(default_factory=TrainCfg)
    loss: LossCfg = dc.field(default_factory=LossCfg)
    prune: PruneCfg = dc.field(default_factory=PruneCfg)
    quant: QuantCfg = dc.field(default_factory=QuantCfg)
    alt: AltCfg = dc.field(default_factory=AltCfg)
    eval: EvalCfg = dc.field(default_factory=EvalCfg)
    bench: BenchCfg = dc.field(default_factory=BenchCfg)
    paths: PathsCfg = dc.field(default_factory=PathsCfg)
    protocol_frozen: bool = False

    # ------------------------------------------------------------------ io
    def to_dict(self) -> dict:
        return dc.asdict(self)

    def hash(self) -> str:
        d = self.to_dict()
        d.pop("protocol_frozen", None)
        d.pop("paths", None)              # where files live does not change the science
        d["train"].pop("seeds", None)     # which seeds are run does not change any single run
        d["train"].pop("pilot_seed", None)
        blob = json.dumps(d, sort_keys=True, default=str).encode()
        return hashlib.sha256(blob).hexdigest()[:16]

    def save_yaml(self, path: str | Path) -> None:
        d = self.to_dict()
        d["config_hash"] = self.hash()
        Path(path).write_text(yaml.safe_dump(json.loads(json.dumps(d, default=list)), sort_keys=False))

    @staticmethod
    def load_yaml(path: str | Path) -> "Config":
        raw = yaml.safe_load(Path(path).read_text())
        raw.pop("config_hash", None)
        return _from_dict(raw)


def _from_dict(raw: dict) -> Config:
    cfg = Config()
    for key, val in raw.items():
        sub = getattr(cfg, key, None)
        if dc.is_dataclass(sub) and isinstance(val, dict):
            for k, v in val.items():
                if not hasattr(sub, k):
                    raise KeyError(f"Unknown config key {key}.{k}")
                cur = getattr(sub, k)
                setattr(sub, k, tuple(v) if isinstance(cur, tuple) and isinstance(v, list) else v)
        elif hasattr(cfg, key):
            setattr(cfg, key, val)
        else:
            raise KeyError(f"Unknown config key {key}")
    return cfg


def make_config(mode: str = "full", **overrides: Any) -> Config:
    """mode='full' -> paper protocol; mode='smoke' -> tiny models/data that run on a T4 or CPU.

    overrides use dotted keys, e.g. make_config('full', **{'train.stage_steps': 300}).
    """
    cfg = Config(mode=mode)
    if mode == "smoke":
        cfg.model.teacher_id = "HuggingFaceTB/SmolLM2-360M-Instruct"   # same vocab -> exercises token-level KD
        cfg.model.student_id = "HuggingFaceTB/SmolLM2-135M-Instruct"   # GQA: 9 q-heads / 3 kv-heads
        cfg.data.n_ultrachat_train = 64
        cfg.data.n_gsm8k_train_prompts = 16
        cfg.data.n_gsm8k_dev = 16
        cfg.data.n_ultrachat_dev = 16
        cfg.data.n_calibration = 16
        cfg.data.max_prompt_tokens = 256
        cfg.teacher_gen.max_new_tokens = 96
        cfg.teacher_gen.batch_size = 8
        cfg.teacher_gen.shard_size = 32
        cfg.train.stage_steps = 12
        cfg.train.seq_len = 256
        cfg.train.micro_batch = 2
        cfg.train.grad_accum = 2
        cfg.train.lr = 5e-5
        cfg.train.ckpt_every = 6
        cfg.train.log_every = 2
        cfg.train.dev_eval_every = 6
        cfg.train.seeds = (0, 1)
        cfg.prune.head_keep_ratio = 2 / 3
        cfg.prune.sensitivity_ratios = (0.8, 0.6)
        cfg.alt.cycle_steps = 3
        cfg.eval.gsm8k_n = 16
        cfg.eval.ifeval_n = 16
        cfg.eval.gsm8k_max_new_tokens = 128
        cfg.eval.ifeval_max_new_tokens = 128
        cfg.eval.eval_batch_size = 8
        cfg.eval.ppl_max_chunks = 8
        cfg.eval.bootstrap_iters = 1000
        cfg.bench.repetitions = 2
        cfg.paths.drive_root = cfg.paths.drive_root.rstrip("/") + "_smoke"
    elif mode != "full":
        raise ValueError("mode must be 'full' or 'smoke'")
    for dotted, val in overrides.items():
        sec, key = dotted.split(".", 1)
        sub = getattr(cfg, sec)
        if not hasattr(sub, key):
            raise KeyError(f"Unknown override {dotted}")
        setattr(sub, key, val)
    return cfg


def freeze_protocol(cfg: Config, results_dir: Path) -> dict:
    """Write an immutable record of the final protocol. Final-evaluation code refuses to run
    without it, and pilot-tuning code refuses to run after it (unless explicitly overridden,
    which is logged)."""
    f = Path(results_dir) / "protocol_frozen.json"
    if f.exists():
        rec = json.loads(f.read_text())
        if rec["config_hash"] != cfg.hash():
            raise RuntimeError(
                f"Protocol already frozen with hash {rec['config_hash']} but current config hash is "
                f"{cfg.hash()}. Changing the protocol after freezing invalidates held-out results. "
                "Revert the change, or delete protocol_frozen.json AND all final results deliberately.")
        return rec
    import datetime as _dt
    rec = {"config_hash": cfg.hash(), "frozen_at_utc": _dt.datetime.utcnow().isoformat() + "Z",
           "config": json.loads(json.dumps(cfg.to_dict(), default=list))}
    f.write_text(json.dumps(rec, indent=2))
    return rec


def is_frozen(cfg: Config, results_dir: Path) -> bool:
    f = Path(results_dir) / "protocol_frozen.json"
    return f.exists() and json.loads(f.read_text())["config_hash"] == cfg.hash()


__all__ = ["Config", "make_config", "freeze_protocol", "is_frozen"]

**Module `env.py`** — seeds, determinism flags, hardware report, memory guards, credentials from Colab secrets.

In [ ]:
%%writefile {PKG}/jlc/env.py
"""Reproducibility, hardware detection, memory guards and environment capture."""
from __future__ import annotations

import gc
import importlib
import json
import os
import platform
import random
import subprocess
import sys
from pathlib import Path

import numpy as np

REQUIRED = {  # package -> minimum version actually relied upon (APIs verified in Section 3)
    "torch": "2.4",
    "transformers": "4.48",
    "datasets": "2.20",
    "accelerate": "0.33",
    "safetensors": "0.4",
    "numpy": "1.26",
    "pandas": "2.0",
    "scipy": "1.11",
    "matplotlib": "3.7",
    "yaml": "6.0",
    "psutil": "5.9",
}
OPTIONAL = ["torchao", "vllm", "bitsandbytes", "lm_eval", "gguf", "huggingface_hub"]


def _ver(mod) -> str:
    return getattr(mod, "__version__", "unknown")


def _vtuple(v: str):
    out = []
    for p in v.split("+")[0].split("."):
        num = "".join(c for c in p if c.isdigit())
        out.append(int(num) if num else 0)
    return tuple(out)


def check_versions(strict: bool = True) -> dict:
    """Import every dependency, compare against REQUIRED, and return a version table.
    Raises on a missing/too-old required package when strict=True (never silently continues)."""
    table, problems = {}, []
    for pkg, minv in REQUIRED.items():
        try:
            m = importlib.import_module(pkg)
            v = _ver(m)
            table[pkg] = v
            if v != "unknown" and _vtuple(v) < _vtuple(minv):
                problems.append(f"{pkg} {v} < required {minv}")
        except Exception as e:  # noqa: BLE001
            table[pkg] = f"MISSING ({e.__class__.__name__})"
            problems.append(f"{pkg} missing")
    for pkg in OPTIONAL:
        try:
            table[pkg] = _ver(importlib.import_module(pkg))
        except Exception:  # noqa: BLE001
            table[pkg] = "not installed"
    if problems and strict:
        raise RuntimeError("Dependency check failed: " + "; ".join(problems))
    return table


def set_seed(seed: int, deterministic: bool = True) -> None:
    """Seed every RNG we use. Full bitwise determinism on GPU is not guaranteed for all kernels
    (e.g. SDPA backward, atomics); we enable the deterministic flags that are safe and record them."""
    import torch

    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        try:
            torch.use_deterministic_algorithms(True, warn_only=True)
        except Exception:  # noqa: BLE001
            pass


def hardware_report() -> dict:
    import psutil
    import torch

    rep = {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "cpu": platform.processor() or platform.machine(),
        "cpu_count": os.cpu_count(),
        "ram_total_gb": round(psutil.virtual_memory().total / 2**30, 2),
        "ram_available_gb": round(psutil.virtual_memory().available / 2**30, 2),
        "cuda_available": torch.cuda.is_available(),
        "gpus": [],
        "in_colab": "google.colab" in sys.modules or os.path.exists("/content"),
    }
    if torch.cuda.is_available():
        rep["cuda_version"] = torch.version.cuda
        rep["cudnn"] = torch.backends.cudnn.version()
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            rep["gpus"].append({"name": p.name, "total_gb": round(p.total_memory / 2**30, 2),
                                "capability": f"{p.major}.{p.minor}"})
        rep["bf16_supported"] = torch.cuda.is_bf16_supported()
    try:
        rep["nvidia_smi"] = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total",
                                            "--format=csv,noheader"], capture_output=True, text=True,
                                           timeout=20).stdout.strip()
    except Exception:  # noqa: BLE001
        rep["nvidia_smi"] = "unavailable"
    return rep


def resolve_dtype(pref: str = "bfloat16"):
    import torch

    if not torch.cuda.is_available():
        return torch.float32
    if pref == "bfloat16" and not torch.cuda.is_bf16_supported():
        print("[env] bf16 unsupported on this GPU (e.g. T4) -> using float16")
        return torch.float16
    return getattr(torch, pref)


def free_gpu_gb() -> float:
    import torch

    if not torch.cuda.is_available():
        return 0.0
    free, _ = torch.cuda.mem_get_info()
    return free / 2**30


def require_memory(gpu_gb: float = 0.0, cpu_gb: float = 0.0, what: str = "") -> None:
    """Fail *before* a large allocation with an actionable message instead of crashing mid-run."""
    import psutil

    cleanup()
    if gpu_gb and free_gpu_gb() < gpu_gb:
        raise MemoryError(f"[{what}] needs ~{gpu_gb:.1f} GB free GPU memory, only {free_gpu_gb():.1f} GB free. "
                          "Use smoke mode, a larger GPU, 4-bit teacher loading, or reduce micro_batch/seq_len.")
    avail = psutil.virtual_memory().available / 2**30
    if cpu_gb and avail < cpu_gb:
        raise MemoryError(f"[{what}] needs ~{cpu_gb:.1f} GB free RAM, only {avail:.1f} GB available.")


def estimate_model_gb(n_params: float, bytes_per_param: float = 2.0) -> float:
    return n_params * bytes_per_param / 2**30


def cleanup() -> None:
    import torch

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def write_environment_txt(path: str | Path, versions: dict, hw: dict, extra: dict | None = None) -> None:
    lines = ["# Environment record (auto-generated)", "", "## Packages"]
    lines += [f"{k}=={v}" for k, v in sorted(versions.items())]
    lines += ["", "## Hardware", json.dumps(hw, indent=2)]
    if extra:
        lines += ["", "## Extra", json.dumps(extra, indent=2, default=str)]
    try:
        freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True,
                                timeout=120).stdout
        lines += ["", "## pip freeze", freeze]
    except Exception:  # noqa: BLE001
        pass
    Path(path).write_text("\n".join(lines))


def hf_token() -> str | None:
    """Read a Hugging Face token from Colab secrets or the environment. Never hard-coded."""
    tok = os.environ.get("HF_TOKEN")
    if tok:
        return tok
    try:
        from google.colab import userdata  # type: ignore

        return userdata.get("HF_TOKEN")
    except Exception:  # noqa: BLE001
        return None


_LAUNCHER = r"""
import json, os, sys, time
quiet = sys.argv[1] == "1"
t = time.perf_counter()
pid = os.fork()
if pid == 0:
    if quiet:
        dn = os.open(os.devnull, os.O_WRONLY); os.dup2(dn, 1); os.dup2(dn, 2)
    os.execv(sys.argv[2], sys.argv[2:])
_, st, ru = os.wait4(pid, 0)
print("JLC_MEASURE " + json.dumps({"maxrss_kb": ru.ru_maxrss, "exit": os.waitstatus_to_exitcode(st),
                                    "wall_s": time.perf_counter() - t}))
"""


_FLOOR = {}


def launcher_floor_mb() -> float:
    """Peak RSS of a trivial command measured the same way: the lower bound of any measurement."""
    if "mb" not in _FLOOR:
        import subprocess

        p = subprocess.run([sys.executable, "-S", "-c", _LAUNCHER, "1", "/bin/true"], capture_output=True, text=True)
        line = [l for l in p.stdout.splitlines() if l.startswith("JLC_MEASURE ")]
        _FLOOR["mb"] = json.loads(line[-1].split(" ", 1)[1])["maxrss_kb"] / 1024 if line else float("nan")
    return _FLOOR["mb"]


def run_measured(argv: list, quiet: bool = True, timeout: int = 3600) -> dict:
    """Run argv and return its exact peak RSS (kernel rusage), exit code, wall time and stdout.

    Why a launcher: ru_maxrss of a process forked directly from the notebook kernel includes the
    kernel's own resident memory at fork time (it would report GBs for a 1 MB model). The command is
    therefore forked from a fresh minimal Python launcher, so the measured peak is the command's own
    (plus at most the launcher's ~10 MB footprint, reported as `launcher_floor_mb`)."""
    import subprocess

    p = subprocess.run([sys.executable, "-S", "-c", _LAUNCHER, "1" if quiet else "0", *map(str, argv)],
                       capture_output=True, text=True, timeout=timeout)
    lines = [l for l in p.stdout.splitlines() if l.startswith("JLC_MEASURE ")]
    if not lines:
        raise RuntimeError(f"measurement launcher failed: {p.stderr[-800:]}")
    rec = json.loads(lines[-1].split(" ", 1)[1])
    rec["peak_rss_mb"] = rec["maxrss_kb"] / 1024
    rec["stdout"] = "\n".join(l for l in p.stdout.splitlines() if not l.startswith("JLC_MEASURE "))
    rec["stderr"] = p.stderr
    rec["launcher_floor_mb"] = launcher_floor_mb()
    return rec

**Module `testing.py`** — test-only offline fixture hook (inactive unless `JLC_OFFLINE_FIXTURES` is set; never on Colab).

In [ ]:
%%writefile {PKG}/jlc/testing.py
"""TEST-ONLY offline mode. Activated exclusively when the environment variable JLC_OFFLINE_FIXTURES
points to a fixtures directory (created by tests/fixtures.py). It replaces Hub downloads with local
tiny random models and synthetic datasets that have the real schemas, so the full notebook can be
executed end-to-end in CI without network or GPU. Numbers produced in this mode are meaningless and
every result file is stamped with offline_fixture=True."""
from __future__ import annotations

import os
import sys
from pathlib import Path


def active() -> bool:
    return bool(os.environ.get("JLC_OFFLINE_FIXTURES"))


def install(cfg) -> dict:
    d = Path(os.environ["JLC_OFFLINE_FIXTURES"])
    sys.path.insert(0, str(d))
    import fixtures  # noqa: E402  (tests/fixtures.py copied into the fixtures dir)

    import importlib

    for name, patches in (("data", {"_load": fixtures.fake_datasets(), "resolve_revision": lambda *a, **k: None}),
                          ("models", {"resolve_model_revision": lambda *a, **k: None})):
        try:   # modules are written section by section; patch the ones that exist (re-applied after reloads)
            mod = importlib.import_module(f"jlc.{name}")
        except ImportError:
            continue
        for k, v in patches.items():
            setattr(mod, k, v)
    cfg.model.student_id = str(d / "student")
    cfg.model.teacher_id = str(d / "teacher")
    cfg.teacher_gen.exclude_truncated_from_training = False   # random teacher never emits EOS
    cfg.teacher_gen.backend = "hf"
    cfg.prune.mlp_multiple = 64
    return {"offline_fixture": True, "fixtures_dir": str(d)}

**Module `ckpt.py`** — Drive layout, atomic writes, SHA-256 checksums, stage markers, resumable training checkpoints with safe loading.

In [ ]:
%%writefile {PKG}/jlc/ckpt.py
"""Google-Drive-backed artifact store: atomic writes, checksums, stage markers, resumable
training checkpoints, failure logging and safe loading.

Layout under cfg.paths.drive_root:
    checkpoints/<exp>/seed_<s>/<stage>/          final stage model (+ DONE.json)
    checkpoints/<exp>/seed_<s>/<stage>/train/    rolling training checkpoints (step_XXXXXX)
    teacher_targets/                             cached teacher generations (jsonl shards)
    data/                                        split manifests and processed splits
    exports/                                     GGUF files + parity reports
    logs/                                        training logs, failures.jsonl
    figures/                                     plots
    results/                                     metrics, runs/*.json, statistical tests, report
"""
from __future__ import annotations

import datetime as dt
import functools
import hashlib
import json
import os
import shutil
import time
import traceback
from pathlib import Path
from typing import Any, Callable

SUBDIRS = ["checkpoints", "teacher_targets", "data", "exports", "logs", "figures", "results", "android"]


def mount_drive(root: str) -> Path:
    """Mount Google Drive on Colab; elsewhere just create the directory (local workstation)."""
    if root.startswith("/content/drive"):
        try:
            from google.colab import drive  # type: ignore

            if not os.path.ismount("/content/drive"):
                drive.mount("/content/drive")
        except ImportError:
            print("[ckpt] Not on Colab: /content/drive path will be a plain local directory.")
    p = Path(root)
    p.mkdir(parents=True, exist_ok=True)
    for s in SUBDIRS:
        (p / s).mkdir(exist_ok=True)
    return p


def now() -> str:
    return dt.datetime.utcnow().isoformat(timespec="seconds") + "Z"


def sha256_file(path: str | Path, chunk: int = 1 << 22) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def dir_checksums(d: str | Path, skip: tuple = ("DONE.json", "CHECKSUMS.json")) -> dict:
    d = Path(d)
    return {str(p.relative_to(d)): sha256_file(p) for p in sorted(d.rglob("*"))
            if p.is_file() and p.name not in skip and "train" not in p.relative_to(d).parts[:1]}


def atomic_write_text(path: str | Path, text: str) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + f".tmp{os.getpid()}")
    tmp.write_text(text)
    os.replace(tmp, path)


def atomic_write_json(path: str | Path, obj: Any) -> None:
    atomic_write_text(path, json.dumps(obj, indent=2, default=str))


def atomic_copy_dir(src: str | Path, dst: str | Path) -> None:
    """Copy a directory to Drive so that `dst` either fully exists or does not exist."""
    dst = Path(dst)
    tmp = dst.with_name(dst.name + ".partial")
    if tmp.exists():
        shutil.rmtree(tmp)
    shutil.copytree(src, tmp)
    if dst.exists():
        old = dst.with_name(dst.name + ".old")
        if old.exists():
            shutil.rmtree(old)
        os.replace(dst, old)
        os.replace(tmp, dst)
        shutil.rmtree(old, ignore_errors=True)
    else:
        os.replace(tmp, dst)


# ----------------------------------------------------------------------------- stage markers
def stage_dir(root: Path, exp: str, seed: int, stage: str) -> Path:
    return Path(root) / "checkpoints" / exp / f"seed_{seed}" / stage


def is_done(d: str | Path, verify: bool = False) -> bool:
    marker = Path(d) / "DONE.json"
    if not marker.exists():
        return False
    if verify:
        rec = json.loads(marker.read_text())
        bad = verify_checksums(d, rec.get("checksums", {}))
        if bad:
            print(f"[ckpt] checksum mismatch in {d}: {bad[:3]} -> stage will be recomputed")
            return False
    return True


def mark_done(d: str | Path, info: dict | None = None, checksum: bool = True) -> dict:
    d = Path(d)
    rec = {"completed_utc": now(), **(info or {})}
    if checksum:
        rec["checksums"] = dir_checksums(d)
    atomic_write_json(d / "DONE.json", rec)
    return rec


def read_done(d: str | Path) -> dict:
    return json.loads((Path(d) / "DONE.json").read_text())


def verify_checksums(d: str | Path, expected: dict) -> list:
    d = Path(d)
    bad = []
    for rel, h in expected.items():
        p = d / rel
        if not p.exists() or sha256_file(p) != h:
            bad.append(rel)
    return bad


# ----------------------------------------------------------------------------- failures
def log_failure(root: Path, where: str, exc: BaseException, context: dict | None = None) -> dict:
    rec = {"utc": now(), "where": where, "error": f"{exc.__class__.__name__}: {exc}",
           "traceback": traceback.format_exc(), "context": context or {}}
    with open(Path(root) / "logs" / "failures.jsonl", "a") as f:
        f.write(json.dumps(rec, default=str) + "\n")
    return rec


def retry(times: int = 5, wait: float = 10.0, exceptions: tuple = (Exception,)) -> Callable:
    """Retry decorator for network operations (model/dataset downloads)."""

    def deco(fn):
        @functools.wraps(fn)
        def wrapper(*a, **k):
            last = None
            for i in range(times):
                try:
                    return fn(*a, **k)
                except exceptions as e:  # noqa: PERF203
                    last = e
                    msg = str(e).lower()
                    if any(s in msg for s in ("gated", "401", "403", "not found", "404")):
                        raise  # permanent errors: retrying will not help
                    print(f"[retry] {fn.__name__} failed ({e.__class__.__name__}: {e}); attempt {i + 1}/{times}")
                    time.sleep(wait * (i + 1))
            raise RuntimeError(f"{fn.__name__} failed after {times} attempts") from last

        return wrapper

    return deco


# ----------------------------------------------------------------------------- training ckpts
def _ckpt_dirs(train_dir: Path) -> list[Path]:
    return sorted([p for p in train_dir.glob("step_*") if (p / "META.json").exists()],
                  key=lambda p: int(p.name.split("_")[1]))


def save_train_ckpt(train_dir: str | Path, step: int, model, extra_state: dict, optimizer=None,
                    keep_last: int = 2, local_scratch: str | None = None) -> Path:
    """Save model weights (safetensors), gate/aux state, optimizer + RNG (torch.save) and
    META.json written LAST, so a half-written checkpoint is never picked up on resume."""
    import torch
    from safetensors.torch import save_file

    train_dir = Path(train_dir)
    train_dir.mkdir(parents=True, exist_ok=True)
    scratch = Path(local_scratch or "/tmp") / f"ckpt_{os.getpid()}_{step}"
    if scratch.exists():
        shutil.rmtree(scratch)
    scratch.mkdir(parents=True)
    sd = {k: v.detach().contiguous().cpu() for k, v in model.state_dict().items()}
    _dedupe_shared(sd)
    save_file(sd, str(scratch / "model.safetensors"))
    model.config.to_json_file(str(scratch / "config.json"))
    nps = __import__("numpy").random.get_state()
    rng = {"python": __import__("random").getstate(),
           "numpy": [nps[0], nps[1].tolist(), int(nps[2]), int(nps[3]), float(nps[4])],  # no numpy objects
           "torch": torch.get_rng_state(),
           "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []}
    torch.save({"extra": _to_cpu(extra_state), "rng": rng}, scratch / "state.pt")
    if optimizer is not None:
        torch.save(_to_cpu(optimizer.state_dict()), scratch / "optimizer.pt")
    meta = {"step": step, "utc": now(), "files": {p.name: sha256_file(p) for p in scratch.iterdir()}}
    (scratch / "META.json").write_text(json.dumps(meta, indent=2))
    dst = train_dir / f"step_{step:06d}"
    atomic_copy_dir(scratch, dst)
    shutil.rmtree(scratch, ignore_errors=True)
    for old in _ckpt_dirs(train_dir)[:-keep_last]:
        shutil.rmtree(old, ignore_errors=True)
    return dst


def latest_train_ckpt(train_dir: str | Path, verify: bool = True) -> Path | None:
    train_dir = Path(train_dir)
    if not train_dir.exists():
        return None
    for d in reversed(_ckpt_dirs(train_dir)):
        meta = json.loads((d / "META.json").read_text())
        if not verify or all((d / f).exists() and sha256_file(d / f) == h for f, h in meta["files"].items()):
            return d
        print(f"[ckpt] corrupt checkpoint {d.name} skipped")
    return None


def load_train_ckpt(d: str | Path, model, optimizer=None, restore_rng: bool = True) -> dict:
    """Safe loading: safetensors for weights, torch.load(weights_only=True) for the rest."""
    import torch
    from safetensors.torch import load_file

    d = Path(d)
    sd = load_file(str(d / "model.safetensors"))
    missing, unexpected = model.load_state_dict(sd, strict=False)
    missing = [m for m in missing if not _is_tied(model, m)]
    if missing or unexpected:
        raise RuntimeError(f"Checkpoint/model mismatch: missing={missing[:5]} unexpected={unexpected[:5]}")
    st = _safe_torch_load(d / "state.pt")
    if optimizer is not None and (d / "optimizer.pt").exists():
        optimizer.load_state_dict(_safe_torch_load(d / "optimizer.pt"))
    if restore_rng:
        import random

        import numpy as np

        random.setstate(_as_tuple(st["rng"]["python"]))
        n = st["rng"]["numpy"]
        np.random.set_state((n[0], np.array(n[1], dtype=np.uint32), n[2], n[3], n[4]))
        torch.set_rng_state(st["rng"]["torch"])
        if torch.cuda.is_available() and st["rng"]["cuda"]:
            torch.cuda.set_rng_state_all(st["rng"]["cuda"])
    meta = json.loads((d / "META.json").read_text())
    return {"step": meta["step"], "extra": st["extra"]}


def _safe_torch_load(p):
    """weights_only=True: only tensors and plain Python containers can be deserialised."""
    import torch

    try:
        return torch.load(p, map_location="cpu", weights_only=True)
    except Exception as e:  # noqa: BLE001
        raise RuntimeError(f"Refusing unsafe torch.load of {p}: {e}") from e


def _as_tuple(x):
    if isinstance(x, list):
        return tuple(_as_tuple(i) for i in x)
    return x


def _to_cpu(obj):
    import torch

    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu()
    if isinstance(obj, dict):
        return {k: _to_cpu(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return type(obj)(_to_cpu(v) for v in obj)
    return obj


def _dedupe_shared(sd: dict) -> None:
    seen = {}
    for k in list(sd):
        ptr = (sd[k].data_ptr(), tuple(sd[k].shape))
        if ptr in seen and sd[k].numel() > 0:
            sd[k] = sd[k].clone()  # safetensors forbids shared storage; tied weights restored on load
        seen[ptr] = k


def _is_tied(model, name: str) -> bool:
    return name == "lm_head.weight" and getattr(model.config, "tie_word_embeddings", False)


# ----------------------------------------------------------------------------- final models
def save_model_validated(model, tokenizer, out_dir: str | Path, probe_text: str = "The capital of France is",
                         atol: float = 2e-2, local_scratch: str | None = None, extra_files: dict | None = None,
                         info: dict | None = None) -> dict:
    """save_pretrained -> reload -> compare logits on a probe -> atomic copy to Drive -> DONE.json."""
    import torch
    from transformers import AutoModelForCausalLM

    out_dir = Path(out_dir)
    scratch = Path(local_scratch or "/tmp") / f"final_{os.getpid()}_{out_dir.name}"
    if scratch.exists():
        shutil.rmtree(scratch)
    model.save_pretrained(scratch, safe_serialization=True)
    tokenizer.save_pretrained(scratch)
    for name, content in (extra_files or {}).items():
        (scratch / name).write_text(content if isinstance(content, str) else json.dumps(content, indent=2))
    dev = next(model.parameters()).device
    ids = tokenizer(probe_text, return_tensors="pt").input_ids.to(dev)
    was_training = model.training
    model.eval()
    with torch.no_grad():
        ref = model(ids).logits.float().cpu()
        re = AutoModelForCausalLM.from_pretrained(scratch, torch_dtype=next(model.parameters()).dtype).to(dev).eval()
        got = re(ids).logits.float().cpu()
    model.train(was_training)
    diff = (ref - got).abs().max().item()
    del re
    if diff > atol:
        raise RuntimeError(f"Reload validation failed for {out_dir}: max |Δlogit| = {diff:.4g} > {atol}")
    atomic_copy_dir(scratch, out_dir)
    shutil.rmtree(scratch, ignore_errors=True)
    return mark_done(out_dir, {"reload_max_abs_logit_diff": diff, **(info or {})})

**Module `models.py`** — loading with memory pre-checks, revision pinning, inspection and vocabulary compatibility.

In [ ]:
%%writefile {PKG}/jlc/models.py
"""Model loading, inspection (config / tokenizer / license / hardware needs) and vocabulary
compatibility checks for token-level distillation."""
from __future__ import annotations

import json
from pathlib import Path

from .ckpt import retry
from .env import estimate_model_gb, free_gpu_gb, hf_token, require_memory, resolve_dtype


@retry(times=4, wait=20)
def _from_pretrained(cls, model_id, **kw):
    return cls.from_pretrained(model_id, token=hf_token(), **kw)


def resolve_model_revision(model_id: str) -> str | None:
    try:
        from huggingface_hub import HfApi

        return HfApi(token=hf_token()).model_info(model_id).sha
    except Exception as e:  # noqa: BLE001
        print(f"[models] could not resolve revision of {model_id}: {e}")
        return None


def inspect_model(model_id: str, revision: str | None = None) -> dict:
    """Config, tokenizer, architecture, license and memory estimate WITHOUT loading weights."""
    from huggingface_hub import HfApi
    from transformers import AutoConfig, AutoTokenizer

    cfg = _from_pretrained(AutoConfig, model_id, revision=revision)
    tok = _from_pretrained(AutoTokenizer, model_id, revision=revision)
    rep = {"model_id": model_id, "revision": revision, "model_type": cfg.model_type,
           "architectures": getattr(cfg, "architectures", None), "hidden_size": cfg.hidden_size,
           "num_hidden_layers": cfg.num_hidden_layers, "num_attention_heads": cfg.num_attention_heads,
           "num_key_value_heads": getattr(cfg, "num_key_value_heads", None),
           "head_dim": getattr(cfg, "head_dim", None), "intermediate_size": cfg.intermediate_size,
           "vocab_size": cfg.vocab_size, "tokenizer_len": len(tok), "tie_word_embeddings": cfg.tie_word_embeddings,
           "max_position_embeddings": getattr(cfg, "max_position_embeddings", None),
           "chat_template_present": bool(getattr(tok, "chat_template", None)),
           "eos_token": tok.eos_token, "bos_token": tok.bos_token, "pad_token": tok.pad_token}
    try:
        info = HfApi(token=hf_token()).model_info(model_id, revision=revision)
        rep["license"] = (info.card_data or {}).get("license") if info.card_data else None
        rep["gated"] = getattr(info, "gated", None)
        rep["safetensors_params"] = info.safetensors.total if getattr(info, "safetensors", None) else None
        rep["resolved_revision"] = info.sha
    except Exception as e:  # noqa: BLE001
        rep["license"] = f"unresolved ({e.__class__.__name__})"
    n = rep.get("safetensors_params") or _approx_params(cfg)
    rep["params_estimate"] = n
    rep["bf16_weights_gb"] = round(estimate_model_gb(n, 2), 2)
    rep["full_ft_adamw_gb_estimate"] = round(estimate_model_gb(n, 2 + 2 + 8 + 4), 1)   # w,g,adam(fp32 m,v),master
    rep["full_ft_adamw8bit_gb_estimate"] = round(estimate_model_gb(n, 2 + 2 + 2 + 4), 1)
    return rep


def _approx_params(cfg) -> int:
    h, L, i, v = cfg.hidden_size, cfg.num_hidden_layers, cfg.intermediate_size, cfg.vocab_size
    nh, nkv = cfg.num_attention_heads, getattr(cfg, "num_key_value_heads", cfg.num_attention_heads)
    hd = getattr(cfg, "head_dim", None) or h // nh
    per = h * nh * hd * 2 + h * nkv * hd * 2 + 3 * h * i + 2 * h
    return L * per + v * h * (1 if cfg.tie_word_embeddings else 2)


def vocab_compatibility(tok_a, tok_b, probe_texts: list[str]) -> dict:
    """Token-level KD requires an identical output space: same vocab size, same token->id map and
    identical tokenization of probe texts. Any mismatch => token-level KD disabled (logged)."""
    va, vb = tok_a.get_vocab(), tok_b.get_vocab()
    same_map = va == vb
    same_tok = all(tok_a(t, add_special_tokens=False).input_ids == tok_b(t, add_special_tokens=False).input_ids
                   for t in probe_texts)
    overlap = len(set(va) & set(vb)) / max(len(va), len(vb))
    compatible = same_map and same_tok and len(tok_a) == len(tok_b)
    return {"compatible": compatible, "same_token_to_id_map": same_map, "same_probe_tokenization": same_tok,
            "len_a": len(tok_a), "len_b": len(tok_b), "token_string_overlap": round(overlap, 4),
            "decision": "token-level KD ENABLED" if compatible else
            "token-level KD DISABLED (incompatible output spaces) -> response-level KD only"}


def load_model(model_id: str, revision: str | None = None, role: str = "student", cfg=None,
               device: str | None = None, dtype=None):
    """Load model+tokenizer with a memory pre-check. Teacher may be loaded 4-bit (bitsandbytes NF4)
    when VRAM is short; that choice is recorded by the caller (it changes teacher targets)."""
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    dtype = dtype or resolve_dtype(cfg.model.dtype if cfg else "bfloat16")
    tok = _from_pretrained(AutoTokenizer, model_id, revision=revision)
    rep_params = None
    try:
        rep_params = inspect_model(model_id, revision)["params_estimate"]
    except Exception:  # noqa: BLE001
        pass
    kw = {"revision": revision, "torch_dtype": dtype}
    if cfg is not None:
        kw["attn_implementation"] = cfg.model.attn_implementation
    use4 = False
    if role == "teacher" and cfg is not None:
        use4 = cfg.model.teacher_load_4bit
        if use4 is None and torch.cuda.is_available() and rep_params:
            use4 = free_gpu_gb() < estimate_model_gb(rep_params, 2) * 1.6
    if use4:
        from transformers import BitsAndBytesConfig

        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                       bnb_4bit_compute_dtype=dtype)
        kw["device_map"] = "auto"
        print(f"[models] loading {model_id} in 4-bit NF4 (VRAM-limited); recorded in teacher manifest")
    elif torch.cuda.is_available() and rep_params:
        require_memory(gpu_gb=estimate_model_gb(rep_params, 2) * 1.15, what=f"load {model_id}")
    model = _from_pretrained(AutoModelForCausalLM, model_id, **kw)
    if not use4:
        model.to(device or ("cuda" if torch.cuda.is_available() else "cpu"))
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    model.eval()
    model._jlc_load_info = {"model_id": model_id, "revision": revision, "dtype": str(dtype), "4bit": bool(use4)}
    return model, tok


def load_local(path: str | Path, cfg=None, device=None):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    dtype = resolve_dtype(cfg.model.dtype if cfg else "bfloat16")
    model = AutoModelForCausalLM.from_pretrained(str(path), torch_dtype=dtype)
    model.to(device or ("cuda" if torch.cuda.is_available() else "cpu")).eval()
    tok = AutoTokenizer.from_pretrained(str(path))
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    return model, tok


def write_model_manifest(path: Path, entries: dict) -> None:
    Path(path).write_text(json.dumps(entries, indent=2, default=str))

**Module `prune.py`** — GQA-aware structured pruning: inspection, importance, gates, masks, physical slicing, verification.

In [ ]:
%%writefile {PKG}/jlc/prune.py
"""Architecture-aware structured pruning for Llama-family decoders (SmolLM2 is `llama`).

Four distinct objects, never conflated (Section 10):
  1. *Differentiable gates*   GateBank: sigmoid(logit/T) multipliers on the input of o_proj (per KV
                              group) and down_proj (per MLP channel). Parameter count unchanged.
  2. *Permanent masks*        the same hooks with fixed 0/1 values. Function == pruned model, but
                              tensors keep full size: NO memory or latency saving.
  3. *Physical reduction*     physically_prune(): slices q/k/v/o and gate/up/down weights and rewrites
                              the config. Only this changes dimensions, file size and latency.
  4. *Exported parameters*    counted from the saved safetensors / GGUF file (evaluate.py, export.py).

Grouped-query attention: a KV head is shared by n_rep = n_heads / n_kv query heads. Removing a KV
head therefore removes its whole query group (granularity 'kv_group'). This keeps n_rep, so
repeat_kv and the GGUF converter remain valid. Pruning is uniform in count across layers (which
units are kept is chosen per layer) so the result is a standard LlamaConfig that reloads with
from_pretrained and converts to GGUF without custom code.
"""
from __future__ import annotations

import math
from dataclasses import dataclass, field

import torch
import torch.nn as nn

SUPPORTED_MODEL_TYPES = {"llama"}


class UnsupportedArchitecture(RuntimeError):
    pass


@dataclass
class Structure:
    n_layers: int
    n_heads: int
    n_kv: int
    head_dim: int
    hidden: int
    inter: int

    @property
    def n_rep(self) -> int:
        return self.n_heads // self.n_kv


@dataclass
class PrunePlan:
    kv_keep: list[list[int]] = field(default_factory=list)    # per layer, sorted KV-group ids kept
    mlp_keep: list[list[int]] = field(default_factory=list)   # per layer, sorted channel ids kept
    importance: str = ""
    meta: dict = field(default_factory=dict)

    def to_json(self) -> dict:
        return {"kv_keep": self.kv_keep, "mlp_keep": self.mlp_keep, "importance": self.importance,
                "meta": self.meta}

    @staticmethod
    def from_json(d: dict) -> "PrunePlan":
        return PrunePlan(d["kv_keep"], d["mlp_keep"], d.get("importance", ""), d.get("meta", {}))


# ----------------------------------------------------------------------------- inspection
def inspect_structure(model) -> Structure:
    """Validate that the model can be pruned safely by this implementation; raise a diagnostic
    UnsupportedArchitecture otherwise (never silently apply an invalid transformation)."""
    cfg = model.config
    problems = []
    if cfg.model_type not in SUPPORTED_MODEL_TYPES:
        problems.append(f"model_type={cfg.model_type!r} (supported: {sorted(SUPPORTED_MODEL_TYPES)}). "
                        "E.g. Phi-3/Phi-4 use fused qkv_proj and gate_up_proj; slicing them needs a "
                        "separate implementation and is out of scope.")
    layers = _layers(model, raise_=False)
    if layers is None:
        problems.append("could not locate model.model.layers")
    else:
        a, m = layers[0].self_attn, layers[0].mlp
        for n in ("q_proj", "k_proj", "v_proj", "o_proj"):
            if not isinstance(getattr(a, n, None), nn.Linear):
                problems.append(f"self_attn.{n} missing or not nn.Linear")
        for n in ("gate_proj", "up_proj", "down_proj"):
            if not isinstance(getattr(m, n, None), nn.Linear):
                problems.append(f"mlp.{n} missing or not nn.Linear")
    if getattr(cfg, "attention_bias", False) or getattr(cfg, "mlp_bias", False):
        problems.append("biases present: supported in principle but untested -> refusing")
    if problems:
        raise UnsupportedArchitecture("Cannot safely prune this model:\n  - " + "\n  - ".join(problems))
    head_dim = getattr(cfg, "head_dim", None) or cfg.hidden_size // cfg.num_attention_heads
    s = Structure(cfg.num_hidden_layers, cfg.num_attention_heads, cfg.num_key_value_heads, head_dim,
                  cfg.hidden_size, cfg.intermediate_size)
    if s.n_heads % s.n_kv:
        raise UnsupportedArchitecture("num_attention_heads not divisible by num_key_value_heads")
    a = layers[0].self_attn
    exp = {"q_proj": (s.n_heads * head_dim, s.hidden), "k_proj": (s.n_kv * head_dim, s.hidden),
           "v_proj": (s.n_kv * head_dim, s.hidden), "o_proj": (s.hidden, s.n_heads * head_dim)}
    for n, shp in exp.items():
        if tuple(getattr(a, n).weight.shape) != shp:
            raise UnsupportedArchitecture(f"{n} weight shape {tuple(getattr(a, n).weight.shape)} != {shp}")
    return s


def _layers(model, raise_: bool = True):
    try:
        return model.model.layers
    except AttributeError:
        if raise_:
            raise UnsupportedArchitecture("model.model.layers not found") from None
        return None


def valid_kv_counts(s: Structure) -> list[int]:
    """KV-group counts whose resulting query-head count divides hidden_size.

    transformers>=5 LlamaConfig.validate_architecture rejects hidden_size % num_attention_heads != 0
    even when head_dim is explicit, so other counts would produce checkpoints that cannot be
    reloaded with from_pretrained. (llama.cpp itself has no such restriction.)"""
    return [k for k in range(1, s.n_kv + 1) if s.hidden % (k * s.n_rep) == 0]


def keep_counts(s: Structure, head_keep_ratio: float, mlp_keep_ratio: float, multiple: int = 64,
                policy: str = "hf_valid") -> tuple[int, int, dict]:
    """Returns (kv_groups_kept, mlp_channels_kept, info). With policy='hf_valid' the head count is
    snapped to the nearest reloadable value and the deviation from the requested ratio is reported."""
    target = s.n_kv * head_keep_ratio
    if policy == "hf_valid":
        cands = valid_kv_counts(s)
        kv = min(cands, key=lambda k: (abs(k - target), -k))
    elif policy == "any":
        kv = max(1, min(s.n_kv, int(round(target))))
    else:
        raise ValueError(policy)
    mlp = min(s.inter, max(multiple, int(round(s.inter * mlp_keep_ratio / multiple)) * multiple))
    info = {"requested_head_keep": head_keep_ratio, "effective_head_keep": kv / s.n_kv,
            "requested_mlp_keep": mlp_keep_ratio, "effective_mlp_keep": mlp / s.inter,
            "valid_kv_counts": valid_kv_counts(s), "policy": policy}
    if abs(info["effective_head_keep"] - head_keep_ratio) > 1e-9:
        print(f"[prune] head keep ratio {head_keep_ratio:.3f} snapped to {kv}/{s.n_kv}="
              f"{kv / s.n_kv:.3f} (reloadable counts: {info['valid_kv_counts']})")
    return kv, mlp, info


# ----------------------------------------------------------------------------- masks / gates
class _Mult:
    """Forward-pre-hook multiplying a Linear's input by a per-feature vector from a provider."""

    def __init__(self, provider, expand: int = 1):
        self.provider, self.expand = provider, expand

    def __call__(self, module, args):
        x = args[0]
        v = self.provider()
        if v is None:
            return None
        if self.expand > 1:
            v = v.repeat_interleave(self.expand)
        return (x * v.to(x.dtype),) + tuple(args[1:])


class MaskBank:
    """Fixed multiplicative masks (permanent masks, or Taylor-importance probes when requires_grad)."""

    def __init__(self, model, s: Structure, requires_grad: bool = False):
        dev = next(model.parameters()).device
        self.s = s
        self.kv = [torch.ones(s.n_kv, device=dev, requires_grad=requires_grad) for _ in range(s.n_layers)]
        self.mlp = [torch.ones(s.inter, device=dev, requires_grad=requires_grad) for _ in range(s.n_layers)]
        self.handles = []
        for i, layer in enumerate(_layers(model)):
            self.handles.append(layer.self_attn.o_proj.register_forward_pre_hook(
                _Mult(lambda i=i: self.kv[i], expand=s.n_rep * s.head_dim)))
            self.handles.append(layer.mlp.down_proj.register_forward_pre_hook(_Mult(lambda i=i: self.mlp[i])))

    def apply_plan(self, plan: PrunePlan):
        with torch.no_grad():
            for i in range(self.s.n_layers):
                self.kv[i].zero_()
                self.kv[i][plan.kv_keep[i]] = 1
                self.mlp[i].zero_()
                self.mlp[i][plan.mlp_keep[i]] = 1

    def reset(self):
        with torch.no_grad():
            for t in self.kv + self.mlp:
                t.fill_(1)

    def remove(self):
        for h in self.handles:
            h.remove()
        self.handles = []


class GateBank(nn.Module):
    """Differentiable structured gates g = sigmoid(logit / T)."""

    def __init__(self, model, s: Structure, init: float = 3.0):
        super().__init__()
        dev = next(model.parameters()).device
        self.s = s
        self.kv_logits = nn.Parameter(torch.full((s.n_layers, s.n_kv), float(init), device=dev))
        self.mlp_logits = nn.Parameter(torch.full((s.n_layers, s.inter), float(init), device=dev))
        self.temperature = 1.0
        self.frozen_open = False
        self.handles = []
        for i, layer in enumerate(_layers(model)):
            self.handles.append(layer.self_attn.o_proj.register_forward_pre_hook(
                _Mult(lambda i=i: self.kv_gate(i), expand=s.n_rep * s.head_dim)))
            self.handles.append(layer.mlp.down_proj.register_forward_pre_hook(
                _Mult(lambda i=i: self.mlp_gate(i))))

    def kv_gate(self, i):
        return torch.sigmoid(self.kv_logits[i] / self.temperature)

    def mlp_gate(self, i):
        return torch.sigmoid(self.mlp_logits[i] / self.temperature)

    def regularizer(self, kv_target: float, mlp_target: float, polarize: float = 0.1) -> torch.Tensor:
        """R_SP: squared deviation of each layer's expected kept fraction from the target keep ratio,
        plus a polarization term sum g(1-g) that drives gates toward {0,1} so that the final hard
        selection perturbs the network as little as possible."""
        gk = torch.sigmoid(self.kv_logits / self.temperature)
        gm = torch.sigmoid(self.mlp_logits / self.temperature)
        r = ((gk.mean(1) - kv_target) ** 2).mean() + ((gm.mean(1) - mlp_target) ** 2).mean()
        p = (gk * (1 - gk)).mean() + (gm * (1 - gm)).mean()
        return r + polarize * p

    @torch.no_grad()
    def stats(self) -> dict:
        gk = torch.sigmoid(self.kv_logits / self.temperature)
        gm = torch.sigmoid(self.mlp_logits / self.temperature)
        return {"kv_mean": gk.mean().item(), "mlp_mean": gm.mean().item(),
                "kv_frac_gt_half": (gk > 0.5).float().mean().item(),
                "mlp_frac_gt_half": (gm > 0.5).float().mean().item(), "T": self.temperature}

    @torch.no_grad()
    def to_plan(self, kv_n: int, mlp_n: int, tiebreak: dict | None = None) -> PrunePlan:
        plan = PrunePlan(importance="learned_gates", meta={"temperature": self.temperature})
        for i in range(self.s.n_layers):
            sk, sm = self.kv_logits[i].float().clone(), self.mlp_logits[i].float().clone()
            if tiebreak:
                sk += 1e-6 * _norm(tiebreak["kv"][i].to(sk.device))
                sm += 1e-6 * _norm(tiebreak["mlp"][i].to(sm.device))
            plan.kv_keep.append(sorted(torch.topk(sk, kv_n).indices.tolist()))
            plan.mlp_keep.append(sorted(torch.topk(sm, mlp_n).indices.tolist()))
        return plan

    @torch.no_grad()
    def gate_values_for(self, plan: PrunePlan) -> tuple[list, list]:
        kv = [self.kv_gate(i)[plan.kv_keep[i]] for i in range(self.s.n_layers)]
        mlp = [self.mlp_gate(i)[plan.mlp_keep[i]] for i in range(self.s.n_layers)]
        return kv, mlp

    def remove(self):
        for h in self.handles:
            h.remove()
        self.handles = []


def _norm(x):
    return (x - x.mean()) / (x.std() + 1e-12)


# ----------------------------------------------------------------------------- importance
def compute_importance(model, s: Structure, batches, method: str = "activation", loss_fn=None) -> dict:
    """Importance score per KV group and per MLP channel for every layer.

    activation : Minitron-style. head h: mean over tokens of ||attn_out_h||_2 (input of o_proj),
                 summed over the query heads of a KV group; channel c: mean |act(gate)*up|_c
                 (input of down_proj).
    taylor     : first-order Taylor estimate of the loss change when a unit is removed:
                 |sum_tokens dL/dm * m| with a multiplicative mask m=1 on the unit.
    magnitude  : data-free. KV group: ||W_q rows||+||W_k rows||+||W_v rows||+||W_o cols||;
                 channel: ||W_gate row|| * ||W_up row|| * ||W_down col||.
    """
    layers = _layers(model)
    dev = next(model.parameters()).device
    kv = [torch.zeros(s.n_kv, device=dev) for _ in range(s.n_layers)]
    mlp = [torch.zeros(s.inter, device=dev) for _ in range(s.n_layers)]
    if method == "magnitude":
        with torch.no_grad():
            for i, L in enumerate(layers):
                a, m, hd, r = L.self_attn, L.mlp, s.head_dim, s.n_rep
                q = a.q_proj.weight.float().view(s.n_kv, r * hd, -1).norm(dim=(1, 2))
                k = a.k_proj.weight.float().view(s.n_kv, hd, -1).norm(dim=(1, 2))
                v = a.v_proj.weight.float().view(s.n_kv, hd, -1).norm(dim=(1, 2))
                o = a.o_proj.weight.float().view(-1, s.n_kv, r * hd).norm(dim=(0, 2))
                kv[i] = q + k + v + o
                mlp[i] = (m.gate_proj.weight.float().norm(dim=1) * m.up_proj.weight.float().norm(dim=1)
                          * m.down_proj.weight.float().norm(dim=0))
        return {"kv": [t.cpu() for t in kv], "mlp": [t.cpu() for t in mlp], "method": method}

    was_training = model.training
    model.eval()
    if method == "activation":
        handles, ntok = [], [0]

        def hk(i):
            def f(mod, args):
                x = args[0].detach().float()
                h = x.view(*x.shape[:-1], s.n_kv, s.n_rep, s.head_dim).norm(dim=-1).sum(-1)  # [B,T,n_kv]
                kv[i] += h.reshape(-1, s.n_kv).sum(0)
            return f

        def hm(i):
            def f(mod, args):
                mlp[i] += args[0].detach().float().abs().reshape(-1, s.inter).sum(0)
            return f

        for i, L in enumerate(layers):
            handles.append(L.self_attn.o_proj.register_forward_pre_hook(hk(i)))
            handles.append(L.mlp.down_proj.register_forward_pre_hook(hm(i)))
        try:
            with torch.no_grad():
                for b in batches:
                    am = b.get("attention_mask")
                    ntok[0] += int(am.sum()) if am is not None else b["input_ids"].numel()
                    model(input_ids=b["input_ids"].to(dev),
                          attention_mask=None if am is None else am.to(dev))
        finally:
            for h in handles:
                h.remove()
        kv = [t / max(ntok[0], 1) for t in kv]
        mlp = [t / max(ntok[0], 1) for t in mlp]
    elif method == "taylor":
        mb = MaskBank(model, s, requires_grad=True)
        try:
            for b in batches:
                out = model(input_ids=b["input_ids"].to(dev),
                            attention_mask=None if b.get("attention_mask") is None else b["attention_mask"].to(dev),
                            labels=b["labels"].to(dev))
                loss = out.loss if loss_fn is None else loss_fn(out, b)
                grads = torch.autograd.grad(loss, mb.kv + mb.mlp)
                for i in range(s.n_layers):
                    kv[i] += grads[i].detach().abs()
                    mlp[i] += grads[s.n_layers + i].detach().abs()
        finally:
            mb.remove()
            model.zero_grad(set_to_none=True)
    else:
        raise ValueError(f"unknown importance method {method}")
    model.train(was_training)
    return {"kv": [t.cpu() for t in kv], "mlp": [t.cpu() for t in mlp], "method": method}


def plan_from_importance(imp: dict, kv_n: int, mlp_n: int, s: Structure, which: str = "both") -> PrunePlan:
    """which: 'heads' | 'mlp' | 'both' (the other component is kept intact)."""
    plan = PrunePlan(importance=imp["method"], meta={"which": which})
    for i in range(s.n_layers):
        plan.kv_keep.append(sorted(torch.topk(imp["kv"][i], kv_n).indices.tolist()) if which in ("heads", "both")
                            else list(range(s.n_kv)))
        plan.mlp_keep.append(sorted(torch.topk(imp["mlp"][i], mlp_n).indices.tolist()) if which in ("mlp", "both")
                             else list(range(s.inter)))
    return plan


# ----------------------------------------------------------------------------- physical pruning
def _slice_linear(lin: nn.Linear, out_idx=None, in_idx=None, in_scale=None) -> None:
    w = lin.weight.data
    if out_idx is not None:
        w = w[out_idx]
    if in_idx is not None:
        w = w[:, in_idx]
    if in_scale is not None:
        w = w * in_scale.to(w.dtype).unsqueeze(0)
    lin.weight = nn.Parameter(w.contiguous(), requires_grad=lin.weight.requires_grad)
    if lin.bias is not None and out_idx is not None:
        lin.bias = nn.Parameter(lin.bias.data[out_idx].contiguous())
    lin.out_features, lin.in_features = lin.weight.shape


@torch.no_grad()
def physically_prune(model, plan: PrunePlan, fold_gates: tuple | None = None) -> dict:
    """Slice weights according to `plan` and rewrite the config. If fold_gates=(kv_vals, mlp_vals) is
    given, the soft gate values of the KEPT units are folded into o_proj/down_proj columns so the
    pruned network computes exactly what the gated network computed for those units."""
    s = inspect_structure(model)
    kv_n, mlp_n = len(plan.kv_keep[0]), len(plan.mlp_keep[0])
    if any(len(k) != kv_n for k in plan.kv_keep) or any(len(m) != mlp_n for m in plan.mlp_keep):
        raise ValueError("Non-uniform plan: per-layer counts differ (would break a standard LlamaConfig)")
    if mlp_n % 32:
        raise ValueError(f"Kept MLP width {mlp_n} not divisible by 32 (GGUF Q4_0 row constraint)")
    before = sum(p.numel() for p in model.parameters())
    hd, r = s.head_dim, s.n_rep
    for i, L in enumerate(_layers(model)):
        a, m = L.self_attn, L.mlp
        g = torch.tensor(plan.kv_keep[i], device=a.q_proj.weight.device)
        q_idx = (g[:, None] * (r * hd) + torch.arange(r * hd, device=g.device)).reshape(-1)
        kv_idx = (g[:, None] * hd + torch.arange(hd, device=g.device)).reshape(-1)
        _slice_linear(a.q_proj, out_idx=q_idx)
        _slice_linear(a.k_proj, out_idx=kv_idx)
        _slice_linear(a.v_proj, out_idx=kv_idx)
        o_scale = None
        if fold_gates is not None:
            o_scale = fold_gates[0][i].float().repeat_interleave(r * hd)
        _slice_linear(a.o_proj, in_idx=q_idx, in_scale=o_scale)
        c = torch.tensor(plan.mlp_keep[i], device=m.gate_proj.weight.device)
        _slice_linear(m.gate_proj, out_idx=c)
        _slice_linear(m.up_proj, out_idx=c)
        _slice_linear(m.down_proj, in_idx=c, in_scale=None if fold_gates is None else fold_gates[1][i].float())
        a.num_key_value_groups = r                        # unchanged by construction
        if hasattr(a, "num_heads"):
            a.num_heads = kv_n * r
        if hasattr(a, "num_key_value_heads"):
            a.num_key_value_heads = kv_n
        if hasattr(m, "intermediate_size"):
            m.intermediate_size = mlp_n
    cfg = model.config
    cfg.head_dim = hd                                     # must be explicit once heads*hd != hidden
    cfg.num_attention_heads = kv_n * r
    cfg.num_key_value_heads = kv_n
    cfg.intermediate_size = mlp_n
    after = sum(p.numel() for p in model.parameters())
    return {"params_before": before, "params_after": after, "kv_heads": kv_n, "q_heads": kv_n * r,
            "intermediate": mlp_n, "head_dim": hd}


def expected_params(s: Structure, kv_n: int, mlp_n: int, vocab: int, tied: bool) -> int:
    r, hd, h = s.n_rep, s.head_dim, s.hidden
    attn = h * (kv_n * r * hd) * 2 + h * (kv_n * hd) * 2
    mlp = 3 * h * mlp_n
    norms = 2 * h
    return s.n_layers * (attn + mlp + norms) + vocab * h * (1 if tied else 2) + h


@torch.no_grad()
def verify_pruned(model, tokenizer, s_orig: Structure, plan: PrunePlan, ref_logits=None, probe_ids=None,
                  atol: float = 5e-2) -> dict:
    """Shape checks, analytic parameter count, finite forward, generation, (optional) equivalence with
    the masked unpruned model. Raises on any failure."""
    s = inspect_structure(model)
    kv_n, mlp_n = len(plan.kv_keep[0]), len(plan.mlp_keep[0])
    rep = {"shape_checks": "ok"}
    assert s.n_kv == kv_n and s.n_heads == kv_n * s_orig.n_rep and s.inter == mlp_n, "config mismatch"
    for i, L in enumerate(_layers(model)):
        a, m = L.self_attn, L.mlp
        assert a.q_proj.weight.shape == (kv_n * s.n_rep * s.head_dim, s.hidden), f"L{i} q_proj"
        assert a.k_proj.weight.shape == (kv_n * s.head_dim, s.hidden), f"L{i} k_proj"
        assert a.o_proj.weight.shape == (s.hidden, kv_n * s.n_rep * s.head_dim), f"L{i} o_proj"
        assert m.down_proj.weight.shape == (s.hidden, mlp_n), f"L{i} down_proj"
    n = sum(p.numel() for p in model.parameters())
    tied = bool(getattr(model.config, "tie_word_embeddings", False))
    exp = expected_params(s_orig, kv_n, mlp_n, model.config.vocab_size, tied)
    rep["params"], rep["params_expected"] = n, exp
    if n != exp:
        raise AssertionError(f"parameter count {n} != analytic {exp}")
    dev = next(model.parameters()).device
    ids = probe_ids if probe_ids is not None else tokenizer("Structured pruning test:", return_tensors="pt").input_ids
    ids = ids.to(dev)
    logits = model(ids).logits
    if not torch.isfinite(logits).all():
        raise AssertionError("non-finite logits after pruning")
    if ref_logits is not None:
        d = (logits.float().cpu() - ref_logits.float().cpu()).abs().max().item()
        rep["max_abs_diff_vs_masked"] = d
        if d > atol:
            raise AssertionError(f"pruned model differs from masked model by {d:.3g} > {atol}")
    gen = model.generate(ids, max_new_tokens=8, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    rep["generated_tokens"] = int(gen.shape[1] - ids.shape[1])
    if rep["generated_tokens"] <= 0:
        raise AssertionError("generation produced no tokens")
    return rep

**Module `data.py`**.

In [ ]:
%%writefile {PKG}/jlc/data.py
"""Dataset acquisition, normalization, deduplication, contamination filtering and split manifests.

Isolation rules (enforced, not just documented):
  * Teacher prompts / training data: UltraChat train_sft + a subset of GSM8K *train*.
  * Pilot tuning (hyper-parameters only): UltraChat test_sft subset + a disjoint GSM8K-train subset
    + WikiText-2 *validation*.
  * Final held-out evaluation: GSM8K *test*, all IFEval prompts, WikiText-2 *test*. Never seen by
    the teacher, never used for tuning. Any training prompt sharing a 13-gram with any evaluation
    prompt is removed (counts logged).
"""
from __future__ import annotations

import hashlib
import json
import re
import unicodedata
from pathlib import Path

from .ckpt import atomic_write_json, retry, sha256_file

GSM8K_INSTRUCTION = ("Solve the following math problem step by step. At the end, write the final "
                     "answer on its own line in the form '#### <number>'.\n\n")


def normalize(text: str) -> str:
    text = unicodedata.normalize("NFKC", text or "")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    return re.sub(r"[ \t]+", " ", text).strip()


def norm_key(text: str) -> str:
    return re.sub(r"\s+", " ", normalize(text).lower())


def uid(text: str) -> str:
    return hashlib.sha1(norm_key(text).encode()).hexdigest()[:16]


def ngrams(text: str, n: int) -> set:
    w = re.findall(r"\w+", text.lower())
    return {" ".join(w[i:i + n]) for i in range(len(w) - n + 1)}


def stable_sample(items: list, k: int | None, seed: int, key=lambda r: r["id"]) -> list:
    """Order by sha256(seed|id): reproducible regardless of upstream row order."""
    ranked = sorted(items, key=lambda r: hashlib.sha256(f"{seed}|{key(r)}".encode()).hexdigest())
    return ranked if k is None else ranked[:k]


def resolve_revision(repo_id: str, repo_type: str = "dataset") -> str | None:
    try:
        from huggingface_hub import HfApi

        api = HfApi()
        info = api.dataset_info(repo_id) if repo_type == "dataset" else api.model_info(repo_id)
        return info.sha
    except Exception as e:  # noqa: BLE001
        print(f"[data] could not resolve revision for {repo_id}: {e}")
        return None


@retry(times=5, wait=15)
def _load(repo, config=None, split=None, revision=None):
    from datasets import load_dataset

    return load_dataset(repo, config, split=split, revision=revision)


def gsm8k_gold(answer: str) -> str:
    return answer.split("####")[-1].strip().replace(",", "")


def gsm8k_reference(answer: str) -> str:
    """Human-written GSM8K rationale in the same '#### <number>' format the teacher is asked for,
    with calculator annotations <<a*b=c>> removed. Used as targets for NON-distillation conditions."""
    body, _, ans = answer.rpartition("####")
    body = re.sub(r"<<[^>]*>>", "", body).strip()
    return f"{body}\n#### {ans.strip()}"


def build_splits(cfg, data_dir: Path, student_tokenizer=None, seed: int = 0) -> dict:
    """Download, normalize, dedup, decontaminate, sample and write all splits + dataset_manifest.json.
    Idempotent: if the manifest exists with the same config, it is returned unchanged."""
    dc = cfg.data
    data_dir = Path(data_dir)
    man_path = data_dir / "dataset_manifest.json"
    settings = {k: v for k, v in vars(dc).items() if k != "revisions"}
    if man_path.exists():
        man = json.loads(man_path.read_text())
        if man.get("settings") == json.loads(json.dumps(settings, default=list)):
            bad = [s for s, m in man["splits"].items() if sha256_file(data_dir / m["file"]) != m["sha256"]]
            if not bad:
                print("[data] manifest found and checksums verified -> reusing splits")
                return man
            print(f"[data] checksum mismatch for {bad} -> rebuilding")

    revs = {}
    for repo in (dc.ultrachat_id, dc.gsm8k_id, dc.wikitext_id, dc.ifeval_id):
        revs[repo] = dc.revisions.get(repo) or resolve_revision(repo)
    log = {}

    # ---------------- evaluation sets first (they define contamination)
    g_test = _load(dc.gsm8k_id, dc.gsm8k_config, "test", revs[dc.gsm8k_id])
    gsm_test = [{"id": uid(r["question"]), "question": normalize(r["question"]), "gold": gsm8k_gold(r["answer"])}
                for r in g_test]
    ife = _load(dc.ifeval_id, None, "train", revs[dc.ifeval_id])
    ifeval = [{"id": str(r["key"]), "key": r["key"], "prompt": r["prompt"],
               "instruction_id_list": r["instruction_id_list"], "kwargs": r["kwargs"]} for r in ife]
    wt = {sp: _load(dc.wikitext_id, dc.wikitext_config, sp, revs[dc.wikitext_id]) for sp in ("validation", "test")}
    wiki = {sp: "".join(x["text"] for x in ds) for sp, ds in wt.items()}  # raw concatenation (llama.cpp style)

    # ---------------- GSM8K train: teacher pool and pilot-dev pool are disjoint
    g_train = _load(dc.gsm8k_id, dc.gsm8k_config, "train", revs[dc.gsm8k_id])
    gtr = [{"id": uid(r["question"]), "question": normalize(r["question"]), "gold": gsm8k_gold(r["answer"]),
            "reference_response": gsm8k_reference(r["answer"])} for r in g_train]
    gtr = list({r["id"]: r for r in gtr}.values())
    gtr = stable_sample(gtr, None, seed)
    gsm_dev = gtr[:dc.n_gsm8k_dev]
    gsm_teacher = gtr[dc.n_gsm8k_dev:dc.n_gsm8k_dev + dc.n_gsm8k_train_prompts]

    eval_ngrams = set()
    for r in gsm_test:
        eval_ngrams |= ngrams(r["question"], dc.contamination_ngram)
    for r in ifeval:
        eval_ngrams |= ngrams(r["prompt"], dc.contamination_ngram)
    eval_ngrams |= ngrams(wiki["test"], dc.contamination_ngram)

    # ---------------- UltraChat
    def uc_rows(split):
        ds = _load(dc.ultrachat_id, None, split, revs[dc.ultrachat_id])
        out = []
        for r in ds:
            msgs = r["messages"]
            if len(msgs) < 2 or msgs[0]["role"] != "user" or msgs[1]["role"] != "assistant":
                continue
            p = normalize(msgs[0]["content"])
            out.append({"id": uid(p), "prompt": p, "reference_response": normalize(msgs[1]["content"])})
        return out

    def clean(rows, name):
        n0 = len(rows)
        rows = [r for r in rows if len(r["prompt"]) >= dc.min_prompt_chars]
        n1 = len(rows)
        rows = list({r["id"]: r for r in rows}.values())
        n2 = len(rows)
        rows = [r for r in rows if not (ngrams(r["prompt"], dc.contamination_ngram) & eval_ngrams)]
        n3 = len(rows)
        if student_tokenizer is not None:
            rows = [r for r in rows if len(student_tokenizer(r["prompt"]).input_ids) <= dc.max_prompt_tokens]
        log[name] = {"raw": n0, "after_min_chars": n1, "after_exact_dedup": n2,
                     "after_contamination_filter": n3, "after_length_filter": len(rows)}
        return rows

    uc_train = clean(uc_rows(dc.ultrachat_train_split), "ultrachat_train")
    uc_dev = clean(uc_rows(dc.ultrachat_dev_split), "ultrachat_dev")
    train_ids = {r["id"] for r in uc_train}
    uc_dev = [r for r in uc_dev if r["id"] not in train_ids]  # cross-split leakage
    uc_train = stable_sample(uc_train, dc.n_ultrachat_train, seed)
    uc_dev = stable_sample(uc_dev, dc.n_ultrachat_dev, seed)
    gsm_teacher_n0 = len(gsm_teacher)
    gsm_teacher = [r for r in gsm_teacher if not (ngrams(r["question"], dc.contamination_ngram) & eval_ngrams)]
    log["gsm8k_teacher"] = {"sampled": gsm_teacher_n0, "after_contamination_filter": len(gsm_teacher)}
    calib = stable_sample(uc_train, dc.n_calibration, seed + 1)

    teacher_prompts = ([{"id": f"uc_{r['id']}", "source": "ultrachat", "prompt": r["prompt"],
                         "reference_response": r["reference_response"]} for r in uc_train] +
                       [{"id": f"gsm_{r['id']}", "source": "gsm8k_train", "prompt": GSM8K_INSTRUCTION + r["question"],
                         "gold": r["gold"], "reference_response": r["reference_response"]} for r in gsm_teacher])

    splits = {
        "teacher_prompts": teacher_prompts,
        "ultrachat_dev": uc_dev,
        "gsm8k_dev": gsm_dev,
        "calibration": calib,
        "gsm8k_test": gsm_test,
        "ifeval": ifeval,
        "wikitext2_validation": [{"id": "wikitext2_validation", "text": wiki["validation"]}],
        "wikitext2_test": [{"id": "wikitext2_test", "text": wiki["test"]}],
    }
    roles = {"teacher_prompts": "train", "ultrachat_dev": "pilot", "gsm8k_dev": "pilot", "calibration": "train",
             "wikitext2_validation": "pilot", "gsm8k_test": "final", "ifeval": "final", "wikitext2_test": "final"}
    # hard isolation assertions
    final_ids = {r["id"] for r in gsm_test} | {r["id"] for r in ifeval}
    tp_core = {r["id"].split("_", 1)[1] for r in teacher_prompts}
    assert not (tp_core & final_ids), "evaluation prompt leaked into teacher prompts"
    assert not ({r["id"] for r in gsm_dev} & {r["id"] for r in gsm_teacher}), "GSM8K dev/teacher overlap"

    man = {"settings": json.loads(json.dumps(settings, default=list)), "revisions": revs, "seed": seed,
           "filtering": log, "splits": {}}
    for name, rows in splits.items():
        f = data_dir / f"{name}.jsonl"
        tmp = f.with_suffix(".tmp")
        with open(tmp, "w") as fh:
            for r in rows:
                fh.write(json.dumps(r, ensure_ascii=False) + "\n")
        tmp.replace(f)
        man["splits"][name] = {"file": f.name, "n": len(rows), "sha256": sha256_file(f), "role": roles[name]}
    man["sources"] = {
        "ultrachat": {"repo": dc.ultrachat_id, "splits": [dc.ultrachat_train_split, dc.ultrachat_dev_split],
                      "license": "MIT (see dataset card)"},
        "gsm8k": {"repo": dc.gsm8k_id, "config": dc.gsm8k_config, "license": "MIT"},
        "wikitext": {"repo": dc.wikitext_id, "config": dc.wikitext_config, "license": "CC BY-SA 3.0 / GFDL"},
        "ifeval": {"repo": dc.ifeval_id, "license": "Apache-2.0"},
    }
    atomic_write_json(man_path, man)
    return man


def read_split(data_dir: Path, name: str) -> list:
    with open(Path(data_dir) / f"{name}.jsonl") as f:
        return [json.loads(l) for l in f]


# ----------------------------------------------------------------------------- SFT tensors
def chat_prompt(tokenizer, prompt: str, system: str | None = None) -> str:
    msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": prompt}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


def build_sft_example(tokenizer, prompt: str, response: str, seq_len: int) -> dict | None:
    """Assistant-only loss masking with the model's own chat template. Returns None when the prompt
    alone exceeds seq_len (counted by the caller, never silently dropped)."""
    p_txt = chat_prompt(tokenizer, prompt)
    full = tokenizer.apply_chat_template([{"role": "user", "content": prompt},
                                          {"role": "assistant", "content": response}], tokenize=False)
    p_ids = tokenizer(p_txt, add_special_tokens=False).input_ids
    f_ids = tokenizer(full, add_special_tokens=False).input_ids
    if f_ids[:len(p_ids)] != p_ids:
        raise ValueError("Chat template is not prefix-stable; cannot mask the prompt reliably.")
    if len(p_ids) >= seq_len - 1:
        return None
    f_ids = f_ids[:seq_len]
    labels = [-100] * len(p_ids) + f_ids[len(p_ids):]
    return {"input_ids": f_ids, "labels": labels}


def collate(batch: list, pad_id: int) -> dict:
    import torch

    n = max(len(b["input_ids"]) for b in batch)
    ids = torch.full((len(batch), n), pad_id, dtype=torch.long)
    lab = torch.full((len(batch), n), -100, dtype=torch.long)
    am = torch.zeros((len(batch), n), dtype=torch.long)
    for i, b in enumerate(batch):
        L = len(b["input_ids"])
        ids[i, :L] = torch.tensor(b["input_ids"])
        lab[i, :L] = torch.tensor(b["labels"])
        am[i, :L] = 1
    return {"input_ids": ids, "labels": lab, "attention_mask": am}


def build_train_set(tokenizer, records: list, seq_len: int, response_key: str) -> tuple[list, dict]:
    out, stats = [], {"in": len(records), "missing_response": 0, "prompt_too_long": 0}
    for r in records:
        resp = r.get(response_key)
        if not resp:
            stats["missing_response"] += 1
            continue
        ex = build_sft_example(tokenizer, r["prompt"], resp, seq_len)
        if ex is None:
            stats["prompt_too_long"] += 1
            continue
        ex["id"] = r["id"]
        out.append(ex)
    stats["out"] = len(out)
    return out, stats


def calibration_batches(tokenizer, records: list, seq_len: int, bs: int, pad_id: int) -> list:
    exs = [build_sft_example(tokenizer, r["prompt"], r["reference_response"], seq_len) for r in records]
    exs = [e for e in exs if e is not None]
    return [collate(exs[i:i + bs], pad_id) for i in range(0, len(exs), bs)]

**Module `teacher.py`**.

In [ ]:
%%writefile {PKG}/jlc/teacher.py
"""Teacher response generation with fixed decoding and persistent, resumable caching.

Every prompt ends up in exactly one record with a status:
  ok         : finished with EOS
  truncated  : hit max_new_tokens (kept in the cache; excluded from training only if configured,
               and the count is reported)
  empty      : produced no text
  error      : generation raised (message stored)
  wrong_answer (GSM8K train only): final answer != gold label -> excluded from training (reported)
Nothing is silently dropped.
"""
from __future__ import annotations

import json
import time
from pathlib import Path

from .ckpt import atomic_write_json, now, sha256_file
from .data import chat_prompt


def _shards(cache: Path) -> list[Path]:
    return sorted(cache.glob("shard_*.jsonl"))


def cached_ids(cache: Path) -> set:
    ids = set()
    for s in _shards(cache):
        with open(s) as f:
            for line in f:
                try:
                    r = json.loads(line)
                    if r.get("status") != "error":   # errored prompts are retried on resume
                        ids.add(r["id"])
                except Exception:  # noqa: BLE001  (a torn last line from a crash is ignored, then regenerated)
                    pass
    return ids


def load_cache(cache: Path) -> dict:
    out = {}
    for s in _shards(cache):
        with open(s) as f:
            for line in f:
                try:
                    r = json.loads(line)
                    if r["id"] not in out or out[r["id"]].get("status") == "error":
                        out[r["id"]] = r          # a later successful retry supersedes an error
                except Exception:  # noqa: BLE001
                    pass
    return out


def _decoding(cfg) -> dict:
    g = cfg.teacher_gen
    return {"max_new_tokens": g.max_new_tokens, "temperature": g.temperature, "top_p": g.top_p,
            "do_sample": g.temperature > 0, "system_prompt": g.system_prompt}


def generate_teacher_targets(cfg, prompts: list, cache_dir: Path, teacher_revision: str | None = None,
                             load_fn=None) -> dict:
    """Generate (or resume) teacher responses for `prompts` into jsonl shards in cache_dir."""
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    meta_p = cache_dir / "generation_config.json"
    dec = _decoding(cfg)
    backend = resolve_backend(cfg)
    meta = {"teacher_id": cfg.model.teacher_id, "teacher_revision": teacher_revision, "decoding": dec,
            "backend": backend,
            "teacher_load_4bit": bool(cfg.model.teacher_load_4bit) if backend == "hf" else False}
    if meta_p.exists():
        old = json.loads(meta_p.read_text())
        if {k: old.get(k) for k in meta} != meta:
            raise RuntimeError(f"Teacher cache at {cache_dir} was produced with different settings:\n{old}\n"
                               f"vs now:\n{meta}\nUse a new cache directory instead of mixing targets.")
    else:
        atomic_write_json(meta_p, {**meta, "created_utc": now()})

    done = cached_ids(cache_dir)
    todo = [p for p in prompts if p["id"] not in done]
    print(f"[teacher] {len(done)} cached, {len(todo)} to generate")
    if not todo:
        return summarize_cache(cfg, cache_dir, prompts)

    print(f"[teacher] backend = {backend}")
    gen = _VLLM(cfg, teacher_revision) if backend == "vllm" else _HF(cfg, teacher_revision, load_fn)
    shard_i = len(_shards(cache_dir))
    ss = cfg.teacher_gen.shard_size
    for start in range(0, len(todo), ss):
        chunk = todo[start:start + ss]
        t0 = time.time()
        recs = gen.generate(chunk)
        shard = cache_dir / f"shard_{shard_i:05d}.jsonl"
        tmp = shard.with_suffix(".tmp")
        with open(tmp, "w") as f:
            for r in recs:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        tmp.replace(shard)  # a shard is either complete or absent
        shard_i += 1
        print(f"[teacher] shard {shard.name}: {len(recs)} records in {time.time() - t0:.0f}s "
              f"({start + len(chunk)}/{len(todo)})")
    gen.close()
    return summarize_cache(cfg, cache_dir, prompts)


def resolve_backend(cfg) -> str:
    backend = cfg.teacher_gen.backend
    if backend == "auto":
        try:
            import torch
            import vllm  # noqa: F401

            backend = "vllm" if torch.cuda.is_available() else "hf"
        except Exception:  # noqa: BLE001
            backend = "hf"
    return backend


def _final_number(text: str) -> str | None:
    import re

    m = re.findall(r"####\s*\$?\s*(-?[\d,]*\.?\d+)", text)
    if m:
        return m[-1].replace(",", "").rstrip(".")
    nums = re.findall(r"-?\d[\d,]*\.?\d*", text)
    return nums[-1].replace(",", "").rstrip(".") if nums else None


def numbers_equal(a: str | None, b: str | None) -> bool:
    if a is None or b is None:
        return False
    try:
        return abs(float(a) - float(b)) < 1e-6
    except ValueError:
        return a.strip() == b.strip()


def summarize_cache(cfg, cache_dir: Path, prompts: list) -> dict:
    cache = load_cache(cache_dir)
    missing = [p["id"] for p in prompts if p["id"] not in cache]
    counts = {}
    usable = []
    for p in prompts:
        r = cache.get(p["id"])
        if r is None:
            continue
        st = r["status"]
        if st == "ok" and p.get("gold") is not None and not numbers_equal(_final_number(r["response"]), p["gold"]):
            st = "wrong_answer"
        counts[st] = counts.get(st, 0) + 1
        keep = st == "ok" or (st == "truncated" and not cfg.teacher_gen.exclude_truncated_from_training)
        if keep:
            usable.append({**p, "teacher_response": r["response"]})
    summary = {"n_prompts": len(prompts), "n_cached": len(cache), "missing": len(missing), "status_counts": counts,
               "n_usable_for_training": len(usable),
               "exclusion_policy": {"truncated_excluded": cfg.teacher_gen.exclude_truncated_from_training,
                                    "gsm8k_wrong_answer_excluded": True, "empty_and_error_excluded": True},
               "shards": {s.name: sha256_file(s) for s in _shards(cache_dir)}}
    atomic_write_json(Path(cache_dir) / "summary.json", summary)
    if missing:
        print(f"[teacher] WARNING: {len(missing)} prompts have no cached record (re-run generation)")
    return {"summary": summary, "usable": usable}


class _HF:
    def __init__(self, cfg, revision, load_fn):
        import torch

        from .models import load_model

        self.cfg = cfg
        self.model, self.tok = (load_fn or load_model)(cfg.model.teacher_id, revision, role="teacher", cfg=cfg)
        self.tok.padding_side = "left"
        if self.tok.pad_token_id is None:
            self.tok.pad_token = self.tok.eos_token
        self.torch = torch

    def generate(self, chunk):
        torch, g = self.torch, self.cfg.teacher_gen
        out = []
        bs = g.batch_size
        i = 0
        while i < len(chunk):
            batch = chunk[i:i + bs]
            texts = [chat_prompt(self.tok, p["prompt"], g.system_prompt) for p in batch]
            try:
                enc = self.tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(self.model.device)
                with torch.no_grad():
                    gen = self.model.generate(**enc, max_new_tokens=g.max_new_tokens, do_sample=g.temperature > 0,
                                              temperature=g.temperature if g.temperature > 0 else None,
                                              top_p=g.top_p if g.temperature > 0 else None,
                                              pad_token_id=self.tok.pad_token_id)
                new = gen[:, enc.input_ids.shape[1]:]
                eos = set(self.model.generation_config.eos_token_id if isinstance(
                    self.model.generation_config.eos_token_id, list) else [self.tok.eos_token_id])
                for p, row in zip(batch, new):
                    ids = row.tolist()
                    ended = any(t in eos for t in ids)
                    if ended:
                        ids = ids[:min(ids.index(t) for t in eos if t in ids)]
                    txt = self.tok.decode(ids, skip_special_tokens=True).strip()
                    st = "empty" if not txt else ("ok" if ended else "truncated")
                    out.append({"id": p["id"], "response": txt, "status": st, "n_tokens": len(ids)})
                i += bs
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                if bs == 1:
                    for p in batch:
                        out.append({"id": p["id"], "response": "", "status": "error", "error": "OOM at batch 1"})
                    i += 1
                    continue
                bs = max(1, bs // 2)
                print(f"[teacher] OOM -> batch size {bs}")
            except Exception as e:  # noqa: BLE001
                for p in batch:
                    out.append({"id": p["id"], "response": "", "status": "error", "error": repr(e)[:500]})
                i += len(batch)
        return out

    def close(self):
        del self.model
        from .env import cleanup

        cleanup()


class _VLLM:
    def __init__(self, cfg, revision):
        from vllm import LLM, SamplingParams

        self.cfg = cfg
        g = cfg.teacher_gen
        self.llm = LLM(model=cfg.model.teacher_id, revision=revision, dtype="auto", seed=0,
                       gpu_memory_utilization=0.85, max_model_len=cfg.data.max_prompt_tokens + g.max_new_tokens + 64)
        self.tok = self.llm.get_tokenizer()
        self.sp = SamplingParams(temperature=g.temperature, top_p=g.top_p, max_tokens=g.max_new_tokens, seed=0)

    def generate(self, chunk):
        g = self.cfg.teacher_gen
        texts = [chat_prompt(self.tok, p["prompt"], g.system_prompt) for p in chunk]
        out = []
        try:
            res = self.llm.generate(texts, self.sp, use_tqdm=False)
            for p, r in zip(chunk, res):
                o = r.outputs[0]
                txt = o.text.strip()
                st = "empty" if not txt else ("truncated" if o.finish_reason == "length" else "ok")
                out.append({"id": p["id"], "response": txt, "status": st, "n_tokens": len(o.token_ids),
                            "finish_reason": o.finish_reason})
        except Exception as e:  # noqa: BLE001
            out = [{"id": p["id"], "response": "", "status": "error", "error": repr(e)[:500]} for p in chunk]
        return out

    def close(self):
        del self.llm
        from .env import cleanup

        cleanup()

**Module `evaluate.py`** — GSM8K, IFEval (official checkers from lm-eval), llama.cpp-compatible perplexity, size/load/RAM measurements.

In [ ]:
%%writefile {PKG}/jlc/evaluate.py
"""Quality and compression metrics, all computed from actual model outputs.

Per-item outputs are saved for every evaluation so that paired statistics (Section 21) and audits
are possible: results/items/<exp>/seed_<s>/<task>.jsonl
"""
from __future__ import annotations

import json
import math
import os
import re
import subprocess
import sys
import time
from pathlib import Path

import torch

from .data import GSM8K_INSTRUCTION, chat_prompt
from .teacher import _final_number, numbers_equal


@torch.no_grad()
def generate(model, tok, prompts: list[str], max_new_tokens: int, batch_size: int = 16) -> list[dict]:
    """Greedy chat generation. Returns text + whether EOS was reached (truncation is a format failure)."""
    tok.padding_side = "left"
    dev = next(model.parameters()).device
    eos = model.generation_config.eos_token_id
    eos = set(eos if isinstance(eos, list) else [eos if eos is not None else tok.eos_token_id])
    out, bs, i = [], batch_size, 0
    while i < len(prompts):
        chunk = prompts[i:i + bs]
        texts = [chat_prompt(tok, p) for p in chunk]
        try:
            enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(dev)
            g = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None,
                               top_k=None, pad_token_id=tok.pad_token_id)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if bs == 1:
                raise
            bs = max(1, bs // 2)
            continue
        for row in g[:, enc.input_ids.shape[1]:].tolist():
            hit = [row.index(t) for t in eos if t in row]
            ended = bool(hit)
            row = row[:min(hit)] if hit else row
            out.append({"text": tok.decode(row, skip_special_tokens=True).strip(), "eos": ended, "n_tokens": len(row)})
        i += len(chunk)
    return out


def eval_gsm8k(model, tok, items: list, cfg) -> tuple[dict, list]:
    gens = generate(model, tok, [GSM8K_INSTRUCTION + r["question"] for r in items],
                    cfg.eval.gsm8k_max_new_tokens, cfg.eval.eval_batch_size)
    rows = []
    for r, g in zip(items, gens):
        pred = _final_number(g["text"])
        rows.append({"id": r["id"], "gold": r["gold"], "pred": pred, "correct": numbers_equal(pred, r["gold"]),
                     "format_valid": bool(re.search(r"####\s*-?[\d,]*\.?\d+", g["text"])) and g["eos"],
                     "truncated": not g["eos"], "n_tokens": g["n_tokens"], "output": g["text"]})
    n = len(rows)
    return ({"gsm8k_em": sum(r["correct"] for r in rows) / n,
             "gsm8k_format_valid": sum(r["format_valid"] for r in rows) / n,
             "gsm8k_truncated": sum(r["truncated"] for r in rows), "gsm8k_n": n}, rows)


def eval_ifeval(model, tok, items: list, cfg) -> tuple[dict, list]:
    try:
        from lm_eval.tasks.ifeval import utils as ifu
    except Exception as e:  # noqa: BLE001
        raise RuntimeError("IFEval scoring needs `pip install lm_eval[ifeval]` (official instruction checkers). "
                           f"Import failed: {e}. IFEval is reported as NOT COMPUTED, never as 0.") from e
    gens = generate(model, tok, [r["prompt"] for r in items], cfg.eval.ifeval_max_new_tokens, cfg.eval.eval_batch_size)
    rows = []
    for r, g in zip(items, gens):
        doc = {"key": r["key"], "prompt": r["prompt"], "instruction_id_list": r["instruction_id_list"],
               "kwargs": r["kwargs"]}
        try:
            res = ifu.process_results(doc, [g["text"]])
            err = None
        except Exception as e:  # noqa: BLE001   checker crash counts as failure, and is recorded
            res = {"prompt_level_strict_acc": False, "inst_level_strict_acc": [False] * len(r["instruction_id_list"]),
                   "prompt_level_loose_acc": False, "inst_level_loose_acc": [False] * len(r["instruction_id_list"])}
            err = repr(e)[:300]
        rows.append({"id": r["id"], **res, "truncated": not g["eos"], "format_valid": bool(g["text"]) and g["eos"],
                     "checker_error": err, "output": g["text"]})
    n = len(rows)
    inst_s = [x for r in rows for x in r["inst_level_strict_acc"]]
    inst_l = [x for r in rows for x in r["inst_level_loose_acc"]]
    return ({"ifeval_prompt_strict": sum(r["prompt_level_strict_acc"] for r in rows) / n,
             "ifeval_prompt_loose": sum(r["prompt_level_loose_acc"] for r in rows) / n,
             "ifeval_inst_strict": sum(inst_s) / len(inst_s), "ifeval_inst_loose": sum(inst_l) / len(inst_l),
             "ifeval_format_valid": sum(r["format_valid"] for r in rows) / n,
             "ifeval_checker_errors": sum(r["checker_error"] is not None for r in rows), "ifeval_n": n}, rows)


@torch.no_grad()
def perplexity(model, tok, text: str, ctx: int = 512, max_chunks: int | None = None, add_bos: bool | None = None) -> tuple[dict, list]:
    """llama.cpp `llama-perplexity` protocol: non-overlapping chunks of `ctx` tokens; only the second
    half of each chunk is scored. Matching the protocol lets PyTorch and GGUF numbers be compared
    directly (Section 18 parity check)."""
    dev = next(model.parameters()).device
    ids = tok(text, add_special_tokens=False, return_tensors="pt").input_ids[0]
    if add_bos is None:
        add_bos = tok.bos_token_id is not None and bool(getattr(tok, "add_bos_token", False))
    n_chunks = len(ids) // ctx
    if max_chunks:
        n_chunks = min(n_chunks, max_chunks)
    first = ctx // 2
    nll_chunks = []
    for c in range(n_chunks):
        x = ids[c * ctx:(c + 1) * ctx].clone()
        if add_bos:
            x[0] = tok.bos_token_id
        logits = model(x[None].to(dev)).logits[0].float()
        lp = torch.log_softmax(logits[first:ctx - 1], -1)
        tgt = x[first + 1:ctx].to(dev)
        nll = -lp.gather(-1, tgt[:, None]).squeeze(-1)
        nll_chunks.append({"chunk": c, "nll_sum": nll.sum().item(), "n": int(nll.numel())})
    tot = sum(r["nll_sum"] for r in nll_chunks)
    n = sum(r["n"] for r in nll_chunks)
    return {"ppl": math.exp(tot / max(n, 1)), "ppl_tokens": n, "ppl_chunks": n_chunks, "ppl_ctx": ctx,
            "ppl_add_bos": add_bos}, nll_chunks


# ----------------------------------------------------------------------------- compression stats
def model_stats(model) -> dict:
    c = model.config
    total = sum(p.numel() for p in model.parameters())
    nonzero = sum(int((p != 0).sum()) for p in model.parameters())
    return {"params_total": total, "params_trainable": sum(p.numel() for p in model.parameters() if p.requires_grad),
            "params_nonzero": nonzero, "n_layers": c.num_hidden_layers, "hidden": c.hidden_size,
            "q_heads": c.num_attention_heads, "kv_heads": c.num_key_value_heads,
            "head_dim": getattr(c, "head_dim", None) or c.hidden_size // c.num_attention_heads,
            "intermediate": c.intermediate_size, "vocab": c.vocab_size}


def dir_size_bytes(d: str | Path, pattern: str = "*.safetensors") -> int:
    return sum(p.stat().st_size for p in Path(d).glob(pattern))


_LOAD_PROBE = r"""
import json, sys, time
t0 = time.perf_counter()
import torch
from transformers import AutoModelForCausalLM
t1 = time.perf_counter()
if sys.argv[1] == "--baseline":
    print(json.dumps({"load_s": 0.0, "import_s": t1 - t0})); sys.exit(0)
m = AutoModelForCausalLM.from_pretrained(sys.argv[1], torch_dtype=getattr(torch, sys.argv[2]))
t2 = time.perf_counter()
with torch.no_grad():
    m(torch.tensor([[1, 2, 3, 4]]))
print(json.dumps({"load_s": t2 - t1, "import_s": t1 - t0}))
"""


def load_time_and_peak_ram(model_dir: str | Path, dtype: str = "bfloat16", repeats: int = 3) -> dict:
    """PyTorch checkpoint load time and peak RSS, measured in a fresh CPU-only process launched through
    env.run_measured (so the notebook's own memory is not counted). Peak RSS includes the Python +
    torch + transformers runtime, which is measured separately (`pt_runtime_baseline_rss_mb`) and
    subtracted in `pt_model_rss_mb`. GGUF/llama.cpp RAM is
    measured separately in Section 20."""
    from .env import run_measured

    res = []
    old = os.environ.get("CUDA_VISIBLE_DEVICES")
    os.environ["CUDA_VISIBLE_DEVICES"] = ""
    try:
        for _ in range(repeats):
            r = run_measured([sys.executable, "-c", _LOAD_PROBE, str(model_dir), dtype], quiet=False, timeout=1800)
            if r["exit"] != 0:
                raise RuntimeError(f"load probe failed: {r['stderr'][-800:]}")
            res.append({**json.loads(r["stdout"].strip().splitlines()[-1]), "peak_rss_mb": r["peak_rss_mb"]})
        base = run_measured([sys.executable, "-c", _LOAD_PROBE, "--baseline", dtype], quiet=False, timeout=1800)
    finally:
        if old is None:
            os.environ.pop("CUDA_VISIBLE_DEVICES", None)
        else:
            os.environ["CUDA_VISIBLE_DEVICES"] = old
    lt = sorted(x["load_s"] for x in res)
    peak = max(x["peak_rss_mb"] for x in res)
    return {"pt_load_s_median": lt[len(lt) // 2], "pt_peak_rss_mb": peak,
            "pt_runtime_baseline_rss_mb": base["peak_rss_mb"], "pt_model_rss_mb": peak - base["peak_rss_mb"],
            "pt_load_repeats": repeats}


def save_items(root: Path, exp: str, seed: int, task: str, rows: list) -> Path:
    d = Path(root) / "results" / "items" / exp / f"seed_{seed}"
    d.mkdir(parents=True, exist_ok=True)
    f = d / f"{task}.jsonl"
    with open(f, "w") as fh:
        for r in rows:
            fh.write(json.dumps(r, ensure_ascii=False, default=str) + "\n")
    return f

**Module `quant.py`** — GGUF-exact Q4_0/Q8_0 grids, STE fake-quantizer, torchao alternative, PTQ snapping (used by training and Sections 12-13).

In [ ]:
%%writefile {PKG}/jlc/quant.py
"""Quantization: deployment-exact GGUF grids, fake quantization for QAT, and PTQ snapping.

Design decision (documented in Section 13):
  The deployed artifact is a llama.cpp GGUF file (Q4_0 / Q8_0). QAT is only useful if the grid
  simulated during training is the grid used at deployment. We therefore implement the Q4_0 and
  Q8_0 block quantizers in PyTorch *bit-exactly* (verified in tests against the official gguf-py
  reference implementation shipped with llama.cpp) and use a straight-through estimator (STE).

  Backend 'q4_0_exact'   : the above (default).
  Backend 'torchao_intx' : torchao.quantization.qat.IntxFakeQuantizer (int4, symmetric, group 32),
                           a supported library fake-quantizer. Its grid differs slightly from Q4_0
                           (scale = max|w|/7.5 vs signed-max/-8); the mismatch is measured, not assumed.

  'PTQ' in this project = round-to-nearest onto the same grid with no training (what
  `llama-quantize Q4_0` does), optionally with an importance matrix (llama.cpp imatrix) as a
  stronger baseline. PTQ is never labelled QAT.
"""
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F

QK = 32  # GGUF block size for Q4_0 / Q8_0


# ----------------------------------------------------------------------------- exact grids
def _blocks(w: torch.Tensor) -> torch.Tensor:
    if w.shape[-1] % QK:
        raise ValueError(f"Row length {w.shape[-1]} not divisible by {QK}; GGUF Q4_0/Q8_0 cannot store it.")
    return w.reshape(-1, QK)


def q4_0_dequant(w: torch.Tensor) -> torch.Tensor:
    """Bit-exact torch port of gguf-py Q4_0.quantize_blocks + dequantize_blocks."""
    shape, dtype = w.shape, w.dtype
    b = _blocks(w.float())
    imax = b.abs().argmax(dim=-1, keepdim=True)
    mx = torch.gather(b, -1, imax)
    d = mx / -8.0
    inv = torch.where(d == 0, torch.zeros_like(d), 1.0 / d)
    q = torch.trunc(b * inv + 8.5).clamp(0, 15)
    d16 = d.half().float()
    return ((q - 8.0) * d16).reshape(shape).to(dtype)


def q8_0_dequant(w: torch.Tensor) -> torch.Tensor:
    """Bit-exact torch port of gguf-py Q8_0 (d = amax/127, q = round(x/d))."""
    shape, dtype = w.shape, w.dtype
    b = _blocks(w.float())
    d = b.abs().amax(dim=-1, keepdim=True) / 127.0
    inv = torch.where(d == 0, torch.zeros_like(d), 1.0 / d)
    q = torch.round(b * inv)  # numpy np.round == round-half-to-even == torch.round
    return (q * d.half().float()).reshape(shape).to(dtype)


GRIDS = {"Q4_0": q4_0_dequant, "Q8_0": q8_0_dequant}


class _STE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, w, fn):  # noqa: D401
        return fn(w)

    @staticmethod
    def backward(ctx, g):
        return g, None


def fake_quant_q4_0(w: torch.Tensor) -> torch.Tensor:
    return _STE.apply(w, q4_0_dequant)


# ----------------------------------------------------------------------------- QAT linear
class FakeQuantLinear(nn.Linear):
    """nn.Linear whose weight is fake-quantized in forward when `fq_enabled` is True.

    Subclassing nn.Linear (instead of wrapping) keeps `.weight` a plain Parameter, so the
    physical pruning code can slice it and save_pretrained produces a vanilla checkpoint.
    """

    fq_enabled: bool = True

    def setup_fq(self, backend: str = "q4_0_exact", group_size: int = 32):
        self.fq_backend = backend
        self.fq_enabled = True
        if backend == "torchao_intx":
            from torchao.quantization.qat import IntxFakeQuantizeConfig
            from torchao.quantization.qat.fake_quantizer import IntxFakeQuantizer

            # object.__setattr__: keep the quantizer out of _modules so state_dict stays vanilla
            object.__setattr__(self, "_tao", IntxFakeQuantizer(IntxFakeQuantizeConfig(
                torch.int4, group_size=group_size, is_symmetric=True)))
        elif backend != "q4_0_exact":
            raise ValueError(f"Unknown QAT backend {backend}")
        return self

    def quantized_weight(self) -> torch.Tensor:
        if self.fq_backend == "q4_0_exact":
            return fake_quant_q4_0(self.weight)
        return self._tao(self.weight)

    def forward(self, x):
        w = self.quantized_weight() if self.fq_enabled else self.weight
        return F.linear(x, w, self.bias)


def _swap_linear(lin: nn.Linear, backend: str, gs: int) -> FakeQuantLinear:
    """In-place class swap: same object, same Parameters, same registered hooks (pruning gates)."""
    lin.__class__ = FakeQuantLinear
    return lin.setup_fq(backend, gs)


def prepare_qat(model: nn.Module, backend: str = "q4_0_exact", group_size: int = 32,
                exclude: tuple = ("embed_tokens", "lm_head")) -> list[str]:
    """Replace every decoder nn.Linear with FakeQuantLinear in place. Returns the swapped names."""
    if group_size != QK and backend == "q4_0_exact":
        raise ValueError("q4_0_exact requires group_size=32")
    swapped = []
    for full, child in list(model.named_modules()):
        if type(child) is nn.Linear and not any(e in full for e in exclude):
            _swap_linear(child, backend, group_size)
            swapped.append(full)
    if not swapped:
        raise RuntimeError("prepare_qat found no nn.Linear modules to fake-quantize")
    return swapped


def set_fake_quant(model: nn.Module, enabled: bool) -> None:
    for m in model.modules():
        if isinstance(m, FakeQuantLinear):
            m.fq_enabled = enabled


def qat_active(model: nn.Module) -> bool:
    return any(isinstance(m, FakeQuantLinear) and m.fq_enabled for m in model.modules())


def convert_qat(model: nn.Module, grid: str = "Q4_0") -> int:
    """End of QAT: turn FakeQuantLinear back into nn.Linear holding weights snapped to the
    deployment grid (fp16/bf16 storage, INT4-equivalent values). Returns #converted."""
    n = 0
    for _, m in list(model.named_modules()):
        if isinstance(m, FakeQuantLinear):
            for attr in ("_tao", "fq_backend", "fq_enabled"):
                m.__dict__.pop(attr, None)
            m.__class__ = nn.Linear
            with torch.no_grad():
                m.weight.data.copy_(GRIDS[grid](m.weight.data))
            n += 1
    return n


@torch.no_grad()
def snap_model(model: nn.Module, grid: str = "Q4_0", exclude: tuple = ("embed_tokens", "lm_head"),
               embed_grid: str | None = "Q8_0") -> dict:
    """PTQ by round-to-nearest onto a GGUF grid, in place. Mirrors `llama-quantize <grid>` with
    --token-embedding-type/--output-tensor-type = embed_grid. Returns per-tensor relative error."""
    errs = {}
    fn = GRIDS[grid]
    for name, mod in model.named_modules():
        if isinstance(mod, nn.Linear) and not any(e in name for e in exclude):
            w = mod.weight.data
            q = fn(w)
            errs[name] = ((q.float() - w.float()).norm() / w.float().norm().clamp_min(1e-12)).item()
            mod.weight.data.copy_(q)
    if embed_grid:
        emb = model.get_input_embeddings()
        emb.weight.data.copy_(GRIDS[embed_grid](emb.weight.data))
        out = model.get_output_embeddings()
        if out is not None and out.weight.data_ptr() != emb.weight.data_ptr():
            out.weight.data.copy_(GRIDS[embed_grid](out.weight.data))
    return errs


def quant_record(cfg_quant, kind: str, calib: str | None = None) -> dict:
    """Metadata required by the protocol for every quantized artifact."""
    import importlib

    rec = {
        "kind": kind,                                  # ptq_rtn | ptq_imatrix | qat
        "weight_bits": 4 if "4" in kind or kind == "qat" else 8,
        "activation_bits": cfg_quant.a_bits or 16,
        "group_size": QK,
        "scaling": "per-block symmetric, fp16 scale (GGUF Q4_0: d=signed_max/-8; Q8_0: d=amax/127)",
        "qat_backend": cfg_quant.qat_backend if kind == "qat" else None,
        "calibration_data": calib,
        "unquantized_or_excluded": list(cfg_quant.exclude) + ["all RMSNorm weights (f32 in GGUF)"],
        "excluded_layers_storage": cfg_quant.gguf_embedding_type,
        "export_format": "GGUF",
        "runtime": "llama.cpp (CPU, arm64 NEON on Android)",
    }
    for pkg in ("torch", "torchao", "gguf"):
        try:
            rec[f"{pkg}_version"] = importlib.import_module(pkg).__version__
        except Exception:  # noqa: BLE001
            rec[f"{pkg}_version"] = None
    return rec

**Module `train.py`** — the single training engine and the joint objective.

In [ ]:
%%writefile {PKG}/jlc/train.py
"""One training engine for every condition (distillation, pruning recovery, QAT, joint, alternating).

Objective (Section 1, implemented in `_loss`):

    L_joint = λ_KD · L_KD(student_Q)            token-level KD from a vocab-compatible logit teacher
            + λ_CE · L_CE(student_FP)           next-token CE through the full-precision forward
            + λ_SP · R_SP(gates)                structured-sparsity regularizer (GateBank.regularizer)
            + λ_Q  · L_CE(student_Q)            next-token CE through the fake-quantized forward

  * student_Q is the forward with FakeQuantLinear active; when QAT is off, student_Q == student_FP,
    L_Q is undefined and λ_CE is replaced by λ_CE_noqat.
  * Targets (`labels`) are the teacher's responses for distillation conditions (response-level KD)
    and the original dataset responses for non-distillation conditions.
  * λ_KD is forced to 0 when no compatible logit teacher exists (never silently approximated).

Budget control: every compression stage runs exactly `spec.steps` optimizer steps with the same
effective batch, data order (per seed) and LR schedule. The final checkpoint is always the last
step (no checkpoint selection on evaluation data).
"""
from __future__ import annotations

import json
import math
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path

import torch
import torch.nn.functional as F

from . import prune as P
from . import quant as Q
from .ckpt import latest_train_ckpt, load_train_ckpt, log_failure, save_train_ckpt
from .data import collate
from .env import cleanup, set_seed


@dataclass
class TrainSpec:
    name: str
    steps: int
    use_logit_kd: bool = False
    qat: bool = False
    gates: bool = False                    # learn structured gates, then hard-prune at hard_prune_at
    alternating: bool = False              # E9: alternate gate phases (FP) and QAT phases (gates frozen)
    hard_prune_at: float = 0.6
    kv_keep: int | None = None
    mlp_keep: int | None = None
    qat_aware_gating: bool = True          # H4 ablation: False => fake-quant OFF while gates are learned
    loss_overrides: dict = field(default_factory=dict)


def kd_loss(s_logits, t_logits, T: float, mode: str = "forward_kl"):
    s = F.log_softmax(s_logits.float() / T, -1)
    t = F.log_softmax(t_logits.float() / T, -1)
    if mode == "forward_kl":
        loss = (t.exp() * (t - s)).sum(-1)
    elif mode == "reverse_kl":
        loss = (s.exp() * (s - t)).sum(-1)
    elif mode == "jsd":
        m = torch.logsumexp(torch.stack([s, t]), 0) - math.log(2)
        loss = 0.5 * ((s.exp() * (s - m)).sum(-1) + (t.exp() * (t - m)).sum(-1))
    else:
        raise ValueError(mode)
    return loss.mean() * T * T


def _shifted(logits, labels):
    lg = logits[:, :-1]
    lb = labels[:, 1:]
    sel = lb != -100
    return lg[sel], lb[sel], sel


def _make_optimizer(model, gates, cfg, lr_scale: float = 1.0):
    t = cfg.train
    params = [p for p in model.parameters() if p.requires_grad]
    groups = [{"params": params, "lr": t.lr * lr_scale, "weight_decay": t.weight_decay}]
    if gates is not None:
        groups.append({"params": list(gates.parameters()), "lr": t.gate_lr, "weight_decay": 0.0})
    kind = t.optimizer
    if kind == "auto":
        kind = "adamw"
        if torch.cuda.is_available():
            try:
                import bitsandbytes  # noqa: F401

                kind = "adamw8bit"
            except Exception:  # noqa: BLE001
                pass
    if kind == "adamw8bit":
        import bitsandbytes as bnb

        return bnb.optim.AdamW8bit(groups, betas=(0.9, 0.95)), kind
    return torch.optim.AdamW(groups, betas=(0.9, 0.95), fused=torch.cuda.is_available()), kind


def _lr_lambda(total: int, warmup_ratio: float):
    w = max(1, int(total * warmup_ratio))

    def f(step):
        if step < w:
            return (step + 1) / w
        prog = (step - w) / max(1, total - w)
        return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1.0, prog)))
    return f


class Trainer:
    def __init__(self, model, tokenizer, train_examples: list, spec: TrainSpec, cfg, out_dir: Path, seed: int,
                 log_path: Path, logit_teacher=None, dev_examples: list | None = None, root: Path | None = None):
        self.model, self.tok, self.data, self.spec, self.cfg = model, tokenizer, train_examples, spec, cfg
        self.out_dir, self.seed, self.log_path, self.teacher = Path(out_dir), seed, Path(log_path), logit_teacher
        self.dev, self.root = dev_examples or [], root
        self.L = {**asdict(cfg.loss), **spec.loss_overrides}
        if spec.use_logit_kd and logit_teacher is None:
            raise ValueError("use_logit_kd=True but no logit teacher supplied")
        if not spec.use_logit_kd:
            self.L["l_kd"] = 0.0
        self.mb, self.ga = cfg.train.micro_batch, cfg.train.grad_accum
        self.dev_ = next(model.parameters()).device
        self.gates = None
        self.plan = None
        self.struct = None
        self.events = []

    # ------------------------------------------------------------------ helpers
    def _log(self, rec: dict):
        self.log_path.parent.mkdir(parents=True, exist_ok=True)
        with open(self.log_path, "a") as f:
            f.write(json.dumps(rec, default=float) + "\n")

    def _order(self, epoch: int) -> list[int]:
        g = torch.Generator().manual_seed(self.seed * 1000 + epoch)
        return torch.randperm(len(self.data), generator=g).tolist()

    def _batches(self, start_seq: int):
        """Infinite deterministic stream of micro-batches starting at global sequence index."""
        n = len(self.data)
        if n == 0:
            raise RuntimeError("empty training set")
        idx = start_seq
        while True:
            ep, pos = divmod(idx, n)
            order = self._order(ep)
            take = []
            while len(take) < self.mb:
                if pos >= n:
                    ep, pos = ep + 1, 0
                    order = self._order(ep)
                take.append(order[pos])
                pos += 1
            idx += self.mb
            yield idx, collate([self.data[i] for i in take], self.tok.pad_token_id)

    def _phase(self, step: int) -> dict:
        """Which components are active at `step` (pre-hard-prune vs post; alternating sub-phases)."""
        s = self.spec
        hard = int(round(s.hard_prune_at * s.steps)) if s.gates else -1
        pre = s.gates and self.gates is not None and step < hard
        ph = {"gates_trainable": pre, "fq": s.qat, "r_sp": pre, "hard_step": hard}
        if pre and s.qat and not s.qat_aware_gating:
            ph["fq"] = False
        if pre and s.alternating:
            k = self.cfg.alt.cycle_steps
            prune_phase = ((step // k) % 2) == 0
            ph.update({"gates_trainable": prune_phase, "r_sp": prune_phase, "fq": (not prune_phase) and s.qat,
                       "alt_phase": "prune" if prune_phase else "quant"})
        if s.gates and self.gates is not None and hard > 0:
            frac = min(1.0, step / max(1, hard))
            pc = self.cfg.prune
            self.gates.temperature = pc.gate_temperature_start * (pc.gate_temperature_end /
                                                                   pc.gate_temperature_start) ** frac
        return ph

    def _loss(self, batch, ph) -> tuple[torch.Tensor, dict]:
        L, m = self.L, self.model
        ids, am, lab = (batch["input_ids"].to(self.dev_), batch["attention_mask"].to(self.dev_),
                        batch["labels"].to(self.dev_))
        terms = {}
        total = 0.0
        qat_on = self.spec.qat and ph["fq"]
        Q.set_fake_quant(m, qat_on)
        out = m(input_ids=ids, attention_mask=am).logits
        lg, lb, sel = _shifted(out, lab)
        ce = F.cross_entropy(lg.float(), lb)
        if qat_on:
            terms["L_Q"] = ce.detach()
            total = total + L["l_q"] * ce
            if L["l_ce"] > 0:
                Q.set_fake_quant(m, False)
                lg_fp, lb_fp, _ = _shifted(m(input_ids=ids, attention_mask=am).logits, lab)
                ce_fp = F.cross_entropy(lg_fp.float(), lb_fp)
                Q.set_fake_quant(m, True)
                terms["L_CE"] = ce_fp.detach()
                total = total + L["l_ce"] * ce_fp
        else:
            terms["L_CE"] = ce.detach()
            total = total + L["l_ce_noqat"] * ce
        if L["l_kd"] > 0:
            with torch.no_grad():
                t_out = self.teacher(input_ids=ids, attention_mask=am).logits
            t_lg = t_out[:, :-1][sel]
            kd = kd_loss(lg, t_lg, L["kd_temperature"], L["kd_mode"])
            terms["L_KD"] = kd.detach()
            total = total + L["l_kd"] * kd
        if ph["r_sp"] and self.gates is not None:
            r = self.gates.regularizer(self.kv_target, self.mlp_target)
            terms["R_SP"] = r.detach()
            total = total + L["l_sp"] * r
        terms["n_tokens"] = int(sel.sum())
        return total, terms

    # ------------------------------------------------------------------ hard prune
    def _hard_prune(self, step: int, imp: dict | None = None):
        plan = self.gates.to_plan(self.spec.kv_keep, self.spec.mlp_keep, tiebreak=imp)
        kvv, mlpv = self.gates.gate_values_for(plan)
        self.gates.remove()
        self.gates = None
        info = P.physically_prune(self.model, plan, fold_gates=(kvv, mlpv))
        plan.meta.update({"hard_prune_step": step, **info})
        self.plan = plan
        self._log({"event": "hard_prune", "step": step, **info})
        self.events.append({"hard_prune_step": step})

    # ------------------------------------------------------------------ main loop
    def run(self, init_plan_json: dict | None = None) -> dict:
        cfg, spec, model = self.cfg, self.spec, self.model
        set_seed(self.seed)
        if spec.gates:
            self.struct = P.inspect_structure(model)
            self.kv_target = spec.kv_keep / self.struct.n_kv
            self.mlp_target = spec.mlp_keep / self.struct.inter
        train_dir = self.out_dir / "train"
        ck = latest_train_ckpt(train_dir)
        start_step = 0
        if ck is not None:
            st_meta = json.loads((ck / "META.json").read_text())
            print(f"[train:{spec.name}] resuming from {ck.name}")
            extra = _peek_extra(ck)
            if extra.get("plan"):
                # hard prune already happened before this checkpoint -> rebuild the pruned shapes first
                self.plan = P.PrunePlan.from_json(extra["plan"])
                P.physically_prune(model, self.plan)
            start_step = st_meta["step"]
        if spec.qat:
            Q.prepare_qat(model, cfg.quant.qat_backend, cfg.quant.group_size, cfg.quant.exclude)
        if spec.gates and self.plan is None:
            self.gates = P.GateBank(model, self.struct, init=cfg.prune.gate_init)
        model.config.use_cache = False
        if cfg.train.gradient_checkpointing and hasattr(model, "gradient_checkpointing_enable"):
            model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        model.train()
        opt, opt_kind = _make_optimizer(model, self.gates, cfg)
        sched = torch.optim.lr_scheduler.LambdaLR(opt, [_lr_lambda(spec.steps, cfg.train.warmup_ratio)] * len(opt.param_groups))
        if ck is not None:
            st = load_train_ckpt(ck, model, optimizer=None)
            ex = st["extra"]
            if self.gates is not None and ex.get("gates") is not None:
                self.gates.load_state_dict(ex["gates"])
            if ex.get("mb"):
                self.mb, self.ga = ex["mb"], ex["ga"]
            opt, opt_kind = _make_optimizer(model, self.gates, cfg)
            if (ck / "optimizer.pt").exists() and ex.get("n_param_groups") == len(opt.param_groups):
                try:
                    opt.load_state_dict(torch.load(ck / "optimizer.pt", map_location="cpu", weights_only=True))
                except Exception as e:  # noqa: BLE001
                    print(f"[train] optimizer state not restored ({e}); moments re-initialised (logged)")
                    self._log({"event": "optimizer_state_not_restored", "step": start_step, "error": str(e)})
            sched = torch.optim.lr_scheduler.LambdaLR(opt, [_lr_lambda(spec.steps, cfg.train.warmup_ratio)] * len(opt.param_groups))
            for _ in range(start_step):
                sched.step()
        seq_cursor = start_step * self.mb * self.ga
        stream = self._batches(seq_cursor)
        self._log({"event": "start", "spec": asdict(spec), "start_step": start_step, "optimizer": opt_kind,
                   "loss_weights": self.L, "seed": self.seed, "n_train_examples": len(self.data)})
        t0, tok_count = time.time(), 0
        step = start_step
        while step < spec.steps:
            ph = self._phase(step)
            if spec.gates and self.gates is not None and step == ph["hard_step"]:
                self._hard_prune(step)
                opt, opt_kind = _make_optimizer(model, None, cfg)   # shapes changed: fresh moments (all methods alike)
                sched = torch.optim.lr_scheduler.LambdaLR(opt, [_lr_lambda(spec.steps, cfg.train.warmup_ratio)])
                for _ in range(step):
                    sched.step()
                ph = self._phase(step)
            if self.gates is not None:
                for p in self.gates.parameters():
                    p.requires_grad_(ph["gates_trainable"])
            try:
                opt.zero_grad(set_to_none=True)
                agg = {}
                for _ in range(self.ga):
                    seq_cursor, batch = next(stream)
                    loss, terms = self._loss(batch, ph)
                    (loss / self.ga).backward()
                    for k, v in terms.items():
                        agg[k] = agg.get(k, 0.0) + (float(v) / self.ga if k != "n_tokens" else v)
                    tok_count += terms["n_tokens"]
                gn = torch.nn.utils.clip_grad_norm_([p for g in opt.param_groups for p in g["params"]],
                                                    cfg.train.max_grad_norm)
                if not torch.isfinite(gn):
                    self._log({"event": "nonfinite_grad", "step": step})
                    raise FloatingPointError(f"non-finite gradient norm at step {step}")
                opt.step()
                sched.step()
            except torch.cuda.OutOfMemoryError as e:
                opt.zero_grad(set_to_none=True)
                cleanup()
                if self.mb == 1:
                    if self.root:
                        log_failure(self.root, f"train:{spec.name}", e, {"step": step})
                    raise
                # keep the effective batch identical; only the micro-batch split changes
                self.mb, self.ga = self.mb // 2, self.ga * 2
                self._log({"event": "oom_rebatch", "step": step, "micro_batch": self.mb, "grad_accum": self.ga})
                print(f"[train] OOM -> micro_batch={self.mb}, grad_accum={self.ga} (effective batch unchanged)")
                stream = self._batches(step * cfg.train.micro_batch * cfg.train.grad_accum)
                continue
            step += 1
            if step % cfg.train.log_every == 0 or step == spec.steps:
                rec = {"step": step, "lr": sched.get_last_lr()[0], "grad_norm": float(gn),
                       "tok_per_s": tok_count / max(1e-9, time.time() - t0), **agg,
                       **{k: v for k, v in ph.items() if k != "hard_step"}}
                if self.gates is not None:
                    rec.update({f"gate_{k}": v for k, v in self.gates.stats().items()})
                if torch.cuda.is_available():
                    rec["gpu_mem_gb"] = torch.cuda.max_memory_allocated() / 2**30
                self._log(rec)
                print(f"[{spec.name}] step {step}/{spec.steps} " +
                      " ".join(f"{k}={v:.4g}" for k, v in agg.items() if k != "n_tokens"))
            if self.dev and (step % cfg.train.dev_eval_every == 0 or step == spec.steps):
                self._log({"step": step, "dev_loss": self.dev_loss(ph)})
            if step % cfg.train.ckpt_every == 0 and step < spec.steps:
                extra = {"plan": self.plan.to_json() if self.plan else None,
                         "gates": self.gates.state_dict() if self.gates is not None else None,
                         "mb": self.mb, "ga": self.ga, "n_param_groups": len(opt.param_groups)}
                save_train_ckpt(train_dir, step, _unwrapped_for_save(model), extra,
                                optimizer=opt if cfg.train.save_optimizer_state else None,
                                keep_last=cfg.train.keep_last_ckpts, local_scratch=cfg.paths.local_root)
        if self.gates is not None:   # hard_prune_at >= 1.0 edge case
            self._hard_prune(step)
        if cfg.train.gradient_checkpointing and hasattr(model, "gradient_checkpointing_disable"):
            model.gradient_checkpointing_disable()
        model.config.use_cache = True
        n_conv = 0
        if spec.qat:
            n_conv = Q.convert_qat(model, "Q4_0")   # weights now lie exactly on the deployment grid
        model.eval()
        self._log({"event": "end", "step": step, "qat_converted_layers": n_conv})
        return {"steps": step, "plan": self.plan.to_json() if self.plan else None, "optimizer": opt_kind,
                "micro_batch_final": self.mb, "grad_accum_final": self.ga, "qat_converted_layers": n_conv,
                "events": self.events}

    @torch.no_grad()
    def dev_loss(self, ph) -> float:
        was = self.model.training
        self.model.eval()
        Q.set_fake_quant(self.model, self.spec.qat and ph["fq"])
        tot, n = 0.0, 0
        for i in range(0, min(len(self.dev), 64), self.mb):
            b = collate(self.dev[i:i + self.mb], self.tok.pad_token_id)
            lg = self.model(input_ids=b["input_ids"].to(self.dev_),
                            attention_mask=b["attention_mask"].to(self.dev_)).logits
            l, y, _ = _shifted(lg, b["labels"].to(self.dev_))
            tot += F.cross_entropy(l.float(), y, reduction="sum").item()
            n += y.numel()
        self.model.train(was)
        return tot / max(n, 1)


def _peek_extra(ck: Path) -> dict:
    """Read only the non-tensor part of a checkpoint's extra state (safe load)."""
    from .ckpt import _safe_torch_load

    return _safe_torch_load(ck / "state.pt")["extra"]


def _unwrapped_for_save(model):
    return model

**Module `experiments.py`** — the experimental matrix as a resumable stage graph, evaluation and export drivers.

In [ ]:
%%writefile {PKG}/jlc/experiments.py
"""Experimental matrix E0-E9 (+ controls and ablations) as a resumable stage graph.

Budget accounting (one stage = S optimizer steps, identical batch/LR schedule/data order per seed):

  id     start  stages after start                                          trained steps  targets
  E0     base   -                                                                0          -
  R1     base   CE fine-tune on reference responses (non-KD control)             S          reference
  E1     base   response-level KD (+token-level KD iff teacher vocab compatible) S          teacher
  E2     R1     one-shot structured prune -> CE recovery                         2S         reference
  E3     E0     INT4 PTQ (Q4_0 round-to-nearest)                                 0          -
  E3_8   E0     INT8 PTQ (Q8_0)                                                  0          -
  E4     E1     one-shot prune -> KD recovery (logit teacher = E1)               2S         teacher
  E5     E1     QAT(Q4_0) + KD                                                   2S         teacher
  E6     R1     one-shot prune -> CE recovery under QAT                          2S         reference
  E7     E4     INT4 PTQ (sequential KD -> prune -> PTQ)                         2S         teacher
  E8     E1     joint: gates + QAT + KD in one objective, hard prune at 60%      2S         teacher
  E9     E1     alternating: prune-phase(FP, gates) <-> quant-phase(QAT)        2S         teacher
  ablations: A_H4_noQAI (E8 with FP gating), A_E8_noKD (λ_KD=0), A_E8_dualCE (λ_CE=1),
             A_E5_torchao (torchao fake-quant), A_E4_heads / A_E4_mlp (single-axis pruning)

E7 vs E8 is the central comparison: same start (E1), same teacher, same data, same S steps,
same final structure size and the same deployment grid; only the optimisation schedule differs.
"""
from __future__ import annotations

import copy
import json
import time
import traceback
from dataclasses import asdict, dataclass, field
from pathlib import Path

import torch

from . import prune as P
from . import quant as Q
from .ckpt import (atomic_write_json, is_done, log_failure, mark_done, read_done, save_model_validated,
                   stage_dir)
from .data import build_train_set, calibration_batches, read_split
from .env import cleanup, set_seed
from .models import load_local, load_model
from .train import Trainer, TrainSpec


@dataclass
class Cond:
    exp: str
    start: str                         # 'base' or another exp id
    kind: str                          # train | prune_train | ptq | none
    targets: str = "teacher"           # teacher | reference
    use_logit_kd: bool = False
    qat: bool = False
    gates: bool = False
    alternating: bool = False
    qat_aware_gating: bool = True
    prune_which: str = "both"
    ptq_grid: str = "Q4_0"
    loss_overrides: dict = field(default_factory=dict)
    qat_backend: str | None = None
    deploy: str = "F16"                # GGUF type of the deployed artifact
    description: str = ""


CONDITIONS = {
    "E0": Cond("E0", "base", "none", deploy="F16", description="uncompressed student"),
    "R1": Cond("R1", "base", "train", targets="reference", deploy="F16",
               description="control: CE fine-tune on original responses (budget match for non-KD arms)"),
    "E1": Cond("E1", "base", "train", targets="teacher", use_logit_kd=None, deploy="F16",
               description="distillation only"),
    "E2": Cond("E2", "R1", "prune_train", targets="reference", deploy="F16", description="structured pruning only"),
    "E3": Cond("E3", "E0", "ptq", ptq_grid="Q4_0", deploy="Q4_0", description="INT4 PTQ only"),
    "E3_8": Cond("E3_8", "E0", "ptq", ptq_grid="Q8_0", deploy="Q8_0", description="INT8 PTQ only"),
    "E4": Cond("E4", "E1", "prune_train", targets="teacher", use_logit_kd=True, deploy="F16",
               description="distillation + pruning"),
    "E5": Cond("E5", "E1", "train", targets="teacher", use_logit_kd=True, qat=True, deploy="Q4_0",
               description="distillation + QAT"),
    "E6": Cond("E6", "R1", "prune_train", targets="reference", qat=True, deploy="Q4_0",
               description="pruning + QAT"),
    "E7": Cond("E7", "E4", "ptq", ptq_grid="Q4_0", deploy="Q4_0",
               description="sequential: distill -> prune -> INT4 PTQ"),
    "E8": Cond("E8", "E1", "train", targets="teacher", use_logit_kd=True, qat=True, gates=True, deploy="Q4_0",
               description="joint distillation + structured pruning + QAT"),
    "E9": Cond("E9", "E1", "train", targets="teacher", use_logit_kd=True, qat=True, gates=True, alternating=True,
               deploy="Q4_0", description="alternating optimisation"),
    # ---- ablations
    "A_H4_noQAI": Cond("A_H4_noQAI", "E1", "train", targets="teacher", use_logit_kd=True, qat=True, gates=True,
                       qat_aware_gating=False, deploy="Q4_0",
                       description="E8 with fake-quant OFF while gates are learned (tests H4)"),
    "A_E8_noKD": Cond("A_E8_noKD", "E1", "train", targets="teacher", use_logit_kd=False, qat=True, gates=True,
                      deploy="Q4_0", description="E8 without token-level KD (response targets only)"),
    "A_E8_dualCE": Cond("A_E8_dualCE", "E1", "train", targets="teacher", use_logit_kd=True, qat=True, gates=True,
                        loss_overrides={"l_ce": 1.0}, deploy="Q4_0",
                        description="E8 with full-precision CE term (two forwards per step)"),
    "A_E5_torchao": Cond("A_E5_torchao", "E1", "train", targets="teacher", use_logit_kd=True, qat=True,
                         qat_backend="torchao_intx", deploy="Q4_0",
                         description="E5 with torchao IntxFakeQuantizer (grid != Q4_0; mismatch measured)"),
    "A_E4_heads": Cond("A_E4_heads", "E1", "prune_train", targets="teacher", use_logit_kd=True, prune_which="heads",
                       deploy="F16", description="E4 pruning attention heads only"),
    "A_E4_mlp": Cond("A_E4_mlp", "E1", "prune_train", targets="teacher", use_logit_kd=True, prune_which="mlp",
                     deploy="F16", description="E4 pruning MLP channels only"),
}
MAIN = ["E0", "R1", "E1", "E2", "E3", "E3_8", "E4", "E5", "E6", "E7", "E8", "E9"]
ABLATIONS = ["A_H4_noQAI", "A_E8_noKD", "A_E8_dualCE", "A_E5_torchao", "A_E4_heads", "A_E4_mlp"]
SEED_INDEPENDENT = {"E0", "E3", "E3_8"}   # deterministic: computed once, shared by all seeds


def budget_table(cfg) -> list[dict]:
    S = cfg.train.stage_steps
    out = []
    for e, c in CONDITIONS.items():
        own = 0 if c.kind in ("none", "ptq") else S
        inh, cur = 0, c.start
        while cur != "base":
            p = CONDITIONS[cur]
            inh += 0 if p.kind in ("none", "ptq") else S
            cur = p.start
        out.append({"exp": e, "start": c.start, "own_steps": own, "inherited_steps": inh, "total_steps": own + inh,
                    "targets": c.targets if c.kind not in ("none", "ptq") else "-", "deploy": c.deploy,
                    "description": c.description})
    return out


class Runner:
    def __init__(self, cfg, root: Path, ctx: dict):
        """ctx: student_revision, kd_compat (dict), teacher_usable (list of records with teacher_response)"""
        self.cfg, self.root, self.ctx = cfg, Path(root), ctx
        self.data_dir = self.root / "data"

    # ------------------------------------------------------------------ paths
    @property
    def ck_root(self) -> Path:
        """Checkpoints are keyed by the config hash: pilot iterations with different settings never
        reuse each other's models, and the final protocol has its own namespace."""
        return self.root / "runs_by_config" / self.cfg.hash()

    def final_dir(self, exp: str, seed: int) -> Path:
        s = 0 if exp in SEED_INDEPENDENT else seed
        return stage_dir(self.ck_root, exp, s, "final")

    def _load_start(self, start: str, seed: int):
        if start == "base":
            return load_model(self.cfg.model.student_id, self.ctx["student_revision"], cfg=self.cfg)
        d = self.final_dir(start, seed)
        if not is_done(d):
            raise RuntimeError(f"{start} (seed {seed}) must be completed first: {d}")
        return load_local(d, self.cfg)

    def _train_records(self, targets: str) -> list:
        if targets == "teacher":
            return [{**r, "_resp": r["teacher_response"]} for r in self.ctx["teacher_usable"]]
        prompts = read_split(self.data_dir, "teacher_prompts")
        return [{**r, "_resp": r["reference_response"]} for r in prompts]

    # ------------------------------------------------------------------ stages
    def ensure(self, exp: str, seed: int) -> Path:
        """Make sure `exp` for `seed` exists on Drive (recursively building its start point)."""
        c = CONDITIONS[exp]
        out = self.final_dir(exp, seed)
        if is_done(out, verify=False):
            return out
        if c.start != "base":
            self.ensure(c.start, seed)
        t0 = time.time()
        try:
            set_seed(seed)
            info = getattr(self, f"_stage_{c.kind}")(c, seed, out)
        except Exception as e:
            log_failure(self.root, f"ensure:{exp}:seed{seed}", e, {"cond": asdict(c)})
            atomic_write_json(self.root / "results" / "runs" / f"{exp}_seed{seed}_FAILED.json",
                              {"exp": exp, "seed": seed, "status": "failed", "error": repr(e),
                               "traceback": traceback.format_exc()})
            raise
        finally:
            cleanup()
        print(f"[runner] {exp} seed {seed} done in {(time.time() - t0) / 60:.1f} min")
        return out

    def _stage_none(self, c, seed, out):
        model, tok = self._load_start("base", seed)
        save_model_validated(model, tok, out, local_scratch=self.cfg.paths.local_root,
                             info={"exp": c.exp, "seed": seed, "kind": "none"})

    def _stage_ptq(self, c, seed, out):
        model, tok = self._load_start(c.start, seed)
        errs = Q.snap_model(model, c.ptq_grid, exclude=self.cfg.quant.exclude, embed_grid="Q8_0")
        meta = Q.quant_record(self.cfg.quant, "ptq_rtn_int4" if c.ptq_grid == "Q4_0" else "ptq_rtn_int8")
        save_model_validated(model, tok, out, local_scratch=self.cfg.paths.local_root,
                             extra_files={"quantization.json": {**meta, "mean_rel_weight_err":
                                                                sum(errs.values()) / len(errs)}},
                             info={"exp": c.exp, "seed": seed, "kind": "ptq", "grid": c.ptq_grid})

    def _logit_teacher(self, c, seed):
        if c.use_logit_kd is None:   # E1: only the original teacher, only if vocab-compatible
            if not self.ctx["kd_compat"]["compatible"]:
                return None, "disabled: teacher/student vocab incompatible"
            m, _ = load_model(self.cfg.model.teacher_id, self.ctx.get("teacher_revision"), role="teacher", cfg=self.cfg)
            return m, f"teacher {self.cfg.model.teacher_id}"
        if c.use_logit_kd:
            m, _ = load_local(self.final_dir("E1", seed), self.cfg)
            return m, "E1 student (self-distillation, identical vocab)"
        return None, "off"

    def _pruned(self, model, tok, c, seed, out):
        """One-shot pruning using calibration importance; returns plan + report. The plan is stored in
        the work dir first, so a resumed run re-applies exactly the same structure."""
        pc = self.cfg.prune
        s = P.inspect_structure(model)
        kv_n, mlp_n, kinfo = P.keep_counts(s, pc.head_keep_ratio, pc.mlp_keep_ratio, pc.mlp_multiple,
                                           pc.head_count_policy)
        saved = stage_dir(self.ck_root, c.exp, seed, "work") / "oneshot_plan.json"
        if saved.exists():
            plan = P.PrunePlan.from_json(json.loads(saved.read_text()))
        else:
            calib = calibration_batches(tok, read_split(self.data_dir, "calibration"), self.cfg.train.seq_len,
                                        self.cfg.train.micro_batch, tok.pad_token_id)
            imp = P.compute_importance(model, s, calib, pc.importance)
            plan = P.plan_from_importance(imp, kv_n, mlp_n, s, which=c.prune_which)
            atomic_write_json(saved, plan.to_json())
        mb = P.MaskBank(model, s)
        mb.apply_plan(plan)
        probe = tok("Structured pruning verification probe.", return_tensors="pt").input_ids
        with torch.no_grad():
            ref = model(probe.to(next(model.parameters()).device)).logits.cpu()
        mb.remove()
        rep = P.physically_prune(model, plan)
        ver = P.verify_pruned(model, tok, s, plan, ref_logits=ref, probe_ids=probe)
        plan.meta.update({**kinfo, **rep, "verification": ver})
        (out.parent / "plans").mkdir(parents=True, exist_ok=True)
        atomic_write_json(out.parent / "plans" / f"{c.exp}_plan.json", plan.to_json())
        return plan, s

    def _stage_train(self, c, seed, out, pre_pruned: bool = False):
        model, tok = self._load_start(c.start, seed)
        plan_json = None
        s = P.inspect_structure(model)
        pc = self.cfg.prune
        kv_n, mlp_n, kinfo = P.keep_counts(s, pc.head_keep_ratio, pc.mlp_keep_ratio, pc.mlp_multiple,
                                           pc.head_count_policy)
        if pre_pruned:
            plan, _ = self._pruned(model, tok, c, seed, out)
            plan_json = plan.to_json()
        tcfg = copy.deepcopy(self.cfg)
        if c.qat_backend:
            tcfg.quant.qat_backend = c.qat_backend
        recs = self._train_records(c.targets)
        train, tstats = build_train_set(tok, [{**r, "resp": r["_resp"]} for r in recs], self.cfg.train.seq_len, "resp")
        dev, _ = build_train_set(tok, read_split(self.data_dir, "ultrachat_dev"), self.cfg.train.seq_len,
                                 "reference_response")
        teacher, kd_note = self._logit_teacher(c, seed)
        spec = TrainSpec(name=f"{c.exp}_s{seed}", steps=self.cfg.train.stage_steps,
                         use_logit_kd=teacher is not None, qat=c.qat, gates=c.gates, alternating=c.alternating,
                         hard_prune_at=pc.hard_prune_at, kv_keep=kv_n, mlp_keep=mlp_n,
                         qat_aware_gating=c.qat_aware_gating, loss_overrides=c.loss_overrides)
        work = stage_dir(self.ck_root, c.exp, seed, "work")
        tr = Trainer(model, tok, train, spec, tcfg, work, seed,
                     self.root / "logs" / f"{c.exp}_seed{seed}.jsonl", logit_teacher=teacher, dev_examples=dev,
                     root=self.root)
        res = tr.run()
        if teacher is not None:
            del teacher
            cleanup()
        if res["plan"]:
            plan_json = res["plan"]
            (out.parent / "plans").mkdir(parents=True, exist_ok=True)
            atomic_write_json(out.parent / "plans" / f"{c.exp}_plan.json", plan_json)
        extra = {"train_result.json": {**res, "data_stats": tstats, "logit_kd": kd_note, "spec": asdict(spec),
                                       "keep_info": kinfo}}
        if c.qat:
            extra["quantization.json"] = Q.quant_record(tcfg.quant, "qat", calib=None)
        save_model_validated(model, tok, out, local_scratch=self.cfg.paths.local_root, extra_files=extra,
                             info={"exp": c.exp, "seed": seed, "kind": c.kind, "logit_kd": kd_note,
                                   "pruned": bool(plan_json)})

    def _stage_prune_train(self, c, seed, out):
        return self._stage_train(c, seed, out, pre_pruned=True)


# =============================================================================== evaluation
def _eval_sets(runner, split):
    ec = runner.cfg.eval
    if split == "final":
        return (read_split(runner.data_dir, "gsm8k_test")[:ec.gsm8k_n], read_split(runner.data_dir, "ifeval")[:ec.ifeval_n],
                read_split(runner.data_dir, "wikitext2_test")[0]["text"])
    return (read_split(runner.data_dir, "gsm8k_dev")[:ec.gsm8k_n], [],
            read_split(runner.data_dir, "wikitext2_validation")[0]["text"])


def _eval_core(runner, model, tok, split, tag, seed) -> dict:
    from .evaluate import eval_gsm8k, eval_ifeval, perplexity, save_items

    cfg, root = runner.cfg, runner.root
    gsm, ife, wiki = _eval_sets(runner, split)
    rec = {}
    m, rows = eval_gsm8k(model, tok, gsm, cfg)
    rec.update(m)
    save_items(root, tag, seed, "gsm8k", rows)
    if ife:
        try:
            m, rows = eval_ifeval(model, tok, ife, cfg)
            rec.update(m)
            save_items(root, tag, seed, "ifeval", rows)
        except RuntimeError as e:
            rec["ifeval_status"] = f"NOT COMPUTED: {e}"
    m, rows = perplexity(model, tok, wiki, cfg.eval.ppl_ctx, cfg.eval.ppl_max_chunks)
    rec.update({f"wikitext2_{k}": v for k, v in m.items()})
    save_items(root, tag, seed, "wikitext2", rows)
    return rec


def _check_split(cfg, root, split):
    from .config import is_frozen

    frozen = is_frozen(cfg, root / "results")
    if split == "final" and not frozen:
        raise RuntimeError("Final held-out evaluation requires a frozen protocol (Section 16 -> freeze_protocol).")
    if split == "pilot" and frozen:
        raise RuntimeError("Protocol is frozen: pilot evaluation is closed (prevents tuning on reported numbers).")


def _tag(cfg, exp, split):
    return exp if split == "final" else f"{exp}__pilot_{cfg.hash()}"


def evaluate_condition(runner: Runner, exp: str, seed: int, split: str = "final", measure_load: bool = True) -> dict:
    """Evaluate a finished condition. split='pilot' uses only pilot data and is refused after the
    protocol is frozen; split='final' uses held-out data and is refused before it is frozen.
    Existing result files are returned as-is (idempotent re-runs)."""
    from .evaluate import dir_size_bytes, load_time_and_peak_ram, model_stats

    cfg, root = runner.cfg, runner.root
    tag = _tag(cfg, exp, split)
    out_f = root / "results" / "runs" / f"{tag}_seed{seed}.json"
    if out_f.exists():
        return json.loads(out_f.read_text())
    _check_split(cfg, root, split)
    if exp in SEED_INDEPENDENT and seed != 0:
        base = evaluate_condition(runner, exp, 0, split, measure_load)
        rec = {**base, "seed": seed, "replicated_from_seed": 0,
               "note": "deterministic condition (no training, greedy decoding): evaluated once, replicated"}
        for task in ("gsm8k", "ifeval", "wikitext2"):
            src = root / "results" / "items" / tag / "seed_0" / f"{task}.jsonl"
            if src.exists():
                dst = root / "results" / "items" / tag / f"seed_{seed}"
                dst.mkdir(parents=True, exist_ok=True)
                (dst / f"{task}.jsonl").write_text(src.read_text())
        atomic_write_json(out_f, rec)
        return rec
    d = runner.ensure(exp, seed)
    model, tok = load_local(d, cfg)
    rec = {"exp": exp, "seed": seed, "split": split, "status": "ok", "config_hash": cfg.hash(),
           "model_dir": str(d), **model_stats(model), "pt_safetensors_mb": dir_size_bytes(d) / 2**20}
    try:
        rec.update(_eval_core(runner, model, tok, split, tag, seed))
        del model
        cleanup()
        if measure_load:
            rec.update(load_time_and_peak_ram(d, "bfloat16" if "bfloat16" in cfg.model.dtype else "float16"))
    except Exception as e:  # noqa: BLE001
        rec["status"] = "failed"
        rec["error"] = repr(e)
        log_failure(root, f"evaluate:{exp}:seed{seed}", e)
    for f in ("train_result.json", "quantization.json"):
        if (Path(d) / f).exists():
            rec[f.replace(".json", "")] = json.loads((Path(d) / f).read_text())
    rec["done_info"] = read_done(d)
    atomic_write_json(out_f, rec)
    return rec


def evaluate_teacher(runner: Runner, split: str = "final") -> dict:
    """Reference numbers for the teacher (not a compression condition; excluded from tests)."""
    cfg, root = runner.cfg, runner.root
    tag = _tag(cfg, "TEACHER", split)
    out_f = root / "results" / "runs" / f"{tag}_seed0.json"
    if out_f.exists():
        return json.loads(out_f.read_text())
    _check_split(cfg, root, split)
    model, tok = load_model(cfg.model.teacher_id, runner.ctx.get("teacher_revision"), role="teacher", cfg=cfg)
    rec = {"exp": "TEACHER", "seed": 0, "split": split, "status": "ok", "config_hash": cfg.hash(),
           "model_id": cfg.model.teacher_id, "load_info": getattr(model, "_jlc_load_info", {})}
    try:
        rec.update(_eval_core(runner, model, tok, split, tag, 0))
    except Exception as e:  # noqa: BLE001
        rec.update({"status": "failed", "error": repr(e)})
        log_failure(root, "evaluate:TEACHER", e)
    del model
    cleanup()
    atomic_write_json(out_f, rec)
    return rec


# =============================================================================== export
def export_condition(runner: Runner, exp: str, seed: int, llama: dict, parity_chunks: int = 20) -> dict:
    """HF checkpoint -> F16 GGUF -> deployed type; audit; grid identity; llama.cpp parity."""
    from . import export as X
    from .evaluate import perplexity

    cfg, root = runner.cfg, runner.root
    c = CONDITIONS[exp]
    s = 0 if exp in SEED_INDEPENDENT else seed
    out = root / "exports" / cfg.hash() / exp / f"seed_{s}"
    rep_f = out / "export.json"
    if rep_f.exists():
        return json.loads(rep_f.read_text())
    out.mkdir(parents=True, exist_ok=True)
    d = runner.ensure(exp, seed)
    local = Path(cfg.paths.local_root) / "gguf"
    local.mkdir(parents=True, exist_ok=True)
    f16 = X.convert_to_gguf(llama, d, local / f"{exp}_s{s}.f16.gguf")
    if c.deploy == "F16":
        final = f16
    else:
        final = X.quantize_gguf(llama, f16, c.deploy, local / f"{exp}_s{s}.{c.deploy.lower()}.gguf",
                                embed_type=cfg.quant.gguf_embedding_type)
    audit = X.audit_gguf(final, c.deploy)
    if audit["violations"]:
        raise RuntimeError(f"GGUF tensor-type audit failed for {exp}: {audit['violations'][:5]}")
    model, tok = load_local(d, cfg)
    grid = X.grid_identity(final, model)
    wiki_txt = root / "data" / "wikitext2_test.raw"
    if not wiki_txt.exists():
        wiki_txt.write_text(read_split(runner.data_dir, "wikitext2_test")[0]["text"])
    ab = audit["metadata"].get("tokenizer.ggml.add_bos_token")
    add_bos = None if ab is None else bool(ab[0] if isinstance(ab, list) else ab)   # match llama.cpp
    pt, _ = perplexity(model, tok, wiki_txt.read_text(), cfg.eval.ppl_ctx, parity_chunks, add_bos=add_bos)
    del model
    cleanup()
    gg = X.llama_perplexity(llama, final, wiki_txt, cfg.eval.ppl_ctx, parity_chunks, cfg.bench.threads)
    parity = X.parity_report(pt["ppl"], gg["gguf_ppl"], cfg.bench.parity_ppl_rel_tol)
    dst = out / final.name
    import shutil

    shutil.copy2(final, dst)
    rep = {"exp": exp, "seed": s, "deploy_type": c.deploy, "gguf": str(dst), "artifact_mb": audit["file_bytes"] / 2**20,
           "audit": audit, "grid_identity": grid, "parity": parity, "llama_cpp_commit": llama["commit"]}
    atomic_write_json(rep_f, rep)
    if not parity["pass"]:
        print(f"[export] WARNING {exp}: PyTorch/GGUF perplexity differ by {parity['rel_diff']:.2%} "
              f"(> {parity['tolerance']:.0%}). Benchmark numbers for this artifact are flagged.")
    return rep


def collect_runs(root: Path, split: str = "final", config_hash: str | None = None) -> "pd.DataFrame":
    import pandas as pd

    rows = []
    for f in sorted((Path(root) / "results" / "runs").glob("*.json")):
        r = json.loads(f.read_text())
        is_pilot = "__pilot" in f.name
        if (split == "pilot") != is_pilot:
            continue
        if config_hash and r.get("config_hash") not in (config_hash, None):
            continue
        flat = {k: v for k, v in r.items() if not isinstance(v, (dict, list))}
        if "FAILED" in f.name:
            flat["status"] = "failed"
        ex = Path(root) / "exports" / str(r.get("config_hash")) / r["exp"] / \
            f"seed_{0 if r['exp'] in SEED_INDEPENDENT else r['seed']}" / "export.json"
        if ex.exists():
            e = json.loads(ex.read_text())
            flat.update({"artifact_mb": e["artifact_mb"], "deploy_type": e["deploy_type"],
                         "parity_pass": e["parity"]["pass"], "parity_rel_diff": e["parity"]["rel_diff"],
                         "grid_max_abs_diff": e["grid_identity"]["max_abs_diff"]})
        rows.append(flat)
    return pd.DataFrame(rows)

**Module `pipeline.py`** — phase-aware orchestration helpers.

In [ ]:
%%writefile {PKG}/jlc/pipeline.py
"""Notebook-level orchestration helpers: run a list of conditions for the current phase, report
status, never hide failures."""
from __future__ import annotations

import time

import pandas as pd

from .ckpt import is_done
from .experiments import CONDITIONS, evaluate_condition


def seeds_for(cfg, phase: str) -> list[int]:
    return [cfg.train.pilot_seed] if phase == "pilot" else list(cfg.train.seeds)


def run_conditions(runner, exps: list[str], phase: str, stop_on_error: bool = False) -> pd.DataFrame:
    """Train/build each condition for every seed of the phase. In the pilot phase each finished model
    is also evaluated on PILOT data (dev splits). Failures are recorded and shown, then the loop
    continues with independent conditions (a dependent condition fails with a clear message)."""
    rows = []
    for exp in exps:
        for seed in seeds_for(runner.cfg, phase):
            t0 = time.time()
            r = {"exp": exp, "seed": seed, "description": CONDITIONS[exp].description}
            try:
                d = runner.ensure(exp, seed)
                r["status"] = "done"
                r["model_dir"] = str(d)
                if phase == "pilot":
                    ev = evaluate_condition(runner, exp, seed, split="pilot", measure_load=False)
                    r.update({k: ev.get(k) for k in ("gsm8k_em", "wikitext2_ppl", "params_total")})
                    r["status"] = "done+pilot_eval" if ev.get("status") == "ok" else "eval_failed"
            except Exception as e:  # noqa: BLE001   (already logged to logs/failures.jsonl)
                r["status"] = "FAILED"
                r["error"] = f"{e.__class__.__name__}: {str(e)[:300]}"
                if stop_on_error:
                    rows.append(r)
                    print(pd.DataFrame(rows).to_string())
                    raise
            r["minutes"] = round((time.time() - t0) / 60, 2)
            rows.append(r)
            print(f"  -> {exp} seed {seed}: {r['status']} ({r['minutes']} min)")
    df = pd.DataFrame(rows)
    n_fail = int((df["status"] == "FAILED").sum()) if len(df) else 0
    if n_fail:
        print(f"\n!!! {n_fail} run(s) FAILED - see the 'error' column and logs/failures.jsonl. "
              "Fix and re-run this cell; completed runs are skipped automatically.")
    return df


def stage_status(runner, exps: list[str], seeds: list[int]) -> pd.DataFrame:
    rows = []
    for e in exps:
        for s in seeds:
            d = runner.final_dir(e, s)
            rows.append({"exp": e, "seed": s, "done": is_done(d), "path": str(d)})
    return pd.DataFrame(rows)

**Module `analysis.py`**.

In [ ]:
%%writefile {PKG}/jlc/analysis.py
"""Structured-pruning sensitivity analysis (Section 9). Uses permanent MASKS (function-equivalent to
physical pruning, verified in Section 10) so no model copies are needed. Evaluated on pilot data only
(WikiText-2 validation), never on held-out test data."""
from __future__ import annotations

import pandas as pd
import torch

from . import prune as P
from .evaluate import perplexity


def _plan(imp, s, kv_n, mlp_n, which):
    return P.plan_from_importance(imp, kv_n, mlp_n, s, which=which)


@torch.no_grad()
def _ppl(model, tok, text, ctx, chunks):
    return perplexity(model, tok, text, ctx, chunks)[0]["ppl"]


def global_sensitivity(model, tok, calib, text, ratios, methods=("activation", "taylor", "magnitude"),
                       ctx=512, chunks=16, multiple=64) -> pd.DataFrame:
    s = P.inspect_structure(model)
    base = _ppl(model, tok, text, ctx, chunks)
    rows = [{"method": "-", "which": "none", "keep_ratio": 1.0, "ppl": base}]
    for meth in methods:
        imp = P.compute_importance(model, s, calib, meth)
        mb = P.MaskBank(model, s)
        try:
            for r in ratios:
                kv_n, mlp_n, info = P.keep_counts(s, r, r, multiple, policy="any")
                for which in ("heads", "mlp", "both"):
                    mb.reset()
                    mb.apply_plan(_plan(imp, s, kv_n, mlp_n, which))
                    rows.append({"method": meth, "which": which, "keep_ratio": r,
                                 "eff_head_keep": info["effective_head_keep"] if which != "mlp" else 1.0,
                                 "eff_mlp_keep": info["effective_mlp_keep"] if which != "heads" else 1.0,
                                 "ppl": _ppl(model, tok, text, ctx, chunks)})
        finally:
            mb.remove()
    df = pd.DataFrame(rows)
    df["ppl_ratio_vs_dense"] = df["ppl"] / base
    return df


def layer_sensitivity(model, tok, calib, text, keep: float = 0.5, method: str = "activation", ctx=512,
                      chunks=8, multiple=64) -> pd.DataFrame:
    """Prune ONE layer at a time (heads, then MLP) at `keep` and record the perplexity increase."""
    s = P.inspect_structure(model)
    base = _ppl(model, tok, text, ctx, chunks)
    imp = P.compute_importance(model, s, calib, method)
    kv_n, mlp_n, _ = P.keep_counts(s, keep, keep, multiple, policy="any")
    full = P.PrunePlan([list(range(s.n_kv))] * s.n_layers, [list(range(s.inter))] * s.n_layers)
    rows = []
    mb = P.MaskBank(model, s)
    try:
        for layer in range(s.n_layers):
            for which in ("heads", "mlp"):
                plan = P.PrunePlan([k[:] for k in full.kv_keep], [m[:] for m in full.mlp_keep])
                if which == "heads":
                    plan.kv_keep[layer] = sorted(torch.topk(imp["kv"][layer], kv_n).indices.tolist())
                else:
                    plan.mlp_keep[layer] = sorted(torch.topk(imp["mlp"][layer], mlp_n).indices.tolist())
                mb.reset()
                mb.apply_plan(plan)
                p = _ppl(model, tok, text, ctx, chunks)
                rows.append({"layer": layer, "which": which, "keep": keep, "ppl": p, "delta_ppl": p - base})
    finally:
        mb.remove()
    return pd.DataFrame(rows)

**Module `export.py`**.

In [ ]:
%%writefile {PKG}/jlc/export.py
"""GGUF export with llama.cpp, exported-model validation, and Android build/benchmark scripts.

Validation of every exported file (Section 18):
  1. tensor-type audit with gguf-py (decoder linears == requested type, embeddings == q8_0, norms f32)
  2. grid identity: dequantized GGUF tensors vs the PyTorch weights (exact for QAT/snapped models)
  3. load + perplexity with llama.cpp's own `llama-perplexity`, compared with the PyTorch number
     computed under the identical protocol (relative tolerance cfg.bench.parity_ppl_rel_tol)
"""
from __future__ import annotations

import json
import os
import re
import shutil
import subprocess
from pathlib import Path

import numpy as np

from .ckpt import atomic_write_json, sha256_file


def sh(cmd: list | str, cwd=None, timeout=None, check=True) -> subprocess.CompletedProcess:
    p = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True, timeout=timeout, shell=isinstance(cmd, str))
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}\nSTDOUT:\n{p.stdout[-2000:]}\nSTDERR:\n{p.stderr[-3000:]}")
    return p


def build_llama_cpp(dest: str | Path, repo: str, ref: str = "master", jobs: int | None = None) -> dict:
    """Clone (or reuse) and build llama.cpp for the host CPU. Returns paths + the exact commit."""
    dest = Path(dest)
    if not (dest / ".git").exists():
        sh(["git", "clone", "--depth", "50", repo, str(dest)], timeout=1800)
    if ref != "master":
        sh(["git", "fetch", "--depth", "1", "origin", ref], cwd=dest, timeout=600)
        sh(["git", "checkout", "FETCH_HEAD"], cwd=dest)
    commit = sh(["git", "rev-parse", "HEAD"], cwd=dest).stdout.strip()
    bins = dest / "build" / "bin"
    need = ["llama-quantize", "llama-perplexity", "llama-bench", "llama-imatrix"]
    if not all((bins / b).exists() for b in need):
        sh(["cmake", "-B", "build", "-DCMAKE_BUILD_TYPE=Release", "-DLLAMA_CURL=OFF", "-DGGML_NATIVE=ON"],
           cwd=dest, timeout=1800)
        sh(["cmake", "--build", "build", "--config", "Release", "-j", str(jobs or os.cpu_count() or 2),
            "--target", *need], cwd=dest, timeout=3600)
    # NOTE: do NOT `pip install -r requirements-convert_hf_to_gguf.txt`: it pins a CPU-only torch and would
    # replace the CUDA build in Colab. convert_hf_to_gguf.py imports the bundled gguf-py from the
    # checkout and otherwise needs only numpy/sentencepiece/transformers/torch, which are installed.
    sh(["pip", "install", "-q", "sentencepiece", "protobuf"], timeout=900, check=False)
    return {"dir": str(dest), "commit": commit, "bin": str(bins)}


def convert_to_gguf(llama: dict, hf_dir: str | Path, out_f16: str | Path) -> Path:
    out_f16 = Path(out_f16)
    if not out_f16.exists():
        sh(["python", str(Path(llama["dir"]) / "convert_hf_to_gguf.py"), str(hf_dir), "--outtype", "f16",
            "--outfile", str(out_f16)], timeout=3600)
    return out_f16


def quantize_gguf(llama: dict, f16: Path, qtype: str, out: Path, embed_type: str = "q8_0",
                  imatrix: Path | None = None) -> Path:
    if out.exists():
        return out
    cmd = [str(Path(llama["bin"]) / "llama-quantize"), "--pure"]
    if qtype.upper() != "Q8_0":
        cmd += ["--token-embedding-type", embed_type, "--output-tensor-type", embed_type]
    if imatrix is not None:
        cmd += ["--imatrix", str(imatrix)]
    cmd += [str(f16), str(out), qtype.upper()]
    sh(cmd, timeout=3600)
    return out


def make_imatrix(llama: dict, f16: Path, calib_txt: Path, out: Path, chunks: int = 100, threads: int = 4) -> Path:
    if not out.exists():
        sh([str(Path(llama["bin"]) / "llama-imatrix"), "-m", str(f16), "-f", str(calib_txt), "-o", str(out),
            "--chunks", str(chunks), "-t", str(threads)], timeout=7200)
    return out


# ----------------------------------------------------------------------------- validation
def audit_gguf(path: Path, expect_linear: str) -> dict:
    from gguf import GGUFReader

    r = GGUFReader(str(path))
    types, bad = {}, []
    for t in r.tensors:
        tn = t.tensor_type.name
        types[tn] = types.get(tn, 0) + 1
        if re.match(r"blk\.\d+\.(attn_(q|k|v|output)|ffn_(gate|up|down))\.weight", t.name) and tn != expect_linear.upper():
            bad.append((t.name, tn))
        if t.name.endswith("norm.weight") and tn != "F32":
            bad.append((t.name, tn))
    meta = {}
    for k in ("general.architecture", "llama.block_count", "llama.attention.head_count",
              "llama.attention.head_count_kv", "llama.feed_forward_length", "llama.attention.key_length",
              "tokenizer.ggml.add_bos_token"):
        f = r.fields.get(k)
        if f is not None:
            v = f.parts[f.data[0]]
            meta[k] = v.tobytes().decode() if v.dtype == np.uint8 else v.tolist()
    return {"tensor_types": types, "violations": bad, "metadata": meta, "file_bytes": path.stat().st_size,
            "sha256": sha256_file(path)}


def _permute(w, n_head):
    return w.reshape(n_head, 2, w.shape[0] // n_head // 2, *w.shape[1:]).swapaxes(1, 2).reshape(w.shape)


def grid_identity(path: Path, model, max_layers: int = 4) -> dict:
    """Compare dequantized GGUF decoder tensors with q(PyTorch weight). For QAT-converted or snapped
    models the PyTorch weights already lie on the grid, so the expected max error is 0 (up to the
    fp16 scale rounding that both sides share)."""
    from gguf import GGUFReader, quants

    from .quant import GRIDS

    r = GGUFReader(str(path))
    cfg = model.config
    names = {"attn_q": "self_attn.q_proj", "attn_k": "self_attn.k_proj", "attn_v": "self_attn.v_proj",
             "attn_output": "self_attn.o_proj", "ffn_gate": "mlp.gate_proj", "ffn_up": "mlp.up_proj",
             "ffn_down": "mlp.down_proj"}
    sd = dict(model.named_parameters())
    out = {}
    for t in r.tensors:
        m = re.match(r"blk\.(\d+)\.(\w+)\.weight", t.name)
        if not m or int(m.group(1)) >= max_layers or m.group(2) not in names:
            continue
        g = quants.dequantize(t.data, t.tensor_type).reshape(-1, int(t.shape[0]))
        w = sd[f"model.layers.{m.group(1)}.{names[m.group(2)]}.weight"].detach().float().cpu().half().float()
        grid = t.tensor_type.name
        ref = GRIDS[grid](w).numpy() if grid in GRIDS else w.numpy()
        if m.group(2) == "attn_q":
            ref = _permute(ref, cfg.num_attention_heads)
        elif m.group(2) == "attn_k":
            ref = _permute(ref, cfg.num_key_value_heads)
        out[t.name] = float(np.abs(g - ref).max())
    return {"max_abs_diff": max(out.values()) if out else None, "per_tensor": out}


def llama_perplexity(llama: dict, gguf_path: Path, text_file: Path, ctx: int, chunks: int | None, threads: int) -> dict:
    cmd = [str(Path(llama["bin"]) / "llama-perplexity"), "-m", str(gguf_path), "-f", str(text_file), "-c", str(ctx),
           "-t", str(threads), "-b", str(ctx)]
    if chunks:
        cmd += ["--chunks", str(chunks)]
    last = ""
    for attempt in range(3):   # llama.cpp logs asynchronously; the final lines are occasionally lost at exit
        p = sh(cmd, timeout=14400)
        txt = p.stdout + "\n" + p.stderr
        m = re.search(r"Final estimate: PPL = ([\d.]+) \+/- ([\d.]+)", txt)
        if m:
            return {"gguf_ppl": float(m.group(1)), "gguf_ppl_se": float(m.group(2)), "source": "final estimate",
                    "attempts": attempt + 1}
        # the running estimate printed after the LAST chunk equals the final (cumulative) PPL
        runs = re.findall(r"\[(\d+)\]([\d.]+)", p.stdout)
        if runs and chunks is not None and int(runs[-1][0]) == chunks:
            return {"gguf_ppl": float(runs[-1][1]), "gguf_ppl_se": None,
                    "source": f"running estimate after chunk {runs[-1][0]}", "attempts": attempt + 1}
        last = txt
    raise RuntimeError("could not parse llama-perplexity output after 3 attempts:\n" + last[-1500:])


def parity_report(pt_ppl: float, gguf_ppl: float, tol: float) -> dict:
    rel = abs(gguf_ppl - pt_ppl) / pt_ppl
    return {"pt_ppl": pt_ppl, "gguf_ppl": gguf_ppl, "rel_diff": rel, "tolerance": tol, "pass": rel <= tol}


# ----------------------------------------------------------------------------- android
ANDROID_BUILD = r"""#!/usr/bin/env bash
# Cross-compile llama.cpp for Android arm64 (run on Linux/Colab). Requires ANDROID_NDK.
set -euo pipefail
: "${ANDROID_NDK:?set ANDROID_NDK to the NDK path}"
cd "$1"   # llama.cpp checkout (same commit as the desktop build: {commit})
cmake -B build-android -DCMAKE_TOOLCHAIN_FILE=$ANDROID_NDK/build/cmake/android.toolchain.cmake \
  -DANDROID_ABI=arm64-v8a -DANDROID_PLATFORM=android-28 -DCMAKE_BUILD_TYPE=Release \
  -DGGML_OPENMP=OFF -DLLAMA_CURL=OFF -DBUILD_SHARED_LIBS=OFF
cmake --build build-android --config Release -j --target llama-bench llama-perplexity
echo "binaries in $1/build-android/bin"
"""

ANDROID_BENCH = r"""#!/usr/bin/env bash
# Run ON A HOST with adb connected to the phone. Produces android_results/<model>/*.txt|json
# Same phone, same binaries, same settings for every model. Keep the phone plugged-out, screen on,
# airplane mode, ~room temperature; let it cool 2 min between models (thermal throttling).
set -euo pipefail
BIN_DIR=$1          # build-android/bin
MODELS_DIR=$2       # directory with *.gguf to compare
OUT=${{3:-android_results}}
DEV=/data/local/tmp/jlc
THREADS={threads}; REPS={reps}; PROMPTS={prompts}; GEN={gen}
adb shell mkdir -p $DEV
adb push $BIN_DIR/llama-bench $DEV/ >/dev/null
adb shell chmod +x $DEV/llama-bench
mkdir -p $OUT
adb shell getprop ro.product.model > $OUT/device_model.txt
adb shell getprop ro.soc.model > $OUT/soc.txt || true
adb shell getprop ro.build.version.release > $OUT/android_version.txt
adb shell cat /proc/meminfo | head -3 > $OUT/meminfo.txt
for M in $MODELS_DIR/*.gguf; do
  NAME=$(basename $M .gguf); mkdir -p $OUT/$NAME
  adb push $M $DEV/model.gguf >/dev/null
  # warm-up (discarded)
  adb shell "cd $DEV && ./llama-bench -m model.gguf -p 64 -n 16 -r 1 -t $THREADS" > /dev/null
  adb shell "cd $DEV && ./llama-bench -m model.gguf -p $PROMPTS -n $GEN -r $REPS -t $THREADS -o json" > $OUT/$NAME/bench.json
  # peak resident memory (VmHWM) of a model load + 512-token prefill + 64-token decode, polled at 10 Hz
  adb shell "cd $DEV && (./llama-bench -m model.gguf -p 512 -n 64 -r 1 -t $THREADS > /dev/null 2>&1 & P=\$!; H=0; while kill -0 \$P 2>/dev/null; do V=\$(grep VmHWM /proc/\$P/status 2>/dev/null | awk '{{print \$2}}'); [ -n \"\$V\" ] && H=\$V; sleep 0.1; done; echo \$H)" > $OUT/$NAME/peak_kb.txt
  adb shell rm $DEV/model.gguf
  sleep 120
done
echo "Done. Upload $OUT to <drive_root>/android/ and fill android/device.json"
"""

DEVICE_TEMPLATE = {
    "phone": "FILL ME (e.g. Pixel 8)", "chipset": "FILL ME (e.g. Tensor G3)", "ram_gb": None,
    "android_version": None, "runtime": "llama.cpp (commit in exports/llama_cpp.json)",
    "threads": None, "prompt_lengths": None, "gen_length": None, "warmup_policy": "1 discarded llama-bench run",
    "trials": None, "energy_instrumentation": "none (energy NOT measured)",
    "notes": "Battery %, temperature, charger state at start",
}


def write_android_kit(out_dir: Path, llama: dict, cfg) -> list:
    out_dir.mkdir(parents=True, exist_ok=True)
    b = cfg.bench
    (out_dir / "build_android.sh").write_text(ANDROID_BUILD.replace("{commit}", llama.get("commit", "?")))
    (out_dir / "bench_android.sh").write_text(ANDROID_BENCH.format(
        threads=b.threads, reps=b.repetitions, prompts=",".join(map(str, b.prompt_lengths)), gen=b.gen_length))
    dev = out_dir / "device.json"
    if not dev.exists():
        atomic_write_json(dev, {**DEVICE_TEMPLATE, "threads": b.threads, "prompt_lengths": list(b.prompt_lengths),
                                "gen_length": b.gen_length, "trials": b.repetitions})
    for f in ("build_android.sh", "bench_android.sh"):
        os.chmod(out_dir / f, 0o755)
    return [str(out_dir / f) for f in ("build_android.sh", "bench_android.sh", "device.json")]

**Module `bench.py`**.

In [ ]:
%%writefile {PKG}/jlc/bench.py
"""Runtime and resource benchmarking.

Every row in runtime_benchmarks.csv carries `platform` and `measurement` fields:
  platform    : 'android:<phone>' | 'desktop_cpu:<cpu>' | 'desktop_gpu:<gpu>'
  measurement : 'measured' | 'derived' (e.g. TTFT computed from measured prefill+decode rates)
                | 'pending' (no device data yet)
Desktop numbers are NEVER presented as mobile numbers; the Pareto plots use only one platform at a time.
"""
from __future__ import annotations

import json
import platform as _pf
import re
import statistics
import subprocess
import time
from pathlib import Path

from .export import sh


def _parse_bench_json(txt: str) -> list[dict]:
    txt = txt[txt.find("["):txt.rfind("]") + 1]
    return json.loads(txt)


def summarize_llama_bench(rows: list[dict]) -> dict:
    """llama-bench json -> prefill tok/s per prompt length and decode tok/s."""
    out = {"pp": {}, "tg": None}
    for r in rows:
        n_p, n_g = int(r.get("n_prompt", 0)), int(r.get("n_gen", 0))
        ts, sd = float(r["avg_ts"]), float(r.get("stddev_ts", 0.0))
        if n_p > 0 and n_g == 0:
            out["pp"][n_p] = (ts, sd)
        elif n_g > 0 and n_p == 0:
            out["tg"] = (ts, sd)
    return out


def bench_rows_from_summary(s: dict, base: dict, gen_len: int) -> list[dict]:
    rows = []
    if s["tg"] is None:
        return rows
    tg_ts, tg_sd = s["tg"]
    for p, (pp_ts, pp_sd) in sorted(s["pp"].items()):
        prefill_ms = 1000.0 * p / pp_ts
        rows.append({**base, "prompt_len": p, "gen_len": gen_len, "prefill_tok_s": pp_ts, "prefill_tok_s_sd": pp_sd,
                     "prefill_ms": prefill_ms, "decode_tok_s": tg_ts, "decode_tok_s_sd": tg_sd,
                     "ttft_ms": prefill_ms + 1000.0 / tg_ts, "ttft_measurement": "derived(prefill+1 decode step)"})
    return rows


def desktop_cpu_bench(llama: dict, gguf: Path, cfg, base: dict) -> list[dict]:
    b = cfg.bench
    exe = str(Path(llama["bin"]) / "llama-bench")
    sh([exe, "-m", str(gguf), "-p", "64", "-n", "16", "-r", "1", "-t", str(b.threads)], timeout=3600)  # warm-up
    p = sh([exe, "-m", str(gguf), "-p", ",".join(map(str, b.prompt_lengths)), "-n", str(b.gen_length),
            "-r", str(b.repetitions), "-t", str(b.threads), "-o", "json"], timeout=7200)
    s = summarize_llama_bench(_parse_bench_json(p.stdout))
    cpu = _pf.processor() or _pf.machine()
    mem = peak_rss_llama(llama, gguf, b.threads)
    base = {**base, "platform": f"desktop_cpu:{cpu}", "measurement": "measured", "runtime": "llama.cpp",
            "threads": b.threads, "reps": b.repetitions, **mem}
    return bench_rows_from_summary(s, base, b.gen_length)


def peak_rss_llama(llama: dict, gguf: Path, threads: int) -> dict:
    """Exact peak RSS (kernel rusage, via a minimal launcher - see env.run_measured) and wall time of a
    short llama-bench run: model load + 512-token prefill + 64-token decode."""
    from .env import run_measured

    exe = str(Path(llama["bin"]) / "llama-bench")
    r = run_measured([exe, "-m", str(gguf), "-p", "512", "-n", "64", "-r", "1", "-t", str(threads)], quiet=True)
    ok = r["exit"] == 0
    return {"peak_rss_mb": r["peak_rss_mb"] if ok else None, "load_plus_run_s": r["wall_s"],
            "peak_rss_measurement": (f"measured (rusage ru_maxrss of llama-bench p512 n64; launcher floor "
                                     f"{r['launcher_floor_mb']:.1f} MB)") if ok else f"unavailable (exit {r['exit']})"}


def hf_gpu_timing(model, tok, prompt_len: int, gen_len: int, reps: int, base: dict) -> dict:
    """PyTorch eager timing on the training GPU (labelled desktop_gpu; not a deployment number)."""
    import torch

    if not torch.cuda.is_available():
        return {}
    dev = next(model.parameters()).device
    ids = torch.randint(100, 1000, (1, prompt_len), device=dev)
    model.eval()
    pre, dec = [], []
    with torch.no_grad():
        model.generate(ids, max_new_tokens=4, do_sample=False, pad_token_id=tok.pad_token_id)  # warm-up
        for _ in range(reps):
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            model(ids)
            torch.cuda.synchronize()
            pre.append(time.perf_counter() - t0)
            t0 = time.perf_counter()
            out = model.generate(ids, max_new_tokens=gen_len, min_new_tokens=gen_len, do_sample=False,
                                 pad_token_id=tok.pad_token_id)
            torch.cuda.synchronize()
            dec.append((time.perf_counter() - t0, out.shape[1] - prompt_len))
    pm = statistics.median(pre)
    tot = statistics.median(t for t, _ in dec)
    n = dec[0][1]
    return {**base, "platform": f"desktop_gpu:{torch.cuda.get_device_name()}", "measurement": "measured",
            "runtime": "pytorch-eager", "prompt_len": prompt_len, "gen_len": gen_len, "prefill_ms": pm * 1000,
            "decode_tok_s": (n - 1) / max(1e-9, tot - pm), "ttft_ms": None, "reps": reps,
            "peak_gpu_mem_mb": torch.cuda.max_memory_allocated() / 2**20}


def ingest_android(android_dir: Path, artifact_index: dict) -> tuple[list[dict], dict]:
    """Parse results produced by bench_android.sh. Returns (rows, status). If device.json is not
    filled in or no result folders exist, status='PENDING' and no numbers are produced."""
    android_dir = Path(android_dir)
    devf = android_dir / "device.json"
    res = android_dir / "android_results"
    if not devf.exists() or not res.exists():
        return [], {"status": "PENDING", "reason": "no android results uploaded"}
    dev = json.loads(devf.read_text())
    if "FILL ME" in json.dumps(dev):
        return [], {"status": "PENDING", "reason": "device.json not filled in"}
    rows, missing = [], []
    for name, meta in artifact_index.items():
        d = res / name
        if not (d / "bench.json").exists():
            missing.append(name)
            continue
        s = summarize_llama_bench(_parse_bench_json((d / "bench.json").read_text()))
        peak = None
        if (d / "peak_kb.txt").exists():
            v = (d / "peak_kb.txt").read_text().strip().splitlines()
            peak = int(v[-1]) / 1024 if v and v[-1].isdigit() and int(v[-1]) > 0 else None
        base = {**meta, "platform": f"android:{dev['phone']}", "chipset": dev.get("chipset"),
                "measurement": "measured", "runtime": "llama.cpp", "threads": dev.get("threads"),
                "reps": dev.get("trials"), "peak_rss_mb": peak,
                "peak_rss_measurement": "VmHWM polled at 10 Hz" if peak else "unavailable",
                "energy_j": None, "energy_measurement": dev.get("energy_instrumentation")}
        rows += bench_rows_from_summary(s, base, dev.get("gen_length") or 0)
    status = {"status": "MEASURED" if rows else "PENDING", "device": dev, "missing_models": missing}
    return rows, status

**Module `stats.py`**.

In [ ]:
%%writefile {PKG}/jlc/stats.py
"""Aggregation, statistical tests, Pareto frontiers, figures and the final report."""
from __future__ import annotations

import json
import math
from pathlib import Path

import numpy as np
import pandas as pd

PRIMARY_METRICS = ["gsm8k_em", "ifeval_prompt_strict", "ifeval_inst_strict", "wikitext2_ppl"]
HIGHER_BETTER = {"gsm8k_em": True, "ifeval_prompt_strict": True, "ifeval_inst_strict": True,
                 "ifeval_prompt_loose": True, "wikitext2_ppl": False}

# Pre-registered comparisons (A vs B): the family that Holm correction is applied to.
PRIMARY_COMPARISONS = [
    ("E8", "E7", "H1: joint vs sequential"),
    ("E9", "E7", "H2: alternating vs sequential"),
    ("E8", "E9", "H3: joint vs alternating"),
    ("E8", "A_H4_noQAI", "H4: quantization-aware gating vs FP gating"),
    ("E5", "E3", "H5: KD+QAT vs INT4 PTQ"),
]

FAMILY = {"E0": "uncompressed", "R1": "uncompressed", "E1": "uncompressed", "E2": "single technique",
          "E3": "single technique", "E3_8": "single technique",
          "E4": "two techniques", "E5": "two techniques", "E6": "two techniques",
          "E7": "sequential (all three)", "E8": "joint / alternating",
          "E9": "joint / alternating"}
FAMILY_COLOR = {"uncompressed": "#8a8a85", "single technique": "#2a78d6", "two techniques": "#1baf7a",
                "sequential (all three)": "#eb6834", "joint / alternating": "#6250d6", "ablation": "#6250d6"}


def t_ci(x: np.ndarray, alpha: float = 0.05) -> tuple[float, float]:
    from scipy import stats

    x = np.asarray(x, float)
    if len(x) < 2:
        return (float("nan"), float("nan"))
    h = stats.t.ppf(1 - alpha / 2, len(x) - 1) * x.std(ddof=1) / math.sqrt(len(x))
    return (x.mean() - h, x.mean() + h)


def aggregate(runs: pd.DataFrame, alpha: float = 0.05) -> pd.DataFrame:
    """Mean/SD/CI per experiment across seeds; failed runs are counted, never dropped silently."""
    rows = []
    metrics = [c for c in runs.columns if runs[c].dtype.kind in "fi" and c not in ("seed",)]
    for exp, g in runs.groupby("exp"):
        ok = g[g["status"] == "ok"]
        r = {"exp": exp, "n_seeds_ok": len(ok), "n_failed": int((g["status"] != "ok").sum())}
        for m in metrics:
            v = ok[m].dropna().to_numpy(float)
            if len(v) == 0:
                continue
            r[f"{m}_mean"] = v.mean()
            r[f"{m}_sd"] = v.std(ddof=1) if len(v) > 1 else float("nan")
            lo, hi = t_ci(v, alpha)
            r[f"{m}_ci_lo"], r[f"{m}_ci_hi"] = lo, hi
        rows.append(r)
    return pd.DataFrame(rows)


def _load_items(root: Path, exp: str, seed: int, task: str) -> pd.DataFrame | None:
    f = Path(root) / "results" / "items" / exp / f"seed_{seed}" / f"{task}.jsonl"
    if not f.exists():
        return None
    return pd.read_json(f, lines=True)


def paired_bootstrap(root: Path, a: str, b: str, seeds: list, task: str, col: str, iters: int, rng_seed: int = 0,
                     log_scale: bool = False) -> dict:
    """Paired bootstrap over evaluation items, pooling seeds (item i of seed s in A is paired with
    item i of seed s in B). Returns mean difference A-B, 95% CI and a two-sided bootstrap p-value."""
    diffs = []
    for s in seeds:
        A, B = _load_items(root, a, s, task), _load_items(root, b, s, task)
        if A is None or B is None:
            continue
        key = "chunk" if task == "wikitext2" else "id"
        m = A.merge(B, on=key, suffixes=("_a", "_b"))
        if task == "wikitext2":
            da = m["nll_sum_a"] / m["n_a"]
            db = m["nll_sum_b"] / m["n_b"]
            diffs.append((da - db).to_numpy())     # mean NLL difference (log-PPL), lower is better
        else:
            diffs.append(m[f"{col}_a"].astype(float).to_numpy() - m[f"{col}_b"].astype(float).to_numpy())
    if not diffs:
        return {"status": "missing items"}
    D = np.stack(diffs, 1) if len({len(d) for d in diffs}) == 1 else None
    if D is None:
        return {"status": "item sets differ across seeds"}
    rng = np.random.default_rng(rng_seed)
    n = D.shape[0]
    obs = D.mean()
    boots = np.array([D[rng.integers(0, n, n)].mean() for _ in range(iters)])
    p = min(1.0, 2 * min((boots <= 0).mean(), (boots >= 0).mean()))
    return {"status": "ok", "diff": float(obs), "ci_lo": float(np.quantile(boots, 0.025)),
            "ci_hi": float(np.quantile(boots, 0.975)), "p_boot": float(p), "n_items": int(n), "n_seeds": D.shape[1]}


def mcnemar(root: Path, a: str, b: str, seeds: list, task: str, col: str) -> dict:
    from scipy.stats import binomtest

    b01 = b10 = 0
    for s in seeds:
        A, B = _load_items(root, a, s, task), _load_items(root, b, s, task)
        if A is None or B is None:
            continue
        m = A.merge(B, on="id", suffixes=("_a", "_b"))
        x, y = m[f"{col}_a"].astype(bool), m[f"{col}_b"].astype(bool)
        b10 += int((x & ~y).sum())
        b01 += int((~x & y).sum())
    if b01 + b10 == 0:
        return {"mcnemar_p": 1.0, "discordant": 0}
    return {"mcnemar_p": binomtest(b10, b01 + b10, 0.5).pvalue, "a_only": b10, "b_only": b01,
            "note": "exact McNemar on discordant pairs pooled over seeds (treats seeds as extra items)"}


def seed_paired_t(runs: pd.DataFrame, a: str, b: str, metric: str) -> dict:
    from scipy import stats

    A = runs[(runs.exp == a) & (runs.status == "ok")].set_index("seed")[metric]
    B = runs[(runs.exp == b) & (runs.status == "ok")].set_index("seed")[metric]
    j = A.index.intersection(B.index)
    if len(j) < 2:
        return {"seed_paired_t_p": float("nan"), "n_seed_pairs": len(j)}
    t = stats.ttest_rel(A[j], B[j])
    return {"seed_paired_t_p": float(t.pvalue), "n_seed_pairs": len(j), "seed_mean_diff": float((A[j] - B[j]).mean())}


def holm(pvals: list[float]) -> list[float]:
    idx = [i for i, p in enumerate(pvals) if p == p]
    order = sorted(idx, key=lambda i: pvals[i])
    adj = [float("nan")] * len(pvals)
    running = 0.0
    m = len(order)
    for rank, i in enumerate(order):
        running = max(running, min(1.0, (m - rank) * pvals[i]))
        adj[i] = running
    return adj


def run_tests(root: Path, runs: pd.DataFrame, seeds: list, iters: int) -> pd.DataFrame:
    tasks = [("gsm8k", "correct", "gsm8k_em"), ("ifeval", "prompt_level_strict_acc", "ifeval_prompt_strict"),
             ("wikitext2", None, "wikitext2_ppl")]
    rows = []
    for a, b, h in PRIMARY_COMPARISONS:
        for task, col, metric in tasks:
            r = {"hypothesis": h, "A": a, "B": b, "metric": metric}
            r.update(paired_bootstrap(root, a, b, seeds, task, col, iters))
            if task != "wikitext2" and r.get("status") == "ok":
                r.update(mcnemar(root, a, b, seeds, task, col))
            if metric in runs.columns:
                r.update(seed_paired_t(runs, a, b, metric))
            rows.append(r)
    df = pd.DataFrame(rows)
    if "p_boot" in df:
        df["p_holm"] = holm(df["p_boot"].fillna(float("nan")).tolist())
    return df


def pareto_front(df: pd.DataFrame, x: str, y: str, y_higher: bool = True) -> pd.DataFrame:
    d = df.dropna(subset=[x, y]).sort_values(x)
    front, best = [], -np.inf
    for _, r in d.iterrows():
        v = r[y] if y_higher else -r[y]
        if v > best:
            front.append(r)
            best = v
    return pd.DataFrame(front)


def _scatter(ax, df, x, y, yerr=None, label_col="exp"):
    for fam, g in df.groupby(df["exp"].map(lambda e: FAMILY.get(e, "ablation"))):
        ax.errorbar(g[x], g[y], yerr=None if yerr is None else g[yerr], fmt="o" if fam != "ablation" else "D",
                    ms=8, color=FAMILY_COLOR[fam], mfc=FAMILY_COLOR[fam] if fam != "ablation" else "white",
                    mew=2, elinewidth=1.5, capsize=3, label=fam, zorder=3)
    for _, r in df.iterrows():
        ax.annotate(r[label_col], (r[x], r[y]), xytext=(5, 5), textcoords="offset points", fontsize=8,
                    color="#3a3a38")


def _style(ax, xl, yl, title):
    ax.set_xlabel(xl)
    ax.set_ylabel(yl)
    ax.set_title(title, loc="left", fontsize=11)
    ax.grid(True, color="#e6e5e0", lw=0.8, zorder=0)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.legend(frameon=False, fontsize=8, loc="best")


def make_figures(summary: pd.DataFrame, bench: pd.DataFrame, fig_dir: Path, quality: str = "gsm8k_em",
                 latency_platform: str | None = None) -> dict:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig_dir = Path(fig_dir)
    out = {}
    q, qe = f"{quality}_mean", f"{quality}_sd"
    s = summary.dropna(subset=[q]).copy()
    if "artifact_mb_mean" in s and s["artifact_mb_mean"].notna().any():
        fig, ax = plt.subplots(figsize=(7, 4.5))
        _scatter(ax, s, "artifact_mb_mean", q, qe)
        _style(ax, "Deployed GGUF size (MB, measured file size)", quality, f"{quality} vs. model size")
        fig.tight_layout()
        fig.savefig(fig_dir / "quality_vs_size.png", dpi=160)
        plt.close(fig)
        out["quality_vs_size"] = str(fig_dir / "quality_vs_size.png")
    lat = None
    if bench is not None and len(bench):
        plats = bench["platform"].unique().tolist()
        plat = latency_platform or next((p for p in plats if p.startswith("android:")), plats[0])
        b = bench[(bench.platform == plat) & (bench.prompt_len == bench.prompt_len.max())]
        lat = b.groupby("exp", as_index=False)["decode_tok_s"].median()
        lat["ms_per_token"] = 1000.0 / lat["decode_tok_s"]
        m = s.merge(lat, on="exp")
        if len(m):
            fig, ax = plt.subplots(figsize=(7, 4.5))
            _scatter(ax, m, "ms_per_token", q, qe)
            _style(ax, f"Decode latency (ms/token) — {plat}", quality, f"{quality} vs. latency")
            fig.tight_layout()
            fig.savefig(fig_dir / "quality_vs_latency.png", dpi=160)
            plt.close(fig)
            out["quality_vs_latency"] = str(fig_dir / "quality_vs_latency.png")
            out["latency_platform"] = plat
    if "artifact_mb_mean" in s:
        fig, axes = plt.subplots(1, 2 if lat is not None else 1, figsize=(12 if lat is not None else 7, 4.5),
                                 squeeze=False)
        f1 = pareto_front(s, "artifact_mb_mean", q, HIGHER_BETTER.get(quality, True))
        _scatter(axes[0, 0], s, "artifact_mb_mean", q)
        axes[0, 0].step(f1["artifact_mb_mean"], f1[q], where="post", color="#3a3a38", lw=2, label="Pareto frontier")
        _style(axes[0, 0], "GGUF size (MB)", quality, "Pareto: quality vs size")
        if lat is not None:
            m = s.merge(lat, on="exp")
            f2 = pareto_front(m, "ms_per_token", q, HIGHER_BETTER.get(quality, True))
            _scatter(axes[0, 1], m, "ms_per_token", q)
            axes[0, 1].step(f2["ms_per_token"], f2[q], where="post", color="#3a3a38", lw=2, label="Pareto frontier")
            _style(axes[0, 1], f"ms/token ({out.get('latency_platform')})", quality, "Pareto: quality vs latency")
        fig.tight_layout()
        fig.savefig(fig_dir / "pareto_frontier.png", dpi=160)
        plt.close(fig)
        out["pareto_frontier"] = str(fig_dir / "pareto_frontier.png")
    return out

**Module `report.py`**.

In [ ]:
%%writefile {PKG}/jlc/report.py
"""Final report and artifact manifest, generated ONLY from files on disk. If an input is missing the
report says so; it never fills a gap with text that sounds like a result."""
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

from .ckpt import now, sha256_file

IMPLEMENTED = [
    "Centralised config + hash-keyed, resumable Drive checkpoints (atomic writes, SHA-256, safe loading)",
    "Data pipeline: normalisation, exact dedup, 13-gram decontamination against all evaluation sets, manifests",
    "Teacher generation with fixed greedy decoding, sharded resumable cache, failure accounting",
    "Response-level KD (teacher targets) + token-level KD only when output spaces are verified identical",
    "GQA-aware structured pruning: importance (activation/Taylor/magnitude), gates, masks, physical slicing",
    "QAT with a fake-quantizer bit-exact to GGUF Q4_0 (STE) and a torchao IntxFakeQuantizer alternative",
    "Joint objective L_joint = λ_KD L_KD + λ_CE L_CE + λ_SP R_SP + λ_Q L_Q; alternating schedule",
    "GGUF export, tensor-type audit, grid-identity check, llama.cpp perplexity parity",
    "Desktop llama.cpp benchmarks; Android cross-compile + benchmark scripts and result ingestion",
    "Paired bootstrap, McNemar, seed-paired t-tests, Holm correction, Pareto frontiers",
]
NOT_IMPLEMENTED = [
    "Embedding/hidden-size and depth pruning (only heads and MLP channels are pruned)",
    "Non-uniform per-layer widths (would need custom config + llama.cpp per-layer arrays)",
    "Head counts that do not divide hidden_size (transformers>=5 LlamaConfig rejects them on reload)",
    "Activation quantization / KV-cache quantization (weight-only INT4/INT8)",
    "Pruning of the teacher architecture (Phi-3/4 fused qkv/gate_up projections) - diagnosed and refused",
    "Importance-matrix (imatrix) PTQ baseline: helper export.make_imatrix() is provided but the condition is "
    "not in the matrix; E3/E7 use round-to-nearest, the standard `llama-quantize Q4_0` behaviour",
    "Energy measurement (requires external power instrumentation; reported as not measured)",
    "GGUF-native evaluation of GSM8K/IFEval (GGUF validated by perplexity parity; generation quality is "
    "measured in PyTorch on weights snapped to the identical grid)",
]


def _fmt(x, nd=3):
    try:
        return f"{float(x):.{nd}f}"
    except (TypeError, ValueError):
        return "n/a"


def write_report(root: Path, cfg, summary: pd.DataFrame, tests: pd.DataFrame, bench: pd.DataFrame,
                 android_status: dict, figures: dict, validation: dict, budget: list, extra_notes: list | None = None) -> Path:
    root = Path(root)
    L = [f"# Research results: Beyond Sequential Compression", "",
         f"_Generated {now()} from files under `{root}`. Config hash `{cfg.hash()}`, mode `{cfg.mode}`._", ""]
    if cfg.mode == "smoke" or validation.get("offline_fixture"):
        L += ["> **SMOKE / FIXTURE RUN.** Tiny models and data were used to test the pipeline. "
              "None of the numbers below are research results.", ""]
    L += ["## 1. Protocol", "",
          f"- Teacher: `{cfg.model.teacher_id}`; student: `{cfg.model.student_id}`",
          f"- Stage budget S = {cfg.train.stage_steps} steps, effective batch = "
          f"{cfg.train.micro_batch * cfg.train.grad_accum} sequences x <= {cfg.train.seq_len} tokens",
          f"- Pruning target: head keep {cfg.prune.head_keep_ratio}, MLP keep {cfg.prune.mlp_keep_ratio}; "
          f"QAT backend `{cfg.quant.qat_backend}`, deployment grid GGUF Q4_0",
          f"- Seeds: {list(cfg.train.seeds)}", "", "### Compute budget per condition", "",
          pd.DataFrame(budget)[["exp", "start", "own_steps", "inherited_steps", "total_steps", "targets", "deploy",
                                "description"]].to_markdown(index=False), ""]
    L += ["## 2. Quality and size (held-out, mean ± SD over seeds)", ""]
    if summary is None or summary.empty:
        L += ["_No final results found._", ""]
    else:
        cols = ["exp", "n_seeds_ok", "n_failed"]
        show = summary[cols].copy()
        for m in ("gsm8k_em", "ifeval_prompt_strict", "ifeval_inst_strict", "wikitext2_ppl", "artifact_mb", "params_total"):
            if f"{m}_mean" in summary:
                show[m] = [f"{_fmt(a)} ± {_fmt(b)}" if b == b else _fmt(a)
                           for a, b in zip(summary[f"{m}_mean"], summary.get(f"{m}_sd", [float('nan')] * len(summary)))]
        L += [show.to_markdown(index=False), ""]
    L += ["## 3. Pre-registered comparisons", ""]
    if tests is None or tests.empty:
        L += ["_No statistical tests computed._", ""]
    else:
        keep = [c for c in ("hypothesis", "metric", "diff", "ci_lo", "ci_hi", "p_boot", "p_holm", "mcnemar_p",
                            "seed_paired_t_p", "n_items", "n_seeds", "status") if c in tests]
        L += [tests[keep].to_markdown(index=False, floatfmt=".4g"), "",
              "Interpretation rule (fixed in advance): a hypothesis is supported only if the Holm-adjusted "
              "p < 0.05 AND the 95% CI of the difference excludes 0 in the predicted direction. "
              "For WikiText-2 the difference is in mean NLL (negative = A better).", ""]
        if "p_holm" in tests and "ci_lo" in tests:
            ppl = tests["metric"].str.contains("ppl")
            right_dir = (~ppl & (tests["ci_lo"] > 0)) | (ppl & (tests["ci_hi"] < 0))
            sup = tests[(tests["p_holm"] < 0.05) & right_dir]
        else:
            sup = tests.iloc[0:0]
        L += [f"Comparisons meeting the rule (A better than B): {len(sup)} of {len(tests)}" +
              (": " + "; ".join(f"{r.hypothesis} [{r.metric}]" for r in sup.itertuples()) if len(sup) else "."), ""]
    L += ["## 4. Runtime", ""]
    if bench is None or bench.empty:
        L += ["_No runtime measurements found._", ""]
    else:
        for plat, g in bench.groupby("platform"):
            L += [f"### {plat}", "", g[[c for c in ("exp", "seed", "prompt_len", "prefill_ms", "ttft_ms", "decode_tok_s",
                                                    "peak_rss_mb", "measurement") if c in g]].to_markdown(index=False,
                                                                                                         floatfmt=".1f"), ""]
    L += [f"**Android status: {android_status.get('status')}** — {android_status.get('reason', '')}", ""]
    if android_status.get("status") != "MEASURED":
        L += ["Mobile latency, memory and energy are NOT reported. Desktop numbers above are not substitutes.", ""]
    L += ["## 5. Figures", ""] + [f"- `{Path(v).name}`" for k, v in figures.items() if k != "latency_platform"] + [""]
    L += ["## 6. Validation summary", "", "```json", json.dumps(validation, indent=2, default=str), "```", ""]
    L += ["## 7. Limitations", "", "### Implemented"] + [f"- {x}" for x in IMPLEMENTED] + \
         ["", "### Not implemented / future work"] + [f"- {x}" for x in NOT_IMPLEMENTED] + [""]
    if extra_notes:
        L += ["### Run-specific notes"] + [f"- {x}" for x in extra_notes] + [""]
    out = root / "results" / "research_results_report.md"
    out.write_text("\n".join(L))
    return out


def artifact_manifest(root: Path, extra_paths: list | None = None) -> dict:
    root = Path(root)
    entries = []
    for sub in ("results", "figures", "exports", "data", "teacher_targets", "logs", "code", "android"):
        for p in sorted((root / sub).rglob("*")):
            if p.is_file() and not p.name.endswith((".tmp", ".partial")):
                entries.append({"path": str(p.relative_to(root)), "bytes": p.stat().st_size,
                                "sha256": sha256_file(p) if p.stat().st_size < 8 * 2**30 else "skipped(>8GB)"})
    for p in (extra_paths or []):
        p = Path(p)
        if p.exists():
            entries.append({"path": str(p), "bytes": p.stat().st_size, "sha256": sha256_file(p)})
    ck = [str(p.parent.relative_to(root)) for p in (root / "runs_by_config").rglob("DONE.json")]
    man = {"generated_utc": now(), "files": entries, "completed_model_dirs": ck}
    (root / "results" / "artifact_manifest.json").write_text(json.dumps(man, indent=2))
    return man

In [ ]:
from pathlib import Path
OFFLINE = {}

def reload_jlc():
    """Re-import the package after its files were (re)written, so edits take effect without a restart."""
    for name in sorted([m for m in sys.modules if m == "jlc" or m.startswith("jlc.")], reverse=True):
        del sys.modules[name]
    mod = importlib.import_module("jlc")
    if OFFLINE and "cfg" in globals():          # test-only fixture hook survives reloads
        importlib.import_module("jlc.testing").install(cfg)
    return mod

reload_jlc()
from jlc import config as C, env as ENV, ckpt as CK
cfg = C.make_config(MODE, **CONFIG_OVERRIDES)
cfg.paths.drive_root = DRIVE_ROOT
if os.environ.get("JLC_OFFLINE_FIXTURES"):      # test-only: see jlc/testing.py (never set on Colab)
    from jlc import testing
    OFFLINE = testing.install(cfg)
if os.environ.get("JLC_LOCAL_ROOT"):
    cfg.paths.local_root = os.environ["JLC_LOCAL_ROOT"]
ROOT = CK.mount_drive(cfg.paths.drive_root)
os.makedirs(cfg.paths.local_root, exist_ok=True)
ENV.set_seed(cfg.train.pilot_seed if PHASE == "pilot" else cfg.train.seeds[0])
HW = ENV.hardware_report()
CK.atomic_write_json(ROOT / "results" / "hardware.json", HW)
cfg.save_yaml(ROOT / "results" / "experiment.yaml")

# validation
assert C.Config.load_yaml(ROOT / "results" / "experiment.yaml").hash() == cfg.hash(), "YAML round-trip changed the config"
(ROOT / "logs" / ".write_test").write_text("ok")
print("config hash:", cfg.hash())
print(json.dumps({k: HW[k] for k in ("cuda_available", "gpus", "ram_total_gb", "in_colab")}, indent=1))
if not HW["cuda_available"]:
    print("WARNING: no GPU detected. Smoke mode will run on CPU (slow); full mode is not feasible.")

---
## 3. Dependency installation and version verification

**Purpose.** Install the packages the pipeline needs **without touching the preinstalled CUDA PyTorch**, then verify versions and the specific APIs used.

**Expected outputs.** `results/environment.txt` (versions, hardware, pip freeze) and an API-verification table.

**Validation.** Every required import succeeds with a version ≥ the minimum; each API this project calls is checked to exist with the expected signature.

**If something fails.** If the check says transformers/torch are too old, run *Runtime → Restart session* after the install cell and re-run from Section 2. vLLM is optional and **not** installed by default because it pins its own torch build; the HF backend is used instead.

In [ ]:
INSTALL_VLLM = False   # optional faster teacher generation; may replace torch -> restart required
import subprocess
pkgs = ["datasets>=2.20", "accelerate>=0.33", "safetensors>=0.4", "pyyaml", "psutil", "tabulate", "scipy",
        "pandas", "matplotlib", "torchao", "bitsandbytes", "lm_eval[ifeval]>=0.4.5", "gguf>=0.10",
        "sentencepiece", "protobuf", "langdetect", "immutabledict", "nltk"]
if not os.environ.get("JLC_SKIP_PIP"):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-1500:])
    if INSTALL_VLLM:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm"], check=False)

In [ ]:
import inspect
VERSIONS = ENV.check_versions(strict=True)
api = {}
def check(name, fn):
    try:
        api[name] = "ok: " + str(fn())
    except Exception as e:
        api[name] = f"MISSING ({e.__class__.__name__}: {e})"

check("transformers LlamaAttention uses module.num_key_value_groups",
      lambda: "num_key_value_groups" in inspect.getsource(__import__("transformers.models.llama.modeling_llama", fromlist=["x"]).eager_attention_forward))
check("transformers LlamaConfig accepts head_dim", lambda: __import__("transformers").LlamaConfig(head_dim=64).head_dim)
check("torch.load weights_only", lambda: "weights_only" in inspect.signature(__import__("torch").load).parameters)
check("torchao IntxFakeQuantizer", lambda: __import__("torchao.quantization.qat.fake_quantizer", fromlist=["x"]).IntxFakeQuantizer.__name__)
check("torchao IntxFakeQuantizeConfig(group_size,is_symmetric)",
      lambda: list(inspect.signature(__import__("torchao.quantization.qat", fromlist=["x"]).IntxFakeQuantizeConfig).parameters)[:2])
check("gguf-py Q4_0 reference quantizer", lambda: __import__("gguf").quants.Q4_0.__name__)
check("lm_eval IFEval scorer", lambda: __import__("lm_eval.tasks.ifeval.utils", fromlist=["x"]).process_results.__name__)
for k, v in api.items():
    print(f"{'OK ' if v.startswith('ok') else 'XX '} {k}: {v}")
ENV.write_environment_txt(ROOT / "results" / "environment.txt", VERSIONS, HW, {"api_checks": api, "offline": OFFLINE})
CRITICAL = ["transformers LlamaAttention uses module.num_key_value_groups", "gguf-py Q4_0 reference quantizer",
            "torch.load weights_only"]
assert all(api[c].startswith("ok") for c in CRITICAL), "A critical API is missing - see table above"
if not api["torchao IntxFakeQuantizer"].startswith("ok"):
    print("torchao fake-quantizer unavailable -> ablation A_E5_torchao will be skipped (logged); main QAT uses q4_0_exact.")

---
## 4. Model, tokenizer, architecture and license inspection

**Purpose.** Resolve and pin model revisions, inspect architectures and licenses without loading weights, estimate memory, decide whether token-level KD is valid, and verify the student can be pruned safely.

**Expected outputs.** `results/model_manifest.json` (ids, commit SHAs, licenses, architecture, memory estimates, KD decision, prunability).

**Validation.** Student passes `inspect_structure` (Llama with separate q/k/v/o and gate/up/down); the teacher architecture is *diagnosed* (Phi-3 family uses fused projections → refused for pruning, by design). Vocabulary check decides KD mode.

**If something fails.** Gated model → accept the license on huggingface.co and add `HF_TOKEN` as a Colab secret. If the student check fails the notebook stops here with the diagnostic; do not bypass it.

In [ ]:
reload_jlc()
from jlc import models as M, prune as P
import torch
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

REV = {"student": cfg.model.student_revision or M.resolve_model_revision(cfg.model.student_id),
       "teacher": cfg.model.teacher_revision or M.resolve_model_revision(cfg.model.teacher_id)}
cfg.model.student_revision, cfg.model.teacher_revision = REV["student"], REV["teacher"]
info = {"student": M.inspect_model(cfg.model.student_id, REV["student"]),
        "teacher": M.inspect_model(cfg.model.teacher_id, REV["teacher"])}

# prunability, checked on a weight-free (meta-device) instance
def meta_model(mid, rev):
    c = AutoConfig.from_pretrained(mid, revision=rev, token=ENV.hf_token())
    with torch.device("meta"):
        return AutoModelForCausalLM.from_config(c)
for role in ("student", "teacher"):
    try:
        s = P.inspect_structure(meta_model(getattr(cfg.model, f"{role}_id"), REV[role]))
        info[role]["prunable"] = {"ok": True, "structure": vars(s), "reloadable_kv_counts": P.valid_kv_counts(s)}
    except P.UnsupportedArchitecture as e:
        info[role]["prunable"] = {"ok": False, "diagnostic": str(e)}
assert info["student"]["prunable"]["ok"], "Student cannot be pruned safely:\n" + info["student"]["prunable"]["diagnostic"]

tok_s = AutoTokenizer.from_pretrained(cfg.model.student_id, revision=REV["student"], token=ENV.hf_token())
tok_t = AutoTokenizer.from_pretrained(cfg.model.teacher_id, revision=REV["teacher"], token=ENV.hf_token())
probes = ["The quick brown fox jumps over 13 lazy dogs.", "def f(x):\n    return x ** 2", "Ünïcödé — 数学 🙂"]
KD_COMPAT = M.vocab_compatibility(tok_t, tok_s, probes)
MODEL_MANIFEST = {"student": info["student"], "teacher": info["teacher"], "kd_compatibility": KD_COMPAT,
                  "code_sha256": {p.name: CK.sha256_file(p) for p in sorted((ROOT / "code" / "jlc").glob("*.py"))}}
M.write_model_manifest(ROOT / "results" / "model_manifest.json", MODEL_MANIFEST)
cfg.save_yaml(ROOT / "results" / "experiment.yaml")   # now with pinned revisions
import pandas as pd
display(pd.DataFrame({r: {k: v for k, v in info[r].items() if not isinstance(v, (dict, list))} for r in info}))
print("Token-level KD decision:", KD_COMPAT["decision"])
print("Teacher prunability:", info["teacher"]["prunable"].get("diagnostic", "ok")[:300])

---
## 5. Dataset acquisition, normalization, deduplication and split manifests

**Purpose.** Download UltraChat, GSM8K, WikiText-2 and IFEval at pinned revisions; normalize, deduplicate, remove any training prompt sharing a 13-gram with any evaluation item; write split files and checksums.

**Expected outputs.** `data/*.jsonl`, `data/dataset_manifest.json` (revisions, counts after each filter, roles: train / pilot / final, SHA-256).

**Validation.** Hard assertions in `build_splits`: no held-out prompt in teacher prompts, GSM8K dev and teacher pools disjoint. Re-running verifies checksums and reuses the splits.

**If something fails.** Interrupted download → re-run (retries with back-off are built in). Changed data settings → a new manifest is built; old teacher targets are then rejected by the cache check in Section 6.

In [ ]:
reload_jlc()
from jlc import data as D
DATA_MANIFEST = D.build_splits(cfg, ROOT / "data", student_tokenizer=tok_s, seed=0)
CK.atomic_write_json(ROOT / "results" / "dataset_manifest.json", DATA_MANIFEST)
display(pd.DataFrame(DATA_MANIFEST["splits"]).T)
display(pd.DataFrame(DATA_MANIFEST["filtering"]).T)
assert DATA_MANIFEST["splits"]["teacher_prompts"]["n"] > 0
assert {v["role"] for v in DATA_MANIFEST["splits"].values()} == {"train", "pilot", "final"}

---
## 6. Teacher response generation and persistent caching

**Purpose.** Generate the teacher's responses once, with fixed greedy decoding, into resumable shards on Drive. Failed, empty, truncated and (GSM8K) wrong-answer generations are recorded and counted, not discarded silently.

**Expected outputs.** `teacher_targets/<teacher>/shard_*.jsonl`, `generation_config.json`, `summary.json` (status counts, exclusion policy, shard checksums).

**Validation.** Every prompt has a record; the cache refuses to mix targets produced with different decoding settings.

**If something fails.** Colab disconnect → re-run: finished shards are skipped, errored prompts are retried. Out of memory → the batch size halves automatically; on a T4 the teacher is loaded in 4-bit NF4 and this choice is stored with the targets.

In [ ]:
reload_jlc()
from jlc import teacher as TG
if cfg.model.teacher_load_4bit is None:      # decide once and record it (it changes the targets)
    need = ENV.estimate_model_gb(info["teacher"]["params_estimate"], 2) * 1.6
    cfg.model.teacher_load_4bit = bool(HW["cuda_available"] and ENV.free_gpu_gb() < need)
TEACHER_DIR = ROOT / "teacher_targets" / cfg.model.teacher_id.replace("/", "__").strip("_")
prompts = D.read_split(ROOT / "data", "teacher_prompts")
cfg.save_yaml(ROOT / "results" / "experiment.yaml")   # final config: revisions pinned, teacher precision decided
print("config hash used for all checkpoints from here on:", cfg.hash(),
      "\nFor the final phase, pin these in CONFIG_OVERRIDES:", {"model.student_revision": REV["student"],
      "model.teacher_revision": REV["teacher"], "model.teacher_load_4bit": cfg.model.teacher_load_4bit})
T_RES = TG.generate_teacher_targets(cfg, prompts, TEACHER_DIR, teacher_revision=REV["teacher"])
print(json.dumps(T_RES["summary"]["status_counts"], indent=1), "| usable:", T_RES["summary"]["n_usable_for_training"])
assert T_RES["summary"]["missing"] == 0, "some prompts have no record - re-run this cell"
assert T_RES["summary"]["n_usable_for_training"] > 0, "no usable teacher targets"
ENV.cleanup()

---
## 7. Baseline teacher and student evaluation (pilot data)

**Purpose.** Sanity-check the evaluation code and calibrate expectations on **pilot** data (GSM8K-train hold-out, WikiText-2 validation). Held-out baselines are computed in Section 17 together with all other conditions.

**Expected outputs.** `results/runs/E0__pilot_*`, `TEACHER__pilot_*` with per-item outputs in `results/items/`.

**Validation.** GSM8K exact match within [0, 1], perplexity finite, per-item files exist.

**If something fails.** If the protocol is already frozen (re-run of a final phase) this cell only shows the stored pilot results.

In [ ]:
reload_jlc()
from jlc import experiments as X, pipeline as PL, evaluate as EV
CTX = {"student_revision": REV["student"], "teacher_revision": REV["teacher"], "kd_compat": KD_COMPAT,
       "teacher_usable": T_RES["usable"]}
runner = X.Runner(cfg, ROOT, CTX)
display(pd.DataFrame(X.budget_table(cfg)))
if C.is_frozen(cfg, ROOT / "results"):
    print("Protocol frozen: showing stored pilot baselines only.")
else:
    base_rows = [X.evaluate_condition(runner, "E0", 0, split="pilot", measure_load=False),
                 X.evaluate_teacher(runner, split="pilot")]
    display(pd.DataFrame(base_rows)[["exp", "status", "gsm8k_em", "gsm8k_format_valid", "wikitext2_ppl"]])
    for r in base_rows:
        assert r["status"] == "ok", r.get("error")
        assert 0 <= r["gsm8k_em"] <= 1 and r["wikitext2_ppl"] > 1

---
## 8. Distillation-only training

**Purpose.** Train the two uncompressed references with identical budgets: R1 (CE on original responses) and E1 (response-level KD on teacher targets, plus token-level KD only if Section 4 found identical vocabularies).

**Expected outputs.** `runs_by_config/<hash>/{R1,E1}/seed_*/final` with `DONE.json`, `train_result.json`; training curves in `logs/`.

**Validation.** Each final checkpoint is reloaded and its logits compared with the in-memory model before `DONE.json` is written.

**If something fails.** Disconnect → re-run: training resumes from the last Drive checkpoint (weights, optimizer, scheduler, RNG, data cursor). OOM → micro-batch halves, gradient accumulation doubles (effective batch unchanged), logged.

In [ ]:
STATUS = {}
STATUS["sec8"] = PL.run_conditions(runner, ["R1", "E1"], PHASE)
display(STATUS["sec8"])
def plot_logs(exps, keys=("L_CE", "L_KD", "L_Q", "R_SP"), seed=None):
    import matplotlib.pyplot as plt
    seed = PL.seeds_for(cfg, PHASE)[0] if seed is None else seed
    fig, axes = plt.subplots(1, len(keys), figsize=(4 * len(keys), 3))
    for e in exps:
        f = ROOT / "logs" / f"{e}_seed{seed}.jsonl"
        if not f.exists(): continue
        L = pd.read_json(f, lines=True)
        L = L[L["step"].notna()] if "step" in L else L
        for ax, k in zip(axes, keys):
            if k in L: ax.plot(L["step"], L[k], label=e, lw=2)
    for ax, k in zip(axes, keys):
        ax.set_title(k); ax.set_xlabel("step"); ax.grid(alpha=.3)
    axes[0].legend(frameon=False); plt.tight_layout(); plt.show()
plot_logs(["R1", "E1"], keys=("L_CE", "L_KD"))

---
## 9. Structured-pruning sensitivity analysis

**Purpose.** Measure how perplexity (WikiText-2 **validation**) degrades when pruning heads, MLP channels or both, for three importance definitions, and per layer — before any recovery training. This informs (in the pilot phase only) the keep ratios.

**Expected outputs.** `results/sensitivity_global.csv`, `results/sensitivity_layer.csv`, `figures/sensitivity.png`.

**Validation.** Dense row equals the model's unpruned perplexity; masks are removed after the analysis (asserted).

**If something fails.** Long runtime in full mode → reduce `eval.ppl_max_chunks` via overrides (it only affects this analysis and pilot evaluations).

**Importance definitions** (per KV group *g* in layer *l*, per MLP channel *c*):

* **activation** (Minitron): $I_g=\frac1N\sum_t\sum_{h\in g}\lVert o_{t,h}\rVert_2$ over the per-head attention outputs; $I_c=\frac1N\sum_t |a_{t,c}|$ for the SwiGLU activation entering `down_proj`.
* **taylor**: $I=\big|\sum_t \partial\mathcal L/\partial m\big|$ for a multiplicative mask $m=1$ on the unit (first-order estimate of the loss change on removal).
* **magnitude** (data-free): weight norms of the rows/columns that belong to the unit.

In [ ]:
reload_jlc()
from jlc import analysis as AN
import matplotlib.pyplot as plt
sens_f = ROOT / "results" / "sensitivity_global.csv"
if not sens_f.exists():
    m0, t0_ = M.load_model(cfg.model.student_id, REV["student"], cfg=cfg)
    calib = D.calibration_batches(t0_, D.read_split(ROOT / "data", "calibration"), cfg.train.seq_len,
                                  cfg.train.micro_batch, t0_.pad_token_id)
    val = D.read_split(ROOT / "data", "wikitext2_validation")[0]["text"]
    chunks = cfg.eval.ppl_max_chunks or 32
    SG = AN.global_sensitivity(m0, t0_, calib, val, cfg.prune.sensitivity_ratios, ctx=cfg.eval.ppl_ctx,
                               chunks=chunks, multiple=cfg.prune.mlp_multiple)
    SL = AN.layer_sensitivity(m0, t0_, calib, val, keep=0.5, ctx=cfg.eval.ppl_ctx, chunks=max(4, chunks // 4),
                              multiple=cfg.prune.mlp_multiple)
    assert not any(m._forward_pre_hooks for m in m0.modules()), "masks were not removed"
    SG.to_csv(sens_f, index=False); SL.to_csv(ROOT / "results" / "sensitivity_layer.csv", index=False)
    del m0; ENV.cleanup()
SG = pd.read_csv(sens_f); SL = pd.read_csv(ROOT / "results" / "sensitivity_layer.csv")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
colors = {"activation": "#2a78d6", "taylor": "#eb6834", "magnitude": "#1baf7a"}
for (meth, which), g in SG[SG.method != "-"].groupby(["method", "which"]):
    ax[0].plot(g.keep_ratio, g.ppl_ratio_vs_dense, marker={"heads": "o", "mlp": "s", "both": "^"}[which],
               color=colors[meth], lw=2, ms=6, label=f"{meth}/{which}")
ax[0].set_yscale("log"); ax[0].invert_xaxis(); ax[0].set_xlabel("keep ratio"); ax[0].set_ylabel("PPL / dense PPL")
ax[0].legend(fontsize=7, ncol=3, frameon=False); ax[0].grid(alpha=.3); ax[0].set_title("Global sensitivity", loc="left")
for which, g in SL.groupby("which"):
    ax[1].plot(g.layer, g.delta_ppl, marker="o", lw=2, label=which, color="#2a78d6" if which == "heads" else "#eb6834")
ax[1].set_xlabel("layer"); ax[1].set_ylabel("Δ PPL (one layer at 50%)"); ax[1].legend(frameon=False); ax[1].grid(alpha=.3)
ax[1].set_title("Per-layer sensitivity", loc="left")
plt.tight_layout(); fig.savefig(ROOT / "figures" / "sensitivity.png", dpi=150); plt.show()
display(SG.head(12))

---
## 10. Architecture-aware head and MLP-channel pruning

**Purpose.** Demonstrate and verify the four distinct objects — gates, permanent masks, physical reduction, exported parameters — on the real student, then train E2 (pruning only).

**Expected outputs.** A verification table; `runs_by_config/<hash>/E2/...`.

**Validation.** Masked model ≡ physically pruned model (max |Δlogit| ≈ 0), analytic parameter count matches, shapes, generation and save→reload all pass. Zeroed weights reduce *non-zero* parameters but not dimensions — shown explicitly.

**If something fails.** `UnsupportedArchitecture` → stop: the architecture needs its own slicing code. A head ratio that is not reloadable is snapped to the nearest valid count and the deviation is printed.

In [ ]:
import copy, tempfile
m0, t0_ = M.load_model(cfg.model.student_id, REV["student"], cfg=cfg)
s0 = P.inspect_structure(m0)
kv_n, mlp_n, kinfo = P.keep_counts(s0, cfg.prune.head_keep_ratio, cfg.prune.mlp_keep_ratio, cfg.prune.mlp_multiple,
                                   cfg.prune.head_count_policy)
calib = D.calibration_batches(t0_, D.read_split(ROOT / "data", "calibration")[:8], cfg.train.seq_len,
                              cfg.train.micro_batch, t0_.pad_token_id)
imp = P.compute_importance(m0, s0, calib, cfg.prune.importance)
plan = P.plan_from_importance(imp, kv_n, mlp_n, s0)
probe = t0_("Structured pruning verification probe.", return_tensors="pt").input_ids
dense_params = sum(p.numel() for p in m0.parameters())
# (2) permanent masks: same function, same size
mb = P.MaskBank(m0, s0); mb.apply_plan(plan)
with torch.no_grad(): ref = m0(probe.to(m0.device)).logits.cpu()
masked_params = sum(p.numel() for p in m0.parameters()); mb.remove()
# zeroing weights instead of masking: fewer non-zeros, same dimensions
mz = copy.deepcopy(m0)
with torch.no_grad():
    for i, Lr in enumerate(mz.model.layers):
        drop = sorted(set(range(s0.inter)) - set(plan.mlp_keep[i]))
        Lr.mlp.down_proj.weight[:, drop] = 0
zeroed = EV.model_stats(mz); del mz
# (3) physical reduction
rep = P.physically_prune(m0, plan)
ver = P.verify_pruned(m0, t0_, s0, plan, ref_logits=ref, probe_ids=probe)
with tempfile.TemporaryDirectory() as td:
    m0.save_pretrained(td); t0_.save_pretrained(td)
    re = AutoModelForCausalLM.from_pretrained(td, torch_dtype=next(m0.parameters()).dtype).to(m0.device)
    with torch.no_grad():
        reload_diff = (re(probe.to(m0.device)).logits - m0(probe.to(m0.device)).logits).abs().max().item()
    exported_bytes = EV.dir_size_bytes(td)
del re
table = pd.DataFrame([
    {"object": "dense model", "params": dense_params, "nonzero": dense_params, "dims_changed": False},
    {"object": "permanent masks (hooks)", "params": masked_params, "nonzero": masked_params, "dims_changed": False},
    {"object": "zeroed MLP columns", "params": zeroed["params_total"], "nonzero": zeroed["params_nonzero"], "dims_changed": False},
    {"object": "physically pruned", "params": rep["params_after"], "nonzero": EV.model_stats(m0)["params_nonzero"], "dims_changed": True},
    {"object": "exported safetensors (bytes)", "params": None, "nonzero": None, "dims_changed": True, "bytes": exported_bytes}])
display(table); print("keep info:", kinfo); print("verification:", ver, "| reload max|Δ|:", reload_diff)
assert ver["max_abs_diff_vs_masked"] < 5e-2 and reload_diff < 1e-3
del m0; ENV.cleanup()

In [ ]:
STATUS["sec10"] = PL.run_conditions(runner, ["E2"], PHASE)
display(STATUS["sec10"])

---
## 11. Post-pruning recovery through distillation

**Purpose.** E4: prune the distilled student E1 one-shot (calibration importance) and recover with response targets + token-level KD from E1.

**Expected outputs.** `runs_by_config/<hash>/E4/...`, its pruning plan in `plans/E4_plan.json`.

**Validation.** Pruning verification runs inside the stage (masked ≡ physical) before any training step.

**If something fails.** Same as Section 8. The one-shot plan is saved before training, so a resumed run re-applies exactly the same structure.

In [ ]:
STATUS["sec11"] = PL.run_conditions(runner, ["E4"], PHASE)
display(STATUS["sec11"])
plot_logs(["E2", "E4"], keys=("L_CE", "L_KD"))

---
## 12. INT8 and INT4 post-training quantization baselines (and the sequential pipeline E7)

**Purpose.** E3 / E3_8: round-to-nearest onto the GGUF Q4_0 / Q8_0 grids (what `llama-quantize` does), embeddings at Q8_0. E7 = E4 → INT4 PTQ completes the sequential baseline before any joint method is run.

**Expected outputs.** `quantization.json` next to each checkpoint: backend & version, bit widths, group size, scaling, calibration data (none for RTN), excluded layers, export format, runtime.

**Validation.** The PyTorch quantizer is checked bit-for-bit against llama.cpp's gguf-py reference on real student weights; the GGUF itself is checked again in Section 18.

**If something fails.** A row length not divisible by 32 cannot be stored as Q4_0 → the pruning multiple guarantees this; the error message names the tensor otherwise.

In [ ]:
reload_jlc()
from jlc import quant as Q
import numpy as np, gguf
from gguf import quants as GQ
m0, _ = M.load_model(cfg.model.student_id, REV["student"], cfg=cfg)
w = m0.model.layers[0].mlp.down_proj.weight.detach().float().cpu()
for name, qt in (("Q4_0", gguf.GGMLQuantizationType.Q4_0), ("Q8_0", gguf.GGMLQuantizationType.Q8_0)):
    ref = GQ.dequantize(GQ.quantize(w.numpy(), qt), qt)
    diff = np.abs(Q.GRIDS[name](w).numpy() - ref).max()
    print(f"{name}: PyTorch vs gguf-py max |Δ| = {diff}")
    assert diff == 0.0, "quantizer is not bit-exact"
del m0; ENV.cleanup()
STATUS["sec12"] = PL.run_conditions(runner, ["E3", "E3_8", "E7"], PHASE)
display(STATUS["sec12"])

---
## 13. Quantization-aware training with an explicitly supported fake-quantizer

**Purpose.** E5 (KD + QAT from E1) and E6 (prune + QAT, no KD). Fake quantization is active in every forward pass of training (STE); at the end the weights are snapped to the Q4_0 grid. This is **not** PTQ and not INT4 loading.

**Expected outputs.** `runs_by_config/<hash>/{E5,E6}/...` with `quantization.json` (`kind: qat`).

**Validation.** Unit checks below: STE gradient is identity; switching fake-quant off reproduces the FP forward exactly; converted weights lie on the grid; torchao's grid differs from Q4_0 by a measured amount.

**If something fails.** If torchao is unavailable only the torchao ablation is affected.

In [ ]:
w = torch.randn(64, 256, requires_grad=True)
Q.fake_quant_q4_0(w).sum().backward()
assert torch.equal(w.grad, torch.ones_like(w)), "STE gradient is not identity"
lin = torch.nn.Linear(256, 64); x = torch.randn(4, 256)
ref = lin(x).detach(); Q.prepare_qat(torch.nn.Sequential(lin), exclude=())
lin.fq_enabled = False; assert torch.equal(lin(x), ref)
lin.fq_enabled = True; yq = lin(x).detach(); Q.convert_qat(torch.nn.Sequential(lin))
assert torch.equal(lin(x), yq) and torch.equal(Q.q4_0_dequant(lin.weight), lin.weight)
try:
    from torchao.quantization.qat import IntxFakeQuantizeConfig
    from torchao.quantization.qat.fake_quantizer import IntxFakeQuantizer
    tq = IntxFakeQuantizer(IntxFakeQuantizeConfig(torch.int4, group_size=32, is_symmetric=True))
    v = torch.randn(256, 256)
    print("torchao int4 grid vs Q4_0 grid: mean |Δ| =", (tq(v) - Q.q4_0_dequant(v)).abs().mean().item())
except Exception as e:
    print("torchao unavailable:", e)
STATUS["sec13"] = PL.run_conditions(runner, ["E5", "E6"], PHASE)
display(STATUS["sec13"])
plot_logs(["E5", "E6"], keys=("L_Q", "L_KD"))

---
## 14. Joint distillation, structured pruning and quantization (E8)

**Purpose.** One run optimizes $\mathcal L_{joint}$: structured gates with $R_{SP}$, fake-quantized forward ($L_Q$), and token-level KD from E1. Gate temperature anneals; at 60 % of the stage the gates are frozen into a uniform-width plan, their values folded into `o_proj`/`down_proj`, the model is physically pruned, and QAT+KD continues on the pruned network.

**Expected outputs.** `runs_by_config/<hash>/E8/...`, gate statistics in the log, the learned plan in `plans/E8_plan.json`.

**Validation.** Before running on the real model, a short self-test on a copy checks that the gated → hard-pruned transition preserves the function for kept units.

**If something fails.** Resume works across the hard-prune boundary (the plan is stored in the checkpoint and re-applied before loading weights).

In [ ]:
# Self-test on the real student dimensions (2 layers, random weights): gated -> folded hard prune is exact
from transformers import AutoConfig, AutoModelForCausalLM
c2 = AutoConfig.from_pretrained(cfg.model.student_id, revision=REV["student"], token=ENV.hf_token())
c2.num_hidden_layers = 2
torch.manual_seed(0)
tm = AutoModelForCausalLM.from_config(c2).float().eval()
ts = P.inspect_structure(tm)
kv_n, mlp_n, _ = P.keep_counts(ts, cfg.prune.head_keep_ratio, cfg.prune.mlp_keep_ratio, cfg.prune.mlp_multiple,
                               cfg.prune.head_count_policy)
gb = P.GateBank(tm, ts)
with torch.no_grad():
    gb.kv_logits.normal_(); gb.mlp_logits.normal_()
pl = gb.to_plan(kv_n, mlp_n); kvv, mlpv = gb.gate_values_for(pl)
with torch.no_grad():
    for i in range(ts.n_layers):
        drop_kv = sorted(set(range(ts.n_kv)) - set(pl.kv_keep[i])); gb.kv_logits[i, drop_kv] = -1e4
        drop_m = sorted(set(range(ts.inter)) - set(pl.mlp_keep[i])); gb.mlp_logits[i, drop_m] = -1e4
    ids = torch.randint(0, c2.vocab_size, (1, 24)); ref = tm(ids).logits
gb.remove(); P.physically_prune(tm, pl, fold_gates=(kvv, mlpv))
with torch.no_grad(): d = (tm(ids).logits - ref).abs().max().item()
print("gated -> folded physical prune max |Δlogit| =", d); assert d < 1e-3
del tm, gb

STATUS["sec14"] = PL.run_conditions(runner, ["E8"], PHASE)
display(STATUS["sec14"])
plot_logs(["E8"], keys=("L_Q", "L_KD", "R_SP", "gate_kv_mean"))

---
## 15. Alternating optimization (E9)

**Purpose.** Same start, budget, data and final structure as E8, but the objective alternates every `alt.cycle_steps` steps between a *prune phase* (gates trainable, full precision, $R_{SP}$ on) and a *quant phase* (gates frozen, fake-quant on, $L_Q$). After the hard prune all steps are QAT.

**Expected outputs.** `runs_by_config/<hash>/E9/...`; the log's `alt_phase` column shows the schedule.

**Validation.** Both phases appear in the log (asserted).

**If something fails.** As Section 14.

In [ ]:
STATUS["sec15"] = PL.run_conditions(runner, ["E9"], PHASE)
display(STATUS["sec15"])
s_ = PL.seeds_for(cfg, PHASE)[0]
lg = pd.read_json(ROOT / "logs" / f"E9_seed{s_}.jsonl", lines=True)
if "alt_phase" in lg:
    phases = set(lg["alt_phase"].dropna())
    print("phases seen:", phases); assert phases == {"prune", "quant"}
plot_logs(["E8", "E9"], keys=("L_Q", "L_KD", "R_SP", "gate_kv_mean"))

---
## 16. Ablation experiments and protocol freeze

**Purpose.** Ablations isolate the components of E8 (H4 quantization-aware gating, KD, the FP CE term, the fake-quantizer backend) and pruning axes. Then — only in the final phase — the protocol is frozen.

**Expected outputs.** `runs_by_config/<hash>/A_*`, `results/protocol_frozen.json`.

**Validation.** Freezing records the config hash; any later change to the config makes final evaluation refuse to run.

**If something fails.** In the pilot phase: compare pilot metrics, change `CONFIG_OVERRIDES`, re-run Sections 2-16 (new config hash → fresh checkpoints, nothing is overwritten). When satisfied, set `PHASE="final"` in Section 2 and run all.

In [ ]:
ABL = [a for a in X.ABLATIONS if not (a == "A_E5_torchao" and not api["torchao IntxFakeQuantizer"].startswith("ok"))]
STATUS["sec16"] = PL.run_conditions(runner, ABL, PHASE)
display(STATUS["sec16"])
if PHASE == "pilot":
    pilot = X.collect_runs(ROOT, split="pilot", config_hash=cfg.hash())
    cols = [c for c in ("exp", "seed", "gsm8k_em", "wikitext2_ppl", "params_total") if c in pilot]
    display(pilot[cols].sort_values("exp") if len(pilot) else pilot)
    print("Pilot phase: NOT freezing. Tune via CONFIG_OVERRIDES, then set PHASE='final' and run all.")
else:
    FROZEN = C.freeze_protocol(cfg, ROOT / "results")
    code_f = ROOT / "results" / "protocol_code_sha256.json"
    if not code_f.exists():   # code fingerprint at freeze time; later edits are disclosed in the report
        CK.atomic_write_json(code_f, {p.name: CK.sha256_file(p) for p in sorted((ROOT / "code" / "jlc").glob("*.py"))})
    print("Protocol frozen with hash", FROZEN["config_hash"], "at", FROZEN["frozen_at_utc"])

---
## 17. Held-out quality evaluation

**Purpose.** Evaluate every condition and seed on GSM8K test, all IFEval prompts and WikiText-2 test, plus the teacher as a reference. Greedy decoding, the student's own chat template, identical prompts for all models.

**Expected outputs.** `results/runs/<exp>_seed<s>.json`, per-item outputs `results/items/<exp>/seed_<s>/*.jsonl`.

**Validation.** Refuses to run unless the protocol is frozen with the current config hash. Failed evaluations are recorded with status `failed` and surfaced in the table.

**If something fails.** Re-running skips finished evaluations. Out of memory during generation → batch size halves automatically.

In [ ]:
FINAL_EXPS = X.MAIN + ABL
if PHASE != "final":
    print("Skipped: pilot phase (held-out evaluation is only allowed after freezing).")
else:
    rows = []
    for e in FINAL_EXPS:
        for s in cfg.train.seeds:
            try:
                r = X.evaluate_condition(runner, e, s, split="final")
            except Exception as ex:
                r = {"exp": e, "seed": s, "status": "failed", "error": repr(ex)[:300]}
            rows.append({k: r.get(k) for k in ("exp", "seed", "status", "gsm8k_em", "ifeval_prompt_strict",
                                                "wikitext2_ppl", "params_total", "error")})
    TEACHER_FINAL = X.evaluate_teacher(runner, split="final")
    EVAL_TABLE = pd.DataFrame(rows); display(EVAL_TABLE)
    n_bad = (EVAL_TABLE.status != "ok").sum()
    print(f"{n_bad} evaluation(s) not ok" if n_bad else "all evaluations ok")

---
## 18. Exported-model validation (GGUF)

**Purpose.** Build llama.cpp (pinned commit recorded), convert each final checkpoint to GGUF, quantize to its deployment type, and validate the files.

**Expected outputs.** `exports/<hash>/<exp>/seed_<s>/*.gguf` + `export.json` (file size, tensor-type audit, grid identity, perplexity parity); `exports/llama_cpp.json`.

**Validation.** (a) every decoder linear has the requested type, norms are F32, embeddings Q8_0; (b) dequantized GGUF tensors equal the PyTorch weights on the grid (expected max |Δ| = 0 for PTQ/QAT models); (c) llama.cpp perplexity within ±3 % of the PyTorch perplexity under the identical protocol.

**If something fails.** The llama.cpp converter requirements file is deliberately **not** installed (it pins a CPU-only torch). A parity failure is flagged in the report and its runtime numbers are marked.

In [ ]:
reload_jlc()
from jlc import export as XP, experiments as X
runner = X.Runner(cfg, ROOT, CTX)
LLAMA_DIR = os.environ.get("JLC_LLAMA_DIR", "/content/llama.cpp")
LLAMA = XP.build_llama_cpp(LLAMA_DIR, cfg.bench.llama_cpp_repo, cfg.bench.llama_cpp_ref)
CK.atomic_write_json(ROOT / "exports" / "llama_cpp.json", LLAMA)
print("llama.cpp commit:", LLAMA["commit"])
EXPORTS = {}
if PHASE != "final":
    print("Pilot phase: exporting only E8 for a pipeline check.")
    todo = [("E8", PL.seeds_for(cfg, PHASE)[0])]
else:
    todo = [(e, s) for e in FINAL_EXPS for s in cfg.train.seeds if not (e in X.SEED_INDEPENDENT and s != cfg.train.seeds[0])]
for e, s in todo:
    try:
        EXPORTS[(e, s)] = X.export_condition(runner, e, s, LLAMA, parity_chunks=cfg.eval.ppl_max_chunks or 20)
    except Exception as ex:
        CK.log_failure(ROOT, f"export:{e}:{s}", ex); EXPORTS[(e, s)] = {"error": repr(ex)[:300]}
exp_df = pd.DataFrame([{"exp": e, "seed": s, "type": r.get("deploy_type"), "MB": r.get("artifact_mb"),
                        "audit_ok": (not r["audit"]["violations"]) if "audit" in r else None,
                        "grid_max_abs_diff": r.get("grid_identity", {}).get("max_abs_diff"),
                        "parity_rel_diff": r.get("parity", {}).get("rel_diff"), "parity_pass": r.get("parity", {}).get("pass"),
                        "error": r.get("error")} for (e, s), r in EXPORTS.items()])
display(exp_df)

---
## 19. Android-compatible deployment preparation

**Purpose.** Produce everything needed to run the exported GGUF files on a phone with llama.cpp: a cross-compile script (Android NDK, arm64-v8a, same llama.cpp commit), a benchmark script driven by `adb`, and a `device.json` template.

**Expected outputs.** `android/build_android.sh`, `android/bench_android.sh`, `android/device.json`.

**Validation.** Scripts are syntax-checked with `bash -n`.

**If something fails.** **No device → the Android stage stays `PENDING`.** Nothing in the notebook fabricates mobile numbers or substitutes desktop timings.

In [ ]:
ANDROID_FILES = XP.write_android_kit(ROOT / "android", LLAMA, cfg)
for f in ANDROID_FILES[:2]:
    assert subprocess.run(["bash", "-n", f]).returncode == 0, f
print("\n".join(ANDROID_FILES))
print("""
Steps (host computer with adb + the phone in developer mode):
 1. ANDROID_NDK=/path/to/ndk bash android/build_android.sh <llama.cpp checkout at the commit above>
 2. Copy the GGUF files you want to compare from exports/<hash>/... into one folder
 3. bash android/bench_android.sh <llama.cpp>/build-android/bin <gguf folder>
 4. Upload the produced android_results/ folder to <DRIVE_ROOT>/android/ and fill android/device.json
 5. Re-run Sections 20-22.
Optional in Colab: download the NDK and cross-compile here (binaries only; benchmarking still needs a phone).""")

---
## 20. Runtime and resource benchmarking

**Purpose.** Measure prefill throughput/latency, decode tokens/s, derived time-to-first-token and peak RSS for every exported artifact with `llama-bench` on this machine's CPU (labelled `desktop_cpu`), optionally PyTorch GPU timings (`desktop_gpu`), and ingest Android results if present.

**Expected outputs.** `results/runtime_benchmarks.csv` with a `platform` and `measurement` column on every row; `results/android_status.json`.

**Validation.** Android rows exist only if `device.json` is filled and result files are present; otherwise status is `PENDING`.

**If something fails.** Benchmarks are slow on Colab CPUs for full-size models; reduce `bench.repetitions` via overrides if needed (recorded).

In [ ]:
reload_jlc()
from jlc import bench as BN
brows = []
for (e, s), r in EXPORTS.items():
    if "gguf" not in r: continue
    try:
        brows += BN.desktop_cpu_bench(LLAMA, Path(r["gguf"]), cfg, {"exp": e, "seed": s, "artifact_mb": r["artifact_mb"],
                                                                     "parity_pass": r["parity"]["pass"]})
    except Exception as ex:
        CK.log_failure(ROOT, f"bench:{e}:{s}", ex)
art_index = {Path(r["gguf"]).stem: {"exp": e, "seed": s, "artifact_mb": r["artifact_mb"]}
             for (e, s), r in EXPORTS.items() if "gguf" in r}
arows, ANDROID_STATUS = BN.ingest_android(ROOT / "android", art_index)
CK.atomic_write_json(ROOT / "results" / "android_status.json", ANDROID_STATUS)
BENCH = pd.DataFrame(brows + arows)
BENCH.to_csv(ROOT / "results" / "runtime_benchmarks.csv", index=False)
print("Android:", ANDROID_STATUS["status"], ANDROID_STATUS.get("reason", ""))
display(BENCH[[c for c in ("exp", "seed", "platform", "prompt_len", "prefill_ms", "ttft_ms", "decode_tok_s",
                           "peak_rss_mb", "measurement") if c in BENCH]] if len(BENCH) else BENCH)
assert BENCH.empty or BENCH["platform"].notna().all()

---
## 21. Statistical analysis and visualization

**Purpose.** Aggregate over seeds (mean, SD, t-based 95 % CI), run the pre-registered paired comparisons (paired bootstrap over items, exact McNemar, seed-paired t-test, Holm correction), and draw quality-vs-size / quality-vs-latency / Pareto plots.

**Expected outputs.** `metrics.csv`, `metrics.json`, `statistical_tests.csv`, `quality_vs_size.png`, `quality_vs_latency.png`, `pareto_frontier.png`.

**Validation.** Every number comes from `results/runs` + `results/items` + `exports` + `runtime_benchmarks.csv`; failed runs are counted in `n_failed`. Latency plots use one platform only (Android if measured, else desktop CPU, named on the axis).

**If something fails.** Missing per-item files for a pair → that test row has `status = missing items` instead of a number.

In [ ]:
reload_jlc()
from jlc import stats as ST, experiments as X
if PHASE != "final":
    print("Skipped: pilot phase.")
else:
    RUNS = X.collect_runs(ROOT, "final", cfg.hash())
    RUNS = RUNS[RUNS.exp != "TEACHER"]
    RUNS.to_csv(ROOT / "results" / "metrics.csv", index=False)
    RUNS.to_json(ROOT / "results" / "metrics.json", orient="records", indent=1)
    SUMMARY = ST.aggregate(RUNS, cfg.eval.alpha)
    SUMMARY.to_csv(ROOT / "results" / "metrics_summary.csv", index=False)
    TESTS = ST.run_tests(ROOT, RUNS, list(cfg.train.seeds), cfg.eval.bootstrap_iters)
    TESTS.to_csv(ROOT / "results" / "statistical_tests.csv", index=False)
    FIGS = ST.make_figures(SUMMARY, BENCH, ROOT / "figures")
    for k in ("quality_vs_size", "quality_vs_latency", "pareto_frontier"):
        if k in FIGS:
            import shutil; shutil.copy2(FIGS[k], ROOT / "results" / Path(FIGS[k]).name)
    display(SUMMARY[[c for c in SUMMARY if c.endswith("_mean") or c in ("exp", "n_seeds_ok", "n_failed")]].round(4))
    display(TESTS)
    from IPython.display import Image
    for k in ("quality_vs_size", "quality_vs_latency", "pareto_frontier"):
        if k in FIGS: display(Image(FIGS[k]))

---
## 22. Final research report and artifact manifest

**Purpose.** Write `research_results_report.md` from the files on disk, list unexecuted or pending stages, and produce a checksummed manifest of every artifact.

**Expected outputs.** `results/research_results_report.md`, `results/artifact_manifest.json`, a copy of this notebook's outputs in `results/`.

**Validation.** The report contains only computed numbers; a smoke/fixture run is stamped as such at the top.

**If something fails.** If a section was skipped or failed, it appears in the validation summary instead of being silently omitted.

In [ ]:
reload_jlc()
from jlc import report as RP, experiments as X
fails = []
fp = ROOT / "logs" / "failures.jsonl"
if fp.exists():
    fails = [json.loads(l) for l in fp.read_text().splitlines() if l.strip()]
code_now = {p.name: CK.sha256_file(p) for p in sorted((ROOT / "code" / "jlc").glob("*.py"))}
code_f = ROOT / "results" / "protocol_code_sha256.json"
code_changed = [k for k, v in json.loads(code_f.read_text()).items() if code_now.get(k) != v] if code_f.exists() else []
if code_changed:
    print("NOTE: package code changed after the protocol was frozen:", code_changed,
          "-> cached results were produced by the earlier code; disclosed in the report.")
VALIDATION = {
    "code_changed_since_freeze": code_changed,
    "phase": PHASE, "config_hash": cfg.hash(), "offline_fixture": bool(OFFLINE),
    "protocol_frozen": C.is_frozen(cfg, ROOT / "results"),
    "status_tables": {k: v[["exp", "seed", "status"]].to_dict("records") for k, v in STATUS.items()},
    "exports_ok": int(sum("gguf" in r for r in EXPORTS.values())), "exports_failed": int(sum("error" in r for r in EXPORTS.values())),
    "parity_failures": [f"{e}/{s}" for (e, s), r in EXPORTS.items() if r.get("parity") and not r["parity"]["pass"]],
    "android": ANDROID_STATUS.get("status"),
    "logged_failures": len(fails),
    "hardware_dependent_stages_not_run": ([] if ANDROID_STATUS.get("status") == "MEASURED" else ["Android benchmarking (Section 19/20)"])
        + (["held-out evaluation, statistics (pilot phase)"] if PHASE != "final" else [])
        + ["energy measurement (no instrumentation)"],
}
if PHASE == "final":
    REPORT = RP.write_report(ROOT, cfg, SUMMARY, TESTS, BENCH, ANDROID_STATUS, FIGS, VALIDATION,
                             X.budget_table(cfg), extra_notes=[f"KD decision: {KD_COMPAT['decision']}"])
    print(REPORT.read_text()[:4000])
else:
    print(json.dumps(VALIDATION, indent=1, default=str))
MAN = RP.artifact_manifest(ROOT)
print("manifest entries:", len(MAN["files"]), "| completed model dirs:", len(MAN["completed_model_dirs"]))

### Run guide

**Google Colab (recommended).**
1. *Runtime → Change runtime type → GPU* (T4 for smoke mode, A100 for full mode). Add `HF_TOKEN` under the key icon if needed.
2. Section 2: set `MODE`. Run all. Everything persists in `MyDrive/jlc_research[_smoke]/`.
3. Disconnected? Reconnect and *Run all* again: data, teacher shards, finished stages, mid-training checkpoints, evaluations and exports are reused.
4. Full mode: run with `PHASE="pilot"` (pilot seed, pilot data). Adjust `CONFIG_OVERRIDES`; each new config gets its own checkpoint namespace. Then set `PHASE="final"` and run all: the protocol is frozen in Section 16 and held-out evaluation runs for every seed.
5. Rough full-mode cost on one A100-80GB with S = 600 steps (32 × 1024-token sequences/step): ≈ 1–1.5 h per trained stage → ≈ 10 trained stages × 3 seeds ≈ 35–45 GPU-hours, plus teacher generation (≈ 25k prompts) and evaluation. Split across sessions; nothing is lost on disconnect.

**Local CUDA workstation.** `export JLC_DRIVE_ROOT=/data/jlc_research` (any local path), install PyTorch for your CUDA, then run the notebook with Jupyter. Optional: `pip install vllm` in a separate environment for faster teacher generation (`teacher_gen.backend="vllm"`).

**Android benchmarking.** See Section 19. Use one phone for all models, same thread count, cool-down between models, airplane mode, record battery/temperature in `device.json`. Upload `android_results/` and re-run Sections 20–22.

**Offline CI test (no network, no GPU).** `tests/fixtures.py` builds a SentencePiece tokenizer, tiny random Llama teacher/student and schema-identical synthetic datasets; setting `JLC_OFFLINE_FIXTURES` makes the notebook use them. Numbers from that mode are meaningless and stamped as such.